# REXA — Reasoning Analysis of Explainable Descriptive Answers

**Project Goal:** Analyze descriptive student answers by identifying sentence-level reasoning functions, estimating reasoning depth on a 0★–5★ scale, and generating visual explanations.

> ⚠️ REXA does **not** grade answers, predict correctness, or replace teachers. It only analyzes reasoning quality and reasoning progression.

---

## Notebook Structure

| Section | Content | Role |
|---------|---------|------|
| 1–4 | Setup, Config, Datasets | Infrastructure |
| **5** | **CORE REXA SYSTEM** | **Proposed System** |
| 6 | DistilBERT Regression | Comparative Experiment 1 |
| 7 | DeBERTa NLI | Comparative Experiment 2 |
| 8 | BART Zero-shot | Comparative Experiment 3 |
| 9 | Final Comparison | Evaluation |
| 10 | Conclusions | Summary |

---

## Dataset Summary

| Dataset | Role |
|---------|------|
| Curated 0★–5★ CSVs | Primary — reasoning depth evaluation |
| Mohler (train/val/test) | External validation only |
| SNLI / MultiNLI / QASC | Used in Comparative Experiment 2 only |


## Section 1 — Environment Setup

**Purpose:** Install all required libraries and mount Google Drive for persistent storage.

**What this cell does:**
- Reinstalls `transformers==4.36.2`, `huggingface_hub==0.20.3`, `tokenizers==0.15.2`
- Installs: `datasets`, `accelerate`, `scikit-learn`, `torch`, `sentence-transformers`, `spacy`, `nltk`, `matplotlib`, `seaborn`, `networkx`
- Downloads `en_core_web_sm` spaCy model for sentence parsing
- Loads spaCy + sentencizer pipe

**Expected output:** `Setup complete`

**Note:** If running on Colab free tier, GPU availability is not guaranteed. All models include CPU fallbacks.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import subprocess

# ── Pin transformers stack to stable tested version ───────────────────────────
subprocess.run(["pip", "uninstall", "-y", "transformers", "huggingface_hub", "tokenizers"],
               check=False, capture_output=True)
subprocess.run(["pip", "install", "-q",
                "transformers==4.36.2",
                "huggingface_hub==0.20.3",
                "tokenizers==0.15.2"], check=True)

# ── Install all required packages ─────────────────────────────────────────────
subprocess.run(["pip", "install", "-q",
                "datasets", "accelerate", "scikit-learn", "torch",
                "sentence-transformers", "tqdm", "spacy",
                "nltk", "matplotlib", "seaborn",
                "networkx",          # reasoning graph visualisation
                "scipy",             # pearsonr, spearmanr
                "openai",            # OpenRouter Stage 2 client
                ], check=True)

subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"],
               check=True, capture_output=True)

# ── Core imports ──────────────────────────────────────────────────────────────
import os, re, json, warnings, logging, datetime
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")   # non-interactive backend — saves PNGs reliably in Colab
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import spacy
import torch
import networkx as nx
from collections import defaultdict, Counter
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                              f1_score, cohen_kappa_score, confusion_matrix)
from scipy.stats import pearsonr, spearmanr
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments,
                          pipeline as hf_pipeline, TrainerCallback)
from sentence_transformers import SentenceTransformer, util

warnings.filterwarnings("ignore")

# ── NLTK downloads ────────────────────────────────────────────────────────────
for _pkg in ["stopwords", "wordnet", "omw-1.4", "punkt"]:
    nltk.download(_pkg, quiet=True)

# ── spaCy ─────────────────────────────────────────────────────────────────────
nlp = spacy.load("en_core_web_sm")
if "sentencizer" not in nlp.pipe_names:
    nlp.add_pipe("sentencizer")

# ── GPU check ─────────────────────────────────────────────────────────────────
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE_IDX = 0 if torch.cuda.is_available() else -1
print(f"Setup complete | Device: {DEVICE} | PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")


Mounted at /content/drive
Setup complete | Device: cuda | PyTorch: 2.11.0+cu128
  GPU: Tesla T4


## Section 2 — Central Configuration

**Purpose:** Single source of truth for all paths, flags, hyperparameters, constants, and canonical functions.

**What this cell does:**
- Defines all directory and file paths (no hardcoded paths elsewhere)
- Sets model names: DistilBERT (regression), DeBERTa-v3 (NLI graph), BART (roles), SBERT (similarity)
- Sets training hyperparameters: LR, batch, epochs, warmup, dropout, gradient clipping
- Defines `depth_to_stars()`, `reasoning_level_from_depth()`, `_cached()` — used throughout
- Defines all cue word sets: `EVIDENCE_CUES`, `CONCLUSION_CUES`, `CAUSAL_KW`, `ROLE_MAP`, `ROLE_LABELS`
- Sets flags: `FORCE_RETRAIN_FOR_CURVE`, `USE_FOCAL_LOSS`, `USE_ENSEMBLE`

**Expected output:** `Configuration complete` + flag summary

**Strength:** All configuration in one cell — change once, propagates everywhere.
**Weakness:** Long cell — scroll carefully; do not skip.


In [ ]:
import os, random, logging, numpy as np, torch

# ── Logger (REXA) ─────────────────────────────────────────────────────────────
logging.basicConfig(level=logging.INFO, format="[REXA] %(message)s")
logger = logging.getLogger("REXA")

# ── Reproducibility ────────────────────────────────────────────────────────────
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(RANDOM_STATE)

# ── Force / rebuild flags ──────────────────────────────────────────────────────
FORCE_REBUILD_DATASET    = False
FORCE_RETRAIN_MODEL      = False
FORCE_RETRAIN_FOR_CURVE  = True   # True = always retrain to capture epoch logs
FORCE_REBUILD_STAGE1     = True    # FIX: was False, causing stale-cache bug — see Section 17B note. Set back to False after one clean rerun of Sections 10-17B.
FORCE_REBUILD_STAGE2     = False
FORCE_REBUILD_CURATED    = True   # True once after uploading star CSVs

# ── Model selection ────────────────────────────────────────────────────────────
# Default: distilbert-base-uncased  (fast, low GPU)
# Optional: bert-base-uncased | roberta-base  (set below)
DISTILBERT_MODEL_NAME  = "distilbert-base-uncased"
# DISTILBERT_MODEL_NAME = "bert-base-uncased"     # uncomment to test
# DISTILBERT_MODEL_NAME = "roberta-base"          # uncomment to test
NLI_MODEL_NAME         = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
SBERT_MODEL_NAME       = "all-MiniLM-L6-v2"
BART_MODEL_NAME        = "facebook/bart-large-mnli"

# ── Training hyperparameters ──────────────────────────────────────────────────
LEARNING_RATE   = 2e-5
BATCH_SIZE      = 16
NUM_EPOCHS      = 4
WEIGHT_DECAY    = 0.01
WARMUP_RATIO    = 0.1
MAX_SEQ_LEN     = 256
DROPOUT         = 0.1
GRAD_CLIP       = 1.0

# ── Early stopping ────────────────────────────────────────────────────────────
EARLY_STOPPING_PATIENCE = 2      # stop if val MAE does not improve for N epochs

# ── Class imbalance strategy ──────────────────────────────────────────────────
USE_FOCAL_LOSS  = True           # focal loss for star classification
FOCAL_GAMMA     = 2.0            # gamma for focal loss
EDGE_STAR_WEIGHT = 2.0           # extra weight for 0★ and 5★ samples

# ── Ensemble (optional, default off) ─────────────────────────────────────────
USE_ENSEMBLE = False             # DistilBERT + SBERT feature fusion

# ── Graph parameters ──────────────────────────────────────────────────────────
EDGE_THRESHOLD = 0.25
MAX_SENTENCES  = 10

# ── Directories ───────────────────────────────────────────────────────────────
BASE_DIR        = "/content/drive/MyDrive/FYP_Data"
RAW_DIR         = os.path.join(BASE_DIR, "raw_datasets")
MODEL_DIR       = os.path.join(BASE_DIR, "models", "distilbert_rexa")
CHECKPOINT_DIR  = os.path.join(BASE_DIR, "outputs", "checkpoints")
OUTPUT_DIR      = os.path.join(BASE_DIR, "outputs")
PLOTS_DIR       = os.path.join(OUTPUT_DIR, "plots")
FIGURES_DIR     = os.path.join(OUTPUT_DIR, "figures")
LOGS_DIR        = os.path.join(OUTPUT_DIR, "logs")
TB_DIR          = os.path.join(OUTPUT_DIR, "tensorboard")
ERROR_DIR       = os.path.join(OUTPUT_DIR, "error_analysis")

for _d in [RAW_DIR, MODEL_DIR, CHECKPOINT_DIR, OUTPUT_DIR, PLOTS_DIR,
           FIGURES_DIR, LOGS_DIR, TB_DIR, ERROR_DIR]:
    os.makedirs(_d, exist_ok=True)

# ── Output files ───────────────────────────────────────────────────────────────
COMBINED_ALL_CSV        = os.path.join(OUTPUT_DIR, "combined_all_df.csv")
COMBINED_CSV            = os.path.join(OUTPUT_DIR, "combined_asag_df.csv")  # ASAG-only cache (separate from combined_all_df)
AUXILIARY_CSV           = os.path.join(OUTPUT_DIR, "auxiliary_scoring_df.csv")
REASONING_CSV           = os.path.join(OUTPUT_DIR, "reasoning_df.csv")
TRAIN_CSV               = os.path.join(OUTPUT_DIR, "train.csv")
VAL_CSV                 = os.path.join(OUTPUT_DIR, "val.csv")
TEST_CSV                = os.path.join(OUTPUT_DIR, "test.csv")
STAGE1_SENTENCE_CSV     = os.path.join(OUTPUT_DIR, "stage1_sentence_df.csv")
STAGE1_JSON             = os.path.join(OUTPUT_DIR, "stage1_feedback.json")
STAGE1_FEEDBACK_JSON    = STAGE1_JSON   # alias used in Cell 41
STAGE1_FINAL_JSON       = os.path.join(OUTPUT_DIR, "stage1_final.json")
STAGE2_JSON             = os.path.join(OUTPUT_DIR, "stage2_nli_reasoning_feedback.json")
FINAL_SUMMARY_CSV       = os.path.join(OUTPUT_DIR, "final_feedback_summary.csv")
STAR_EVAL_CSV           = os.path.join(OUTPUT_DIR, "star_level_evaluation.csv")
HUMAN_VALIDATION_CSV    = os.path.join(OUTPUT_DIR, "human_validation_template.csv")
GRAPH_DEMO_JSON         = os.path.join(OUTPUT_DIR, "graph_demo_examples.json")
TRAINING_HISTORY_JSON   = os.path.join(OUTPUT_DIR, "training_history.json")
TRAINING_HISTORY_CSV    = os.path.join(OUTPUT_DIR, "training_history.csv")
BEST_HP_JSON            = os.path.join(OUTPUT_DIR, "best_hyperparameters.json")
METRICS_SUMMARY_JSON    = os.path.join(OUTPUT_DIR, "metrics_summary.json")
ABLATION_CSV            = os.path.join(OUTPUT_DIR, "ablation_study.csv")
FAILURE_CSV             = os.path.join(OUTPUT_DIR, "failure_taxonomy_examples.csv")
ORTHO_CSV               = os.path.join(OUTPUT_DIR, "orthogonality_report.csv")
USER_TEST_JSON          = os.path.join(OUTPUT_DIR, "user_test_output.json")
FINAL_REPORT_JSON       = os.path.join(OUTPUT_DIR, "final_report.json")
FINAL_REPORT_MD         = os.path.join(OUTPUT_DIR, "final_report.md")
MAIN_LOG                = os.path.join(LOGS_DIR,   "main.log")
NLI_WARN_LOG            = os.path.join(LOGS_DIR,   "nli_data_warnings.log")
FAILURE_TAXONOMY_JSON   = os.path.join(ERROR_DIR,  "failure_taxonomy_report.json")
FAILURE_TAXONOMY_CSV    = os.path.join(ERROR_DIR,  "failure_taxonomy_examples.csv")
REASONING_EDGE_CSV      = os.path.join(ERROR_DIR,  "reasoning_edge_cases.csv")
TRAINING_CURVE_PNG      = os.path.join(FIGURES_DIR,"training_validation_curve.png")


# ── Additional output paths (missing from original) ──────────────────────────
GT_CSV                  = os.path.join(OUTPUT_DIR, "reasoning_ground_truth.csv")
REASONING_GROUND_TRUTH_CSV = GT_CSV   # alias
STAGE2_NLI_JSON         = os.path.join(OUTPUT_DIR, "stage2_nli_reasoning_feedback.json")
HUMAN_RESULTS_CSV       = os.path.join(OUTPUT_DIR, "human_validation_results.csv")
STAGE1_SENTENCE_CSV     = os.path.join(OUTPUT_DIR, "stage1_sentence_df.csv")

# ── Pipeline output directories (separate pipelines) ─────────────────────────
REASONING_PIPELINE_DIR      = os.path.join(OUTPUT_DIR, "reasoning_pipeline")
SCORE_PREDICTION_DIR        = os.path.join(OUTPUT_DIR, "score_prediction_pipeline")
for _pd in [REASONING_PIPELINE_DIR, SCORE_PREDICTION_DIR]:
    os.makedirs(_pd, exist_ok=True)

# ── Pipeline flags ────────────────────────────────────────────────────────────
USE_REASONING_PIPELINE          = True
USE_SCORE_PREDICTION_PIPELINE   = True

# ══════════════════════════════════════════════════════════════════════════════
# CANONICAL CONSTANTS AND FUNCTIONS
# Define ONCE here — never redefine in other cells.
# ══════════════════════════════════════════════════════════════════════════════

ROLE_LABELS = ["claim", "explanation", "evidence", "conclusion", "irrelevant"]

ROLE_MAP = {
    "definition":    ["claim", "explanation"],
    "explanation":   ["claim", "explanation", "evidence"],
    "comparison":    ["claim", "evidence", "conclusion"],
    "justification": ["claim", "explanation", "evidence", "conclusion"],
    "process":       ["claim", "explanation", "conclusion"],
    "general":       ["claim", "explanation"],
    "entailment":    ["claim", "explanation"],
    "multi_hop":     ["claim", "explanation", "evidence", "conclusion"],
}

EVIDENCE_CUES = {
    "for example","for instance","such as","e.g","eg ","i.e",
    "as shown","as demonstrated","as illustrated","as seen in",
    "study shows","research shows","data shows","experiments show",
    "according to","in fact","specifically","notably","particularly",
    "in one study","tests revealed","results showed","observations indicate",
    "statistically","empirically","figures show","statistics show",
    "take for example","this is seen in","measured","observed","recorded",
    "found that","proved that","demonstrated that","confirmed that",
    "graph shows","table shows","percentage","sample","experiment",
    "temperature","voltage","case study","data indicates","survey",
    "for instance,","as evidence","evidence shows","data suggest",
}
CONCLUSION_CUES = {
    "therefore","thus","hence","in conclusion","as a result",
    "this shows","this means","this demonstrates","overall",
    "in summary","consequently","accordingly","it follows that",
    "we can conclude","to summarise","so we see","this proves",
    "this implies","the result is","the outcome is",
}
CAUSAL_KW = {
    "because","since","due to","caused by","leads to",
    "results in","consequently","owing to","on account of",
    "therefore","thus","hence","which means","which causes",
}
CAUSAL_CUES = CAUSAL_KW  # alias: Cell 35 uses this historical name

EXPLANATION_CUES = {
    "because","since","due to","this is because","the reason","this occurs",
    "this happens","leads to","results in","causes","as a result of","which means",
    "which causes","this allows","enabling","mechanism","process by which",
    "works by","functions by","is achieved by","this process","by which",
    "this is why","the way this works","this enables","allows for",
}  # restored: Cell 35 role classifier needs this
CONTRAST_KW = {
    "but","however","although","despite","yet","whereas",
    "nevertheless","on the other hand","in contrast","unlike",
}
CLAIM_CUES = {
    "is defined as","is a ","are a ","refers to","can be defined",
    "is the process","is an example","is characterised by",
    "is known as","is called","is the ability","is the tendency",
    "means that","is described as","is understood as",
}

def depth_to_stars(depth):
    """
    CANONICAL depth→star mapping (0–5).
    DO NOT redefine in other cells — call this function everywhere.
    0★ = no reasoning  |  5★ = full reasoning chain (claim+expl+evidence+support+conclusion)
    """
    if depth is None:  return 0
    if depth < 0.10:   return 0
    if depth < 0.30:   return 1
    if depth < 0.50:   return 2
    if depth < 0.70:   return 3
    if depth < 0.90:   return 4
    return 5

def reasoning_level_from_depth(depth):
    """CANONICAL reasoning level from depth_score."""
    if depth is None:    return "Weak"
    if depth >= 0.80:    return "Strong"
    if depth >= 0.50:    return "Moderate"
    return "Weak"

def assign_grade(score):
    """ASAG grade for DistilBERT training stratification."""
    if score is None or not isinstance(score, (int, float)): return "low"
    if score >= 8:   return "high"
    if score >= 5:   return "medium"
    return "low"

def _cached(path):
    """True if file exists and has content."""
    return os.path.exists(path) and os.path.getsize(path) > 10

def sent_split(text):
    """Split text into sentences, filter empties."""
    return [s.strip() for s in
            str(text).replace("!",".").replace("?",".").split(".")
            if s.strip() and len(s.strip()) > 3]

# ── Star CSV file paths ───────────────────────────────────────────────────────
# STAR_FILE_MAP: keys = (csv_path, source_dataset_label)
# Filenames match actual Drive files in raw_datasets/
STAR_FILE_MAP = {
    0: (os.path.join(RAW_DIR, "0star.csv"), "curated_0star"),
    1: (os.path.join(RAW_DIR, "1star.csv"), "curated_1star"),
    2: (os.path.join(RAW_DIR, "2star.csv"), "curated_2star"),
    3: (os.path.join(RAW_DIR, "3star.csv"), "curated_3star"),
    4: (os.path.join(RAW_DIR, "4star.csv"), "curated_4star"),
    5: (os.path.join(RAW_DIR, "5star.csv"), "curated_5star"),
}


# ── Schema helpers (available to both Section 5 and Section 4D) ───────────────
REQUIRED_COLS_4D = [
    "source_dataset", "dataset_purpose", "question", "question_type",
    "reference_answer", "student_answer", "normalized_score", "grade",
    "target_stars", "answer_word_count", "sent_count", "is_multi_sentence",
    "nli_premise", "nli_hypothesis", "nli_label",
    "reasoning_facts", "reasoning_chain"
]

def _ensure_schema(df, source_dataset=None, dataset_purpose=None):
    """Enforce uniform column schema.  Moved to Cell 4 so Section 5 can call it
    before Section 4D runs (avoids NameError on cached path)."""
    import re as _re
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=REQUIRED_COLS_4D)
    df = df.copy()
    if source_dataset is not None:
        if "source_dataset" not in df.columns:
            df["source_dataset"] = source_dataset
        else:
            df["source_dataset"] = df["source_dataset"].fillna(source_dataset)
    if dataset_purpose is not None:
        if "dataset_purpose" not in df.columns:
            df["dataset_purpose"] = dataset_purpose
        else:
            df["dataset_purpose"] = df["dataset_purpose"].fillna(dataset_purpose)
    for col in REQUIRED_COLS_4D:
        if col not in df.columns:
            df[col] = np.nan
    df["student_answer"] = df["student_answer"].fillna("").astype(str)
    df["answer_word_count"] = df["student_answer"].apply(lambda x: len(str(x).split()))
    if "count_sentences" in globals():
        df["sent_count"] = df["student_answer"].apply(count_sentences)
    else:
        df["sent_count"] = df["student_answer"].apply(
            lambda x: len([s for s in _re.split(r"(?<=[.!?])\s+", str(x)) if s.strip()]))
    df["is_multi_sentence"] = df["sent_count"].fillna(0).astype(int) >= 2
    return df[REQUIRED_COLS_4D]

print("\nCanonical constants loaded:")
print(f"  ROLE_LABELS       : {ROLE_LABELS}")
print(f"  depth_to_stars(0.6) = {depth_to_stars(0.6)}★")
print(f"  USE_REASONING_PIPELINE     = {USE_REASONING_PIPELINE}")
print(f"  USE_SCORE_PREDICTION_PIPELINE = {USE_SCORE_PREDICTION_PIPELINE}")
print(f"  STAR_FILE_MAP keys : {list(STAR_FILE_MAP.keys())}")



Canonical constants loaded:
  ROLE_LABELS       : ['claim', 'explanation', 'evidence', 'conclusion', 'irrelevant']
  depth_to_stars(0.6) = 3★
  USE_REASONING_PIPELINE     = True
  USE_SCORE_PREDICTION_PIPELINE = True
  STAR_FILE_MAP keys : [0, 1, 2, 3, 4, 5]


## Section 3 — Preprocessing Utilities

**Purpose:** Reusable text cleaning, normalisation, and sentence-splitting functions used throughout the pipeline.

**Functions defined:**

| Function | Task |
|---|---|
| `clean_text_basic(text)` | Unicode normalisation, URL removal, whitespace cleanup |
| `clean_text_lower(text)` | Lowercase + basic clean |
| `lemmatize_text(text)` | WordNet lemmatisation |
| `preprocess_answer(text)` | Full answer cleaning pipeline |
| `preprocess_question(text)` | Question cleaning (preserves case for type detection) |
| `normalize_score(score, min_s, max_s)` | Maps raw score to 0–10 scale |
| `split_into_sentences(text)` | spaCy-based sentence splitting (respects `MAX_SENTENCES`) |
| `count_sentences(text)` | Returns integer sentence count |
| `assign_grade(score)` | Maps 0–10 score → `high` / `medium` / `low` |
| `is_valid_answer(text)` | Checks answer is 3–500 words |

**Expected output:** `Preprocessing utilities ready.`

**Note:** `split_into_sentences` falls back to regex splitting if spaCy `nlp` is not yet loaded.


In [ ]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

STOPWORDS  = set(stopwords.words("english"))
KEEP_WORDS = {"not","no","nor","never","neither","without",
              "because","therefore","however","although","but",
              "more","less","most","least","very","too"}
STOPWORDS  = STOPWORDS - KEEP_WORDS
lemmatizer = WordNetLemmatizer()

def clean_text_basic(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    for src, tgt in [("\u2019","'"),("\u2018","'"),("\u201c",'"'),("\u201d",'"'),
                     ("\u2013","-"),("\u2014","-")]:
        text = text.replace(src, tgt)
    text = re.sub(r"[^\x20-\x7E]", " ", text)
    text = re.sub(r"http\S+|www\S+|\S+@\S+", "", text)
    return re.sub(r"\s+", " ", text).strip()

def clean_text_lower(text):
    text = clean_text_basic(text).lower()
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9\s\.\!\?\-]", " ", text)).strip()

def lemmatize_text(text):
    text = clean_text_lower(text)
    return " ".join(lemmatizer.lemmatize(t)
                    for t in re.findall(r"\b[a-z0-9]+\b", text) if len(t) > 1)

def preprocess_answer(text, remove_stops=False):
    text = lemmatize_text(text)
    if remove_stops:
        text = " ".join(w for w in text.split() if w not in STOPWORDS or w in KEEP_WORDS)
    return text.strip()

def preprocess_question(text):
    return clean_text_basic(text)

def normalize_score(score, min_s, max_s, target=10.0):
    try:
        score = float(score)
        if max_s == min_s: return 5.0
        return round(min(max(((score - min_s) / (max_s - min_s)) * target, 0.0), target), 4)
    except Exception: return None

def is_valid_answer(text, min_words=3, max_words=500):
    if not isinstance(text, str): return False
    n = len(text.strip().split())
    return min_words <= n <= max_words

def split_into_sentences(text):
    text = str(text).strip()
    if not text: return []
    doc = nlp(text)
    return [s.text.strip() for s in doc.sents if s.text.strip()]

def count_sentences(text):
    return max(1, len(split_into_sentences(str(text))))

# assign_grade() is defined canonically in Cell 4 — reused here.

print("Preprocessing utilities ready.")


Preprocessing utilities ready.


## Section 4 — Dataset Loading

### 4A — ASAG Auxiliary Datasets (`dataset_purpose = "auxiliary_scoring"`)

**Purpose:** Load four short-answer grading datasets used **exclusively** for DistilBERT regression training. These datasets have ground-truth correctness scores, which makes them suitable for supervised regression.

**Datasets loaded:**

| Dataset | Source | Score Type |
|---|---|---|
| SemEval 2013 Task 7 Beetle | HuggingFace | 5-way categorical label → numeric |
| Mohler | Local CSV | Human score 0–5 |
| SciEntsBank | HuggingFace / CSV | 5-way categorical or numeric |
| ASAP-SAS | Local CSV | Normalised grade 0–1 or raw points |

**Why ASAG is auxiliary only:** These datasets provide correctness scores, not reasoning depth labels. They train the quality estimation model only — not the depth scoring system.

**Expected output:** Row counts per dataset. Missing datasets are skipped gracefully.

**Error handling:** Each dataset is wrapped in `try/except`. A failed load prints a warning and continues.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Section 4A — ASAG Auxiliary Dataset Loading
# dataset_purpose = "auxiliary_scoring"
# Used ONLY for DistilBERT regression baseline. Not primary reasoning evidence.
# ══════════════════════════════════════════════════════════════════════════════

import urllib.request

# ── SemEval 2013 Task 7 Beetle ────────────────────────────────────────────────
semeval = None
try:
    semeval = load_dataset("Atomi/semeval_2013_task_7_beetle_5way")
    print("SemEval (Beetle) loaded:", {k: len(v) for k, v in semeval.items()})
except Exception as e:
    print(f"SemEval failed (safe skip): {e}")

# ── Mohler ────────────────────────────────────────────────────────────────────
mohler_raw    = None
_mohler_cache = os.path.join(RAW_DIR, "mohler.csv")
MOHLER_URL    = ("https://raw.githubusercontent.com/gsasikiran/"
                 "Comparative-Evaluation-of-Pretrained-Transfer-Learning-Models-on-ASAG/"
                 "master/comparative_evaluation_on_mohler_dataset/dataset/mohler_dataset_edited.csv")
try:
    if not _cached(_mohler_cache):
        urllib.request.urlretrieve(MOHLER_URL, _mohler_cache)
    mohler_raw = pd.read_csv(_mohler_cache)
    print("Mohler loaded:", mohler_raw.shape)
except Exception as e:
    print(f"Mohler failed (safe skip): {e}")

# ── SciEntsBank ───────────────────────────────────────────────────────────────
scienb           = None
SCIENB_LABEL_COL = None
for _hf_id in ["Atomi/semeval_2013_task_7_scientsbank_5way", "nkazi/SciEntsBank"]:
    try:
        scienb = load_dataset(_hf_id)
        _sample = scienb[list(scienb.keys())[0]][0]
        SCIENB_LABEL_COL = next(
            (c for c in ["label_5way", "label", "correct"] if c in _sample), None)
        print(f"SciEntsBank loaded ({_hf_id}), label col: {SCIENB_LABEL_COL}")
        break
    except Exception as e:
        print(f"SciEntsBank {_hf_id} failed: {e}")

if scienb is None:
    _scienb_cache = os.path.join(RAW_DIR, "scientsbank_train.csv")
    try:
        if not _cached(_scienb_cache):
            urllib.request.urlretrieve(
                "https://raw.githubusercontent.com/dbbrandt/"
                "short_answer_granding_capstone_project/master/data/SciEntsBank/train.csv",
                _scienb_cache)
        scienb = {"_csv": pd.read_csv(_scienb_cache)}
        print("SciEntsBank CSV fallback loaded.")
    except Exception as e:
        print(f"SciEntsBank CSV fallback failed: {e}")

# ── ASAP-SAS ──────────────────────────────────────────────────────────────────
asap_raw    = None
_asap_cache = os.path.join(RAW_DIR, "asap_sas_train.tsv")
try:
    if not _cached(_asap_cache):
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/dnleng/asap-sas/main/data/train.tsv",
            _asap_cache)
    asap_raw = pd.read_csv(_asap_cache, sep="\t", on_bad_lines="skip")
    print("ASAP-SAS loaded:", asap_raw.shape)
except Exception as e:
    print(f"ASAP-SAS URL failed: {e} — trying Meyerger/ASAG2024")
    try:
        _hf = load_dataset("Meyerger/ASAG2024",
                           data_files={"train":     "train.parquet",
                                       "validation": "validation.parquet",
                                       "test":       "test.parquet"})
        asap_raw = _hf["train"].to_pandas()
        print("ASAP-SAS (Meyerger/ASAG2024) loaded:", asap_raw.shape)
    except Exception as e2:
        print(f"ASAP-SAS HF fallback failed: {e2}")

print("\nSection 4A complete.")
print(f"  SemEval    : {'loaded' if semeval   is not None else 'SKIPPED'}")
print(f"  Mohler     : {'loaded' if mohler_raw is not None else 'SKIPPED'}")
print(f"  SciEntsBank: {'loaded' if scienb    is not None else 'SKIPPED'}")
print(f"  ASAP-SAS   : {'loaded' if asap_raw  is not None else 'SKIPPED'}")

README.md:   0%|          | 0.00/4.28k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  159kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 43.2kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

test.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

ASAP-SAS (Meyerger/ASAG2024) loaded: (15190, 8)

Section 4A complete.
  SemEval    : loaded
  Mohler     : loaded
  SciEntsBank: loaded
  ASAP-SAS   : loaded


### 4B — Reasoning Datasets (`nli_reasoning` / `multi_hop_reasoning`)

**Purpose:** Load NLI and multi-hop datasets for graph edge evaluation — **not** for DistilBERT training.

**Datasets loaded:**

| Dataset | Purpose | Samples |
|---|---|---|
| SNLI | NLI graph edge evaluation | 5,000 |
| MultiNLI | Cross-genre NLI evaluation | 5,000 |
| QASC / eQASC | Multi-hop reasoning chains | 3,000 |

**Why these are separate:** SNLI/MNLI provide sentence-pair logical relation examples (entailment / neutral / contradiction). QASC provides multi-fact chaining. Neither has per-answer quality scores, so they cannot enter DistilBERT regression.

**Expected output:** Load confirmations and label distributions per dataset.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Section 4B — Reasoning Datasets (NLI + QASC)
# FIX: per-split existence check; missing splits logged, not crashed.
# Outputs: snli_raw, mnli_raw, qasc_raw (any may be None if unavailable)
# ══════════════════════════════════════════════════════════════════════════════

import os

snli_raw = None
mnli_raw = None
qasc_raw = None
_nli_warnings = []

def _log_nli_warn(msg):
    _nli_warnings.append(msg)
    print(f"  NLI WARNING: {msg}")
    try:
        with open(NLI_WARN_LOG, "a") as _f:
            _f.write(msg + "\n")
    except Exception:
        pass

# ── SNLI ─────────────────────────────────────────────────────────────────────
try:
    _snli_ds = load_dataset("stanfordnlp/snli")
    _parts = []
    for _split in ["train","validation","test"]:
        if _split in _snli_ds:
            _sub = _snli_ds[_split].filter(lambda x: x["label"] != -1)
            _parts.append(_sub.to_pandas().sample(min(5000,len(_sub.to_pandas())),
                                                   random_state=RANDOM_STATE))
            print(f"  SNLI {_split}: {len(_parts[-1])} rows loaded")
        else:
            _log_nli_warn(f"SNLI split '{_split}' not available — skipping")
    if _parts:
        import pandas as pd
        snli_raw = pd.concat(_parts, ignore_index=True)
        print(f"SNLI combined: {len(snli_raw)} rows")
    else:
        _log_nli_warn("SNLI: all splits missing — snli_raw=None")
except Exception as _e:
    _log_nli_warn(f"SNLI load failed: {_e} — snli_raw=None")

# ── MultiNLI ─────────────────────────────────────────────────────────────────
try:
    _mnli_ds = load_dataset("nyu-mll/multi_nli")
    _parts = []
    for _split in ["train","validation_matched","validation_mismatched"]:
        if _split in _mnli_ds:
            _sub = _mnli_ds[_split].to_pandas()
            _parts.append(_sub.sample(min(5000,len(_sub)), random_state=RANDOM_STATE))
            print(f"  MultiNLI {_split}: {len(_parts[-1])} rows loaded")
        else:
            _log_nli_warn(f"MultiNLI split '{_split}' not available — skipping")
    if _parts:
        import pandas as pd
        mnli_raw = pd.concat(_parts, ignore_index=True)
        print(f"MultiNLI combined: {len(mnli_raw)} rows")
    else:
        _log_nli_warn("MultiNLI: all splits missing — mnli_raw=None")
except Exception as _e:
    _log_nli_warn(f"MultiNLI load failed: {_e} — mnli_raw=None")

# ── QASC ─────────────────────────────────────────────────────────────────────
try:
    _qasc_ds = load_dataset("allenai/qasc")
    _parts = []
    for _split in ["train","validation","test"]:
        if _split in _qasc_ds:
            _parts.append(_qasc_ds[_split].to_pandas())
            print(f"  QASC {_split}: {len(_parts[-1])} rows")
        else:
            _log_nli_warn(f"QASC split '{_split}' not available — skipping")
    if _parts:
        import pandas as pd
        qasc_raw = pd.concat(_parts, ignore_index=True)
        print(f"QASC combined: {len(qasc_raw)} rows")
    else:
        _log_nli_warn("QASC: all splits missing — qasc_raw=None")
except Exception as _e:
    _log_nli_warn(f"QASC load failed: {_e} — qasc_raw=None")

print(f"\nNLI load summary:")
print(f"  snli_raw  : {len(snli_raw) if snli_raw is not None else 'None'}")
print(f"  mnli_raw  : {len(mnli_raw) if mnli_raw is not None else 'None'}")
print(f"  qasc_raw  : {len(qasc_raw) if qasc_raw is not None else 'None'}")
if _nli_warnings:
    print(f"  {len(_nli_warnings)} warning(s) logged to {NLI_WARN_LOG}")
else:
    print("  No NLI warnings.")
logger.info(f"NLI loading done. SNLI={snli_raw is not None} MNLI={mnli_raw is not None} QASC={qasc_raw is not None}")


README.md:   0%|          | 0.00/16.0k [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  412kB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  413kB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

...

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8134 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/920 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/926 [00:00<?, ? examples/s]

  QASC train: 8134 rows
  QASC validation: 926 rows
  QASC test: 920 rows
QASC combined: 9980 rows

NLI load summary:
  snli_raw  : 15000
  mnli_raw  : 15000
  qasc_raw  : 9980
  No NLI warnings.


### 4C — Curated 0–5★ Reasoning Corpus (`curated_demo`)

**Purpose:** Load the hand-curated reasoning depth corpus — six CSV files labelled by star level.

**Required files in `RAW_DIR`:** `0star.csv`, `1star.csv`, `2star.csv`, `3star.csv`, `4star.csv`, `5star.csv`

**Expected CSV columns:** `question`, `student_answer`, `reference_answer`, `question_type`

**Key properties:**
- `target_stars` is set from the filename (0–5) — this is the ground-truth reasoning depth label
- `normalized_score = NaN` — these rows **never** enter DistilBERT splits
- 3★–5★ rows are multi-sentence by design → eligible for NLI graph analysis
- 0★–1★ rows are short or weak answers → serve as negative examples

**Why this corpus is needed:** ASAG datasets contain only short, single-sentence answers with correctness labels — not multi-sentence reasoning chains. The curated corpus provides the reasoning-depth variation needed to evaluate the star prediction system.

**Expected output:** Row counts per star level and total loaded.

**Strength:** Controlled corpus with clear ground-truth depth labels.
**Weakness:** Labels reflect the project team's reasoning criteria. Independent annotation is recommended for stronger validity.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Section 4C — Curated Demo Dataset Loading from Drive CSVs
# Files: 0star.csv … 5star.csv
# dataset_purpose = "curated_demo"
# target_stars = ground-truth reasoning depth label (0–5)
# normalized_score = None → NEVER enters DistilBERT regression
# ══════════════════════════════════════════════════════════════════════════════

import os
import pandas as pd
import numpy as np

# BASE_DIR / RAW_DIR defined canonically in Cell 4 — reused here.

CURATED_CSV_MAP = {
    0: os.path.join(RAW_DIR, "0star.csv"),
    1: os.path.join(RAW_DIR, "1star.csv"),
    2: os.path.join(RAW_DIR, "2star.csv"),
    3: os.path.join(RAW_DIR, "3star.csv"),
    4: os.path.join(RAW_DIR, "4star.csv"),
    5: os.path.join(RAW_DIR, "5star.csv"),
}

CURATED_DEMO = []
HUMAN_EVAL_LABELS = {}

# _cached() is defined canonically in Cell 4 — reused here.

def _find_col(df_cols, candidates):
    df_cols_lower = {str(c).lower().strip(): c for c in df_cols}
    for c in candidates:
        key = c.lower().strip()
        if key in df_cols_lower:
            return df_cols_lower[key]
    return None

CURATED_EXPECTED_COLS = {
    "question": ["question", "Question", "QUESTION"],
    "student_answer": ["student_answer", "answer", "Answer", "student answer", "response", "Response"],
    "reference_answer": ["reference_answer", "reference answer", "model_answer", "model answer", "reference", "Reference", "ideal_answer"],
    "question_type": ["question_type", "question type", "type", "Type"],
}

print("Checking curated CSV paths:")
for star, path in CURATED_CSV_MAP.items():
    exists = "✓ found" if _cached(path) else "✗ NOT FOUND"
    size = f"  {os.path.getsize(path)//1024} KB" if os.path.exists(path) else ""
    print(f"  {star}★  {os.path.basename(path):<12} [{exists}]{size}")

def _load_curated_star_csv(path, star_level):
    if not _cached(path):
        print(f"  ✗ {star_level}★ — missing/empty: {path}")
        return pd.DataFrame()

    raw = pd.read_csv(path, encoding="utf-8", on_bad_lines="skip")
    if raw.empty:
        print(f"  ✗ {star_level}★ — CSV parsed but empty")
        return pd.DataFrame()

    q_col = _find_col(raw.columns, CURATED_EXPECTED_COLS["question"])
    stu_col = _find_col(raw.columns, CURATED_EXPECTED_COLS["student_answer"])
    ref_col = _find_col(raw.columns, CURATED_EXPECTED_COLS["reference_answer"])
    qt_col = _find_col(raw.columns, CURATED_EXPECTED_COLS["question_type"])

    if not stu_col:
        print(f"  ✗ {star_level}★ — no student_answer column. Columns: {list(raw.columns)}")
        return pd.DataFrame()

    rows = []
    for _, row in raw.iterrows():
        stu = str(row.get(stu_col, "")).strip()
        if not stu or len(stu.split()) < 3:
            continue

        q = str(row.get(q_col, "educational question")).strip() if q_col else "educational question"
        ref = str(row.get(ref_col, stu)).strip() if ref_col else stu
        qt = str(row.get(qt_col, "explanation")).lower().strip() if qt_col else "explanation"

        rows.append({
            "source_dataset": f"curated_{star_level}star",
            "dataset_purpose": "curated_demo",
            "question": q,
            "question_type": qt,
            "reference_answer": ref,
            "student_answer": stu,
            "normalized_score": np.nan,
            "grade": np.nan,
            "target_stars": int(star_level),
            "nli_premise": None,
            "nli_hypothesis": None,
            "nli_label": None,
            "reasoning_facts": None,
            "reasoning_chain": None,
        })

    out = pd.DataFrame(rows)

    out["answer_word_count"] = out["student_answer"].apply(lambda x: len(str(x).split()))

    if "count_sentences" in globals():
        out["sent_count"] = out["student_answer"].apply(count_sentences)
    else:
        import re
        out["sent_count"] = out["student_answer"].apply(
            lambda x: len([s for s in re.split(r"(?<=[.!?])\s+", str(x)) if s.strip()])
        )

    out["is_multi_sentence"] = out["sent_count"] >= 2

    print(
        f"  ✓ {star_level}★  {len(out):>5,} rows  "
        f"multi={out['is_multi_sentence'].sum():>5,}  "
        f"file={os.path.basename(path)}"
    )

    return out

print("\nLoading star-rated curated CSVs...")
curated_parts = []

for star_level, path in CURATED_CSV_MAP.items():
    part = _load_curated_star_csv(path, star_level)
    if not part.empty:
        curated_parts.append(part)

curated_df = pd.concat(curated_parts, ignore_index=True) if curated_parts else pd.DataFrame()

print("\n" + "═" * 70)
print("CURATED DEMO DATASET SUMMARY")
print("═" * 70)
print(f"Curated demo total: {len(curated_df):,} rows")

if not curated_df.empty:
    print("\nStars breakdown:")
    for st, cnt in sorted(curated_df["target_stars"].value_counts().items()):
        multi = curated_df[curated_df["target_stars"] == st]["is_multi_sentence"].sum()
        print(f"  {int(st)}★  {cnt:>5,} rows  multi={multi:>5,}")

    print(
        f"\nMulti-sentence total: {curated_df['is_multi_sentence'].sum():,} "
        f"({curated_df['is_multi_sentence'].mean()*100:.1f}%)"
    )

    print("\nSample row:")
    print("Q:", curated_df["question"].iloc[0][:120])
    print("A:", curated_df["student_answer"].iloc[0][:160])

else:
    print("⚠ curated_df is EMPTY.")
    print("RAW_DIR used:", RAW_DIR)
    print("Files available:")
    try:
        for f in sorted(os.listdir(RAW_DIR)):
            print(" ", f)
    except Exception as e:
        print("Cannot list RAW_DIR:", e)

print("\nSection 4C complete.")

Checking curated CSV paths:
  0★  0star.csv    [✓ found]  794 KB
  1★  1star.csv    [✓ found]  784 KB
  2★  2star.csv    [✓ found]  589 KB
  3★  3star.csv    [✓ found]  718 KB
  4★  4star.csv    [✓ found]  995 KB
  5★  5star.csv    [✓ found]  951 KB

Loading star-rated curated CSVs...
  ✓ 0★  1,000 rows  multi=1,000  file=0star.csv
  ✓ 1★  1,050 rows  multi=1,050  file=1star.csv
  ✓ 2★  1,000 rows  multi=1,000  file=2star.csv
  ✓ 3★  1,000 rows  multi=1,000  file=3star.csv
  ✓ 4★  1,050 rows  multi=1,050  file=4star.csv
  ✓ 5★  1,050 rows  multi=1,050  file=5star.csv

══════════════════════════════════════════════════════════════════════
CURATED DEMO DATASET SUMMARY
══════════════════════════════════════════════════════════════════════
Curated demo total: 6,150 rows

Stars breakdown:
  0★  1,000 rows  multi=1,000
  1★  1,050 rows  multi=1,050
  2★  1,000 rows  multi=1,000
  3★  1,000 rows  multi=1,000
  4★  1,050 rows  multi=1,050
  5★  1,050 rows  multi=1,050

Multi-sentence total: 6,

## Section 5 — Dataset Preprocessing and Schema Standardisation

**Purpose:** Apply uniform column naming, score normalisation, and quality filtering across all datasets.

**What this cell does:**
- Maps dataset-specific label columns → `normalized_score` (0–10 scale) using `SEMEVAL_LABEL_MAP`
- Applies `is_valid_answer` filter (3–500 words)
- Computes `answer_word_count`, `sent_count`, `is_multi_sentence` for all rows
- Sets `question_type = "general"` where not detected
- Preserves `target_stars` and `normalized_score = NaN` for curated rows

**Why normalise to 0–10?** Different ASAG datasets use incompatible raw scales (SemEval: categorical, Mohler: 0–5, ASAP: varies). Normalising to a shared 0–10 scale gives DistilBERT consistent regression targets.

**Expected output:** Shape and dtype summary. DataFrames saved to CSV.

**Strength:** Unified schema prevents column mismatch errors downstream.
**Weakness:** Categorical-to-numeric mapping (e.g. SemEval labels) is heuristic.


> **Repair note:** this section moved ahead of *Section 4D — Dataset Combination*. Section 4D's merge reads the `combined_df` that Section 5 builds; original order (4D→5) produced an empty `auxiliary_df` on fresh runs.


In [ ]:
# ── FIX: combined_df must reference combined_all_df (full dataset) ───────────
# Cell 14 defines combined_all_df with all dataset_purposes.
# combined_df here is set to the full merged dataframe so all
# downstream cells (auxiliary + reasoning) have access to all rows.
if "combined_all_df" in dir():
    combined_df = combined_all_df.copy()
    print(f"combined_df set from combined_all_df: {len(combined_df):,} rows")
    print(f"dataset_purpose breakdown: {combined_df['dataset_purpose'].value_counts().to_dict()}")
else:
    print("WARNING: combined_all_df not yet defined — run Section 4 first.")

import os
import numpy as np
import pandas as pd

SEMEVAL_LABEL_MAP = {
    "correct": 10,
    "partially_correct_incomplete": 5,
    "contradictory": 1,
    "irrelevant": 0,
    "non_domain": 0,
}
NLI_INT_MAP = {0: "entailment", 1: "neutral", 2: "contradiction"}

SHARED_COLS = [
    "row_id", "source_dataset", "dataset_purpose",
    "question", "question_type", "reference_answer", "student_answer",
    "normalized_score", "grade",
    "answer_word_count", "sent_count", "is_multi_sentence",
    "nli_premise", "nli_hypothesis", "nli_label",
    "reasoning_facts", "reasoning_chain",
]

def _empty_nli():
    return {"nli_premise": None, "nli_hypothesis": None, "nli_label": None}
def _empty_hop():
    return {"reasoning_facts": None, "reasoning_chain": None}

if not FORCE_REBUILD_DATASET and _cached(COMBINED_CSV):
    print("Loading ASAG combined_df from cache...")
    combined_df = pd.read_csv(COMBINED_CSV)
    # Ensure all required columns are present for cached data
    # Use the _ensure_schema function defined in Section 4D (ollCD5FjUYVI)
    combined_df = _ensure_schema(combined_df, dataset_purpose="auxiliary_scoring")
    # Also ensure row_id is present if not already, and is the index
    if "row_id" not in combined_df.columns:
        combined_df.insert(0, "row_id", combined_df.index)
    print("Loaded and schema ensured:", combined_df.shape)
else:
    print("Building ASAG combined_df from raw sources...")

    # ── SemEval ───────────────────────────────────────────────────────────────
    semeval_rows = []
    if semeval:
        for split in semeval.keys():
            for row in semeval[split]:
                q   = preprocess_question(row.get("question", ""))
                ref = preprocess_answer(row.get("reference_answer", ""))
                stu = preprocess_answer(row.get("student_answer", ""))
                lbl = str(row.get("label_5way", row.get("label", ""))).lower()
                score = SEMEVAL_LABEL_MAP.get(lbl)
                if q and ref and stu and score is not None:
                    semeval_rows.append({
                        "source_dataset": "semeval_beetle",
                        "dataset_purpose": "auxiliary_scoring",
                        "question": q, "reference_answer": ref,
                        "student_answer": stu, "normalized_score": score,
                        **_empty_nli(), **_empty_hop(),
                    })
    semeval_df = pd.DataFrame(semeval_rows)
    print(f"  SemEval : {semeval_df.shape}")

    # ── Mohler ────────────────────────────────────────────────────────────────
    mohler_rows = []
    if mohler_raw is not None:
        for _, row in mohler_raw.iterrows():
            q    = preprocess_question(row.get("question", ""))
            ref  = preprocess_answer(row.get("desired_answer", row.get("reference_answer", "")))
            stu  = preprocess_answer(row.get("student_answer", ""))
            score= row.get("score_avg", np.nan)
            if pd.notna(score) and q and ref and stu:
                mohler_rows.append({
                    "source_dataset": "mohler",
                    "dataset_purpose": "auxiliary_scoring",
                    "question": q, "reference_answer": ref,
                    "student_answer": stu,
                    "normalized_score": min(max(float(score), 0), 10),
                    **_empty_nli(), **_empty_hop(),
                })
    mohler_df = pd.DataFrame(mohler_rows)
    print(f"  Mohler  : {mohler_df.shape}")

    # ── SciEntsBank ───────────────────────────────────────────────────────────
    scienb_rows = []
    if scienb is not None:
        def _scienb_score(raw_lbl):
            s = SEMEVAL_LABEL_MAP.get(str(raw_lbl).lower())
            if s is not None: return s
            try: return min(max(float(raw_lbl) * 10, 0), 10)
            except: return None
        if "_csv" in scienb:
            _df = scienb["_csv"]
            cols = list(_df.columns)
            q_c  = next((c for c in cols if "question"  in c.lower()), None)
            ref_c= next((c for c in cols if "reference" in c.lower()), None)
            stu_c= next((c for c in cols if "student"   in c.lower() or "answer" in c.lower()), None)
            lbl_c= next((c for c in cols if "label"     in c.lower() or "score"  in c.lower()), None)
            for _, row in _df.iterrows():
                q   = preprocess_question(str(row.get(q_c,""))) if q_c else ""
                ref = preprocess_answer(str(row.get(ref_c,""))) if ref_c else ""
                stu = preprocess_answer(str(row.get(stu_c,""))) if stu_c else ""
                sc  = _scienb_score(row.get(lbl_c,"")) if lbl_c else None
                if q and ref and stu and sc is not None:
                    scienb_rows.append({
                        "source_dataset": "scientsbank",
                        "dataset_purpose": "auxiliary_scoring",
                        "question": q, "reference_answer": ref,
                        "student_answer": stu, "normalized_score": sc,
                        **_empty_nli(), **_empty_hop(),
                    })
        else:
            for split in scienb.keys():
                for row in scienb[split]:
                    q   = preprocess_question(row.get("question", ""))
                    ref = preprocess_answer(row.get("reference_answer", ""))
                    stu = preprocess_answer(row.get("student_answer", ""))
                    raw = row.get(SCIENB_LABEL_COL, "") if SCIENB_LABEL_COL else ""
                    sc  = _scienb_score(raw)
                    if q and ref and stu and sc is not None:
                        scienb_rows.append({
                            "source_dataset": "scientsbank",
                            "dataset_purpose": "auxiliary_scoring",
                            "question": q, "reference_answer": ref,
                            "student_answer": stu, "normalized_score": sc,
                            **_empty_nli(), **_empty_hop(),
                        })
    scienb_df = pd.DataFrame(scienb_rows)
    print(f"  SciEntsBank: {scienb_df.shape}")

    # ── ASAP-SAS ──────────────────────────────────────────────────────────────
    asap_rows = []
    if asap_raw is not None:
        cols    = list(asap_raw.columns)
        stu_col = next((c for c in cols if any(k in c.lower()
                        for k in ["provided_answer","essay","student","response","answer"])), None)
        sc_col  = next((c for c in cols if "normalized_grade" in c.lower()), None)
        if not sc_col: sc_col = next((c for c in cols
            if "score" in c.lower() and "domain" not in c.lower()), None)
        if not sc_col: sc_col = next((c for c in cols if "grade" in c.lower()), None)
        q_col   = next((c for c in cols if "question"  in c.lower()), None)
        ref_col = next((c for c in cols if "reference" in c.lower() or "answer_key" in c.lower()), None)
        if stu_col and sc_col:
            score_max   = asap_raw[sc_col].dropna().max()
            norm_factor = (10.0 if score_max <= 1.01
                           else 10.0/5.0 if score_max <= 5.0 else 10.0/3.0)
            for _, row in asap_raw.iterrows():
                stu   = preprocess_answer(str(row.get(stu_col, "")))
                raw_s = row.get(sc_col, np.nan)
                q_txt = preprocess_question(str(row.get(q_col,"question"))) if q_col else "question"
                ref   = preprocess_answer(str(row.get(ref_col, stu))) if ref_col else stu
                if pd.notna(raw_s) and stu:
                    try:
                        norm = min(max(float(raw_s) * norm_factor, 0), 10)
                        asap_rows.append({
                            "source_dataset": "asap_sas",
                            "dataset_purpose": "auxiliary_scoring",
                            "question": q_txt, "reference_answer": ref,
                            "student_answer": stu, "normalized_score": norm,
                            **_empty_nli(), **_empty_hop(),
                        })
                    except: pass
    asap_df = pd.DataFrame(asap_rows)
    print(f"  ASAP-SAS: {asap_df.shape}")

    # ── Combine ASAG only ─────────────────────────────────────────────────────
    dfs = [d for d in [semeval_df, mohler_df, scienb_df, asap_df] if not d.empty]
    combined_df = pd.concat(dfs, ignore_index=True)
    combined_df = combined_df.dropna(subset=["question","reference_answer",
                                              "student_answer","normalized_score"])
    combined_df = combined_df[combined_df["student_answer"].str.split().str.len() >= 3]
    combined_df = combined_df.drop_duplicates(subset=["question","student_answer"])
    combined_df["normalized_score"]  = combined_df["normalized_score"].clip(0, 10)
    combined_df["answer_word_count"] = combined_df["student_answer"].apply(
        lambda x: len(str(x).split()))
    print("  Computing sentence counts for ASAG...")
    combined_df["sent_count"]        = combined_df["student_answer"].apply(count_sentences)
    combined_df["is_multi_sentence"] = combined_df["sent_count"] >= 2
    combined_df["question_type"]     = "general"
    combined_df["grade"]             = combined_df["normalized_score"].apply(assign_grade)
    combined_df = combined_df.reset_index(drop=True)
    combined_df.insert(0, "row_id", combined_df.index)
    # Ensure 'dataset_purpose' is set when building
    combined_df["dataset_purpose"] = "auxiliary_scoring"
    combined_df.to_csv(COMBINED_CSV, index=False)
    print(f"  ASAG combined_df saved: {COMBINED_CSV}")

# ── NLI → nli_reasoning ───────────────────────────────────────────────────────
def _process_nli(df_raw, source_name):
    rows = []
    if df_raw is None or df_raw.empty:
        return pd.DataFrame()
    for _, row in df_raw.iterrows():
        prem = str(row.get("premise", "")).strip()
        hyp  = str(row.get("hypothesis", "")).strip()
        lbl  = NLI_INT_MAP.get(int(row.get("label", 1)), "neutral")
        if not prem or not hyp: continue
        stu_ans = f"Premise: {prem}. Hypothesis: {hyp}."
        ref_ans = f"The relationship between the premise and hypothesis is {lbl}."
        rows.append({
            "source_dataset":  source_name,
            "dataset_purpose": "nli_reasoning",
            "question":        "Does the premise support, contradict, or remain neutral to the hypothesis?",
            "question_type":   "entailment",
            "reference_answer": ref_ans,
            "student_answer":   stu_ans,
            "normalized_score": None,   # NOT used for regression
            "grade":            None,
            "answer_word_count": len(stu_ans.split()),
            "nli_premise":      prem,
            "nli_hypothesis":   hyp,
            "nli_label":        lbl,
            "reasoning_facts":  None,
            "reasoning_chain":  None,
        })
    out = pd.DataFrame(rows)
    out["sent_count"]        = out["student_answer"].apply(count_sentences)
    out["is_multi_sentence"] = out["sent_count"] >= 2
    return out

snli_df = _process_nli(snli_raw, "snli")
mnli_df = _process_nli(mnli_raw, "mnli")
print(f"NLI processed — SNLI: {snli_df.shape}  MultiNLI: {mnli_df.shape}")

# ── QASC → multi_hop_reasoning ────────────────────────────────────────────────
qasc_rows = []
if qasc_raw is not None and not qasc_raw.empty:
    cols = list(qasc_raw.columns)
    print(f"  QASC columns: {cols}")
    q_col    = next((c for c in cols if "question" in c.lower()), None)
    ans_col  = next((c for c in cols if c.lower() in ["answerkey","correct_answer","answer"]), None)
    f1_col   = next((c for c in cols if "fact1" in c.lower() or "supporting1" in c.lower()), None)
    f2_col   = next((c for c in cols if "fact2" in c.lower() or "supporting2" in c.lower()), None)
    for _, row in qasc_raw.iterrows():
        q    = str(row.get(q_col, "")) if q_col else ""
        f1   = str(row.get(f1_col,"")) if f1_col else ""
        f2   = str(row.get(f2_col,"")) if f2_col else ""
        ans  = str(row.get(ans_col,"")) if ans_col else ""
        if not q or not f1: continue
        facts = "; ".join([f for f in [f1, f2] if f.strip()])
        stu   = f"{f1.strip()} {f2.strip()} Therefore, {ans.strip()}.".strip()
        ref   = f"Using the facts: {facts}. The answer is: {ans.strip()}"
        qasc_rows.append({
            "source_dataset":  "qasc",
            "dataset_purpose": "multi_hop_reasoning",
            "question":        preprocess_question(q),
            "question_type":   "multi_hop",
            "reference_answer": ref,
            "student_answer":   stu,
            "normalized_score": None,
            "grade":            None,
            "answer_word_count": len(stu.split()),
            "nli_premise":      None,
            "nli_hypothesis":   None,
            "nli_label":        None,
            "reasoning_facts":  facts,
            "reasoning_chain":  stu,
        })
qasc_df = pd.DataFrame(qasc_rows)
if not qasc_df.empty:
    qasc_df["sent_count"]        = qasc_df["student_answer"].apply(count_sentences)
    qasc_df["is_multi_sentence"] = qasc_df["sent_count"] >= 2
print(f"QASC processed: {qasc_df.shape}")

print("Dataset merging complete. Curated star-rated CSVs loaded in Section 4B.")

Building ASAG combined_df from raw sources...
  SemEval : (12072, 11)
  Mohler  : (2264, 11)
  SciEntsBank: (10803, 11)
  ASAP-SAS: (15158, 11)
  Computing sentence counts for ASAG...
  ASAG combined_df saved: /content/drive/MyDrive/FYP_Data/outputs/combined_asag_df.csv
NLI processed — SNLI: (15000, 16)  MultiNLI: (15000, 16)
  QASC columns: ['id', 'question', 'choices', 'answerKey', 'fact1', 'fact2', 'combinedfact', 'formatted_question']
QASC processed: (9060, 16)
Dataset merging complete. Curated star-rated CSVs loaded in Section 4B.


### 4D — Dataset Combination and Audit

**Purpose:** Merge all datasets into a unified schema with `dataset_purpose` tags. Print a full audit before proceeding.

**Output DataFrames:**
- `combined_all_df` — all rows, all purposes
- `auxiliary_df` — ASAG rows only (for DistilBERT splits)
- `reasoning_df` — SNLI / MNLI / QASC rows
- `reasoning_eval_df` — subset for NLI graph evaluation

**Schema (all DataFrames):**
`row_id | source_dataset | dataset_purpose | question | question_type | reference_answer | student_answer | normalized_score | grade | answer_word_count | sent_count | is_multi_sentence | nli_premise | nli_hypothesis | nli_label | reasoning_facts | reasoning_chain`

**Expected output:** Row counts per purpose, multi-sentence coverage, NLI label distribution, star distribution.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Section 4D — Dataset Merge, Full Audit & Visualisations
# FIX: reasoning_df is built fresh from snli/mnli/qasc/curated_demo every run.
#      It is NEVER overwritten by auxiliary_scoring rows.
#      curated_demo rows are NEVER written into auxiliary_df.
# ══════════════════════════════════════════════════════════════════════════════

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NLI_INT_MAP = {0: "entailment", 1: "neutral", 2: "contradiction"}

REQUIRED_COLS_4D = [
    "source_dataset", "dataset_purpose", "question", "question_type",
    "reference_answer", "student_answer", "normalized_score", "grade",
    "target_stars", "answer_word_count", "sent_count", "is_multi_sentence",
    "nli_premise", "nli_hypothesis", "nli_label",
    "reasoning_facts", "reasoning_chain"
]

# _ensure_schema already defined in Cell 4; this re-definition is identical
# and harmless — kept as safety net for out-of-order execution.
def _ensure_schema(df, source_dataset=None, dataset_purpose=None):
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=REQUIRED_COLS_4D)
    df = df.copy()
    if source_dataset is not None:
        df["source_dataset"] = df.get("source_dataset", pd.Series(dtype=str)).fillna(source_dataset)
        if "source_dataset" not in df.columns:
            df["source_dataset"] = source_dataset
    if dataset_purpose is not None:
        if "dataset_purpose" not in df.columns:
            df["dataset_purpose"] = dataset_purpose
        else:
            df["dataset_purpose"] = df["dataset_purpose"].fillna(dataset_purpose)
    for col in REQUIRED_COLS_4D:
        if col not in df.columns:
            df[col] = np.nan
    df["student_answer"] = df["student_answer"].fillna("").astype(str)
    df["answer_word_count"] = df["student_answer"].apply(lambda x: len(str(x).split()))
    if "count_sentences" in globals():
        df["sent_count"] = df["student_answer"].apply(count_sentences)
    else:
        import re
        df["sent_count"] = df["student_answer"].apply(
            lambda x: len([s for s in re.split(r"(?<=[.!?])\s+", str(x)) if s.strip()]))
    df["is_multi_sentence"] = df["sent_count"].fillna(0).astype(int) >= 2
    return df[REQUIRED_COLS_4D]

def _process_nli_df(df_raw, source_name):
    if df_raw is None:
        return pd.DataFrame(columns=REQUIRED_COLS_4D)
    if hasattr(df_raw, "to_pandas"):
        df_raw = df_raw.to_pandas()
    rows = []
    for _, row in df_raw.iterrows():
        prem = str(row.get("premise", "")).strip()
        hyp  = str(row.get("hypothesis", "")).strip()
        try:
            lbl = NLI_INT_MAP.get(int(row.get("label", 1)), "neutral")
        except Exception:
            lbl = str(row.get("label", "neutral"))
        if not prem or not hyp:
            continue
        stu = f"Premise: {prem}. Hypothesis: {hyp}."
        ref = f"The relationship between the premise and hypothesis is {lbl}."
        rows.append({
            "source_dataset": source_name,
            "dataset_purpose": "nli_reasoning",
            "question": "Does the premise support, contradict, or remain neutral to the hypothesis?",
            "question_type": "entailment",
            "reference_answer": ref,
            "student_answer": stu,
            "normalized_score": np.nan, "grade": np.nan, "target_stars": np.nan,
            "answer_word_count": len(stu.split()),
            "nli_premise": prem, "nli_hypothesis": hyp, "nli_label": lbl,
            "reasoning_facts": None, "reasoning_chain": None,
        })
    out = pd.DataFrame(rows)
    if not out.empty:
        out["sent_count"] = out["student_answer"].apply(count_sentences)
        out["is_multi_sentence"] = out["sent_count"] >= 2
    return _ensure_schema(out)

def _process_qasc_df(df_raw):
    if df_raw is None:
        return pd.DataFrame(columns=REQUIRED_COLS_4D)
    if hasattr(df_raw, "to_pandas"):
        df_raw = df_raw.to_pandas()
    cols = list(df_raw.columns)
    q_col   = next((c for c in cols if "question" in c.lower()), None)
    ans_col = next((c for c in cols if c.lower() in ["answerkey","correct_answer","answer"]), None)
    f1_col  = next((c for c in cols if "fact1" in c.lower() or "supporting1" in c.lower()), None)
    f2_col  = next((c for c in cols if "fact2" in c.lower() or "supporting2" in c.lower()), None)
    rows = []
    for _, row in df_raw.iterrows():
        q   = str(row.get(q_col,  "")) if q_col   else ""
        f1  = str(row.get(f1_col, "")) if f1_col  else ""
        f2  = str(row.get(f2_col, "")) if f2_col  else ""
        ans = str(row.get(ans_col,"")) if ans_col  else ""
        if not q or not f1:
            continue
        facts = "; ".join([f for f in [f1, f2] if f.strip()])
        stu   = f"{f1.strip()} {f2.strip()} Therefore, {ans.strip()}.".strip()
        ref   = f"Using the facts: {facts}. The answer is: {ans.strip()}"
        rows.append({
            "source_dataset": "qasc",
            "dataset_purpose": "multi_hop_reasoning",
            "question": preprocess_question(q) if "preprocess_question" in globals() else q,
            "question_type": "multi_hop",
            "reference_answer": ref,
            "student_answer": stu,
            "normalized_score": np.nan, "grade": np.nan, "target_stars": np.nan,
            "answer_word_count": len(stu.split()),
            "nli_premise": None, "nli_hypothesis": None, "nli_label": None,
            "reasoning_facts": facts, "reasoning_chain": stu,
        })
    out = pd.DataFrame(rows)
    if not out.empty:
        out["sent_count"] = out["student_answer"].apply(count_sentences)
        out["is_multi_sentence"] = out["sent_count"] >= 2
    return _ensure_schema(out)

# ── Build each sub-dataframe fresh ────────────────────────────────────────────
snli_df    = _process_nli_df(globals().get("snli_raw"), "snli")
mnli_df    = _process_nli_df(globals().get("mnli_raw"), "multi_nli")
qasc_df    = _process_qasc_df(globals().get("qasc_raw"))
curated_df = _ensure_schema(globals().get("curated_df", pd.DataFrame()), dataset_purpose="curated_demo")

print(f"SNLI processed    : {snli_df.shape}")
print(f"MultiNLI processed: {mnli_df.shape}")
print(f"QASC processed    : {qasc_df.shape}")
print(f"Curated demo      : {curated_df.shape}")

# ── ASAG base ─────────────────────────────────────────────────────────────────
if "combined_df" in globals():
    asag_df = combined_df.copy()
elif _cached(COMBINED_CSV):
    print("Loading ASAG combined_df from COMBINED_CSV...")
    asag_df = pd.read_csv(COMBINED_CSV)
else:
    print("WARNING: combined_df not found and COMBINED_CSV missing. Using empty ASAG dataframe.")
    asag_df = pd.DataFrame(columns=REQUIRED_COLS_4D)

asag_df = _ensure_schema(asag_df, dataset_purpose="auxiliary_scoring")

# GUARD: strip any curated_demo that accidentally ended up in asag_df
_leaked_in = asag_df["dataset_purpose"].eq("curated_demo") if "dataset_purpose" in asag_df.columns else pd.Series([False]*len(asag_df))
if _leaked_in.any():
    print(f"  GUARD: removing {_leaked_in.sum()} curated_demo rows from asag_df before merge.")
    asag_df = asag_df[~_leaked_in].reset_index(drop=True)

# ── Build or load combined_all_df ─────────────────────────────────────────────
_must_rebuild = FORCE_REBUILD_CURATED or not _cached(COMBINED_ALL_CSV)
if not _must_rebuild and _cached(COMBINED_ALL_CSV):
    _tmp = pd.read_csv(COMBINED_ALL_CSV, nrows=5)
    if "target_stars" not in _tmp.columns or "dataset_purpose" not in _tmp.columns:
        print("Cache missing required columns — forcing rebuild.")
        _must_rebuild = True

if not _must_rebuild:
    print("Loading combined_all_df from cache...")
    combined_all_df = pd.read_csv(COMBINED_ALL_CSV)
    print(f"Loaded: {combined_all_df.shape}")
else:
    print("Building combined_all_df from all sources...")
    all_parts = [p for p in [asag_df, snli_df, mnli_df, qasc_df, curated_df]
                 if p is not None and not p.empty]
    combined_all_df = pd.concat(all_parts, ignore_index=True)
    for col in REQUIRED_COLS_4D:
        if col not in combined_all_df.columns:
            combined_all_df[col] = np.nan
    combined_all_df = combined_all_df.reset_index(drop=True)
    combined_all_df["row_id"] = combined_all_df.index
    combined_all_df.to_csv(COMBINED_ALL_CSV, index=False)
    print(f"combined_all_df saved: {COMBINED_ALL_CSV}  {combined_all_df.shape}")

# ── Final safety ───────────────────────────────────────────────────────────────
for col in REQUIRED_COLS_4D:
    if col not in combined_all_df.columns:
        combined_all_df[col] = np.nan
combined_all_df["student_answer"] = combined_all_df["student_answer"].fillna("").astype(str)
if "count_sentences" in globals():
    combined_all_df["sent_count"] = combined_all_df["student_answer"].apply(count_sentences)
combined_all_df["is_multi_sentence"] = combined_all_df["sent_count"].fillna(0).astype(int) >= 2
if "row_id" not in combined_all_df.columns:
    combined_all_df["row_id"] = combined_all_df.index

# ── Split by purpose ──────────────────────────────────────────────────────────
# auxiliary_df: ONLY auxiliary_scoring rows
auxiliary_df = combined_all_df[
    combined_all_df["dataset_purpose"].eq("auxiliary_scoring")
].copy().reset_index(drop=True)

# reasoning_df: NLI + multi-hop + curated_demo ONLY
# NEVER includes auxiliary_scoring rows — reasoning_df must not be overwritten
reasoning_df = combined_all_df[
    combined_all_df["dataset_purpose"].isin(
        ["nli_reasoning", "multi_hop_reasoning", "curated_demo"])
].copy().reset_index(drop=True)

reasoning_eval_df = reasoning_df.copy()

# ── GUARD: curated_demo must not be in auxiliary_df ───────────────────────────
_leaked = auxiliary_df[auxiliary_df["dataset_purpose"].eq("curated_demo")]
assert len(_leaked) == 0, f"GUARD FAILED: {len(_leaked)} curated_demo rows leaked into auxiliary_df!"
print("GUARD OK: curated_demo not in auxiliary_df — DistilBERT safe ✓")
print(f"GUARD OK: reasoning_df has {len(reasoning_df)} rows — SNLI/MNLI/QASC/curated_demo preserved ✓")

auxiliary_df.to_csv(AUXILIARY_CSV, index=False)
reasoning_df.to_csv(REASONING_CSV, index=False)
print(f"auxiliary_df saved : {AUXILIARY_CSV}  ({len(auxiliary_df):,} rows)")
print(f"reasoning_df saved : {REASONING_CSV}  ({len(reasoning_df):,} rows)")

# ── Audit summary ─────────────────────────────────────────────────────────────
_total = len(combined_all_df)
_pur_dist = combined_all_df["dataset_purpose"].value_counts(dropna=False)
_star_rows = combined_all_df[combined_all_df["target_stars"].notna()]
_multi = int(combined_all_df["is_multi_sentence"].sum())
_single = _total - _multi

print("\n" + "═"*62)
print("  A. AUXILIARY SCORING DATASET SUMMARY")
print("═"*62)
for ds in auxiliary_df["source_dataset"].dropna().unique():
    sub = auxiliary_df[auxiliary_df["source_dataset"] == ds]
    sc  = sub["normalized_score"].dropna()
    avg = f"{sc.mean():.2f}" if len(sc) else "N/A"
    pct = sub["is_multi_sentence"].mean() * 100 if len(sub) else 0
    print(f"  {ds:<22} {len(sub):>7,}  avg_score={avg}  multi={pct:.1f}%")

print("\n" + "═"*62)
print("  B. REASONING DATASET SUMMARY")
print("═"*62)
for ds in reasoning_df["source_dataset"].dropna().unique():
    sub = reasoning_df[reasoning_df["source_dataset"] == ds]
    pur = sub["dataset_purpose"].iloc[0] if len(sub) else ""
    pct = sub["is_multi_sentence"].mean() * 100 if len(sub) else 0
    print(f"  {ds:<22} {pur:<22} {len(sub):>7,}  multi={pct:.1f}%")

print("\n" + "═"*62)
print("  C. SINGLE vs MULTI-SENTENCE SUMMARY")
print("═"*62)
print(f"  Total          : {_total:,}")
print(f"  Multi-sentence : {_multi:,}  ({_multi/max(_total,1)*100:.1f}%)")
print(f"  Single-sentence: {_single:,}  ({_single/max(_total,1)*100:.1f}%)")

print("\n" + "═"*62)
print("  D. DATASET PURPOSE DISTRIBUTION")
print("═"*62)
for p, cnt in _pur_dist.items():
    print(f"  {str(p):<30} {cnt:>7,}  ({cnt/max(_total,1)*100:.1f}%)")

print("\n" + "═"*62)
print("  F. TARGET STARS DISTRIBUTION (curated_demo)")
print("═"*62)
if len(_star_rows):
    for st, cnt in sorted(_star_rows["target_stars"].astype(int).value_counts().items()):
        stars = "★"*int(st) + "☆"*(5-int(st))
        print(f"  {int(st)}★  {stars:<5}  {cnt:>6,}")
else:
    print("  No target_stars data found. Check Section 4C and FORCE_REBUILD_CURATED=True.")

# ── Plots ─────────────────────────────────────────────────────────────────────
os.makedirs(PLOTS_DIR, exist_ok=True)

try:
    fig, ax = plt.subplots(figsize=(6,6))
    ax.pie([_multi,_single], labels=[f"Multi-sentence\n({_multi:,})",f"Single-sentence\n({_single:,})"],
           autopct="%1.1f%%", startangle=140, explode=(0.03,0.03))
    ax.set_title("Single vs Multi-Sentence Distribution")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR,"single_multi_sentence_pie.png"),dpi=150,bbox_inches="tight")
    plt.show(); plt.close()
except Exception as _pe: print(f"Plot warning: {_pe}")

try:
    fig, ax = plt.subplots(figsize=(8,4))
    _pur_dist.plot(kind="barh",ax=ax,edgecolor="white")
    ax.set(title="Dataset Purpose Distribution",xlabel="Count")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR,"dataset_purpose_distribution.png"),dpi=150,bbox_inches="tight")
    plt.show(); plt.close()
except Exception as _pe: print(f"Plot warning: {_pe}")

if len(_star_rows):
    try:
        fig, ax = plt.subplots(figsize=(8,4))
        _sv = _star_rows["target_stars"].astype(int).value_counts().sort_index()
        ax.bar([f"{s}★" for s in _sv.index],_sv.values,edgecolor="white")
        ax.set(title="Target Stars Distribution (Curated Demo)",xlabel="Stars",ylabel="Count")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR,"target_stars_distribution.png"),dpi=150,bbox_inches="tight")
        plt.show(); plt.close()
    except Exception as _pe: print(f"Plot warning: {_pe}")

print("\nSection 4D complete.")
print(f"  combined_all_df   : {combined_all_df.shape}")
print(f"  auxiliary_df      : {auxiliary_df.shape}")
print(f"  reasoning_df      : {reasoning_df.shape}")


SNLI processed    : (15000, 17)
MultiNLI processed: (15000, 17)
QASC processed    : (9060, 17)
Curated demo      : (6150, 17)
Building combined_all_df from all sources...
combined_all_df saved: /content/drive/MyDrive/FYP_Data/outputs/combined_all_df.csv  (64354, 18)
GUARD OK: curated_demo not in auxiliary_df — DistilBERT safe ✓
GUARD OK: reasoning_df has 45210 rows — SNLI/MNLI/QASC/curated_demo preserved ✓
auxiliary_df saved : /content/drive/MyDrive/FYP_Data/outputs/auxiliary_scoring_df.csv  (19,144 rows)
reasoning_df saved : /content/drive/MyDrive/FYP_Data/outputs/reasoning_df.csv  (45,210 rows)

══════════════════════════════════════════════════════════════
  A. AUXILIARY SCORING DATASET SUMMARY
══════════════════════════════════════════════════════════════
  semeval_beetle           4,457  avg_score=5.72  multi=0.6%
  mohler                   2,050  avg_score=4.18  multi=1.4%
  scientsbank              9,962  avg_score=5.95  multi=1.6%
  asap_sas                 2,675  avg_score=7.5

### Section 5B — Dataset Reporting

**Purpose:** Print per-dataset summary statistics to verify that loading and preprocessing succeeded.

**What this cell does:**
- Reports row counts, mean score, and multi-sentence coverage per dataset
- Confirms `curated_demo` rows have `target_stars` set and `normalized_score = NaN`
- Flags any unexpected data issues

**Interpretation:** Multi-sentence coverage in `auxiliary_scoring` is expected to be low (ASAG answers are short). Coverage in `curated_demo` should be ≥ 40% for meaningful NLI graph evaluation.


In [ ]:
# ── A: Auxiliary scoring summary ─────────────────────────────────────────────
_report_df = combined_all_df if "combined_all_df" in dir() else combined_df
aux_df_r       = _report_df[_report_df["dataset_purpose"] == "auxiliary_scoring"]
reasoning_df_r = _report_df[_report_df["dataset_purpose"].isin(
                     ["nli_reasoning", "multi_hop_reasoning", "curated_demo"])]
print("=" * 62)
print("  A. AUXILIARY SCORING DATASET SUMMARY")
print("=" * 62)
print(f"  Total auxiliary rows : {len(aux_df_r):,}")
_multi_a = aux_df_r["is_multi_sentence"].sum()
print(f"  Multi-sentence       : {_multi_a:,}  ({_multi_a/max(len(aux_df_r),1)*100:.1f}%)")
print(f"  {'Dataset':<20} {'N':>7}  {'Multi%':>7}")
print("  " + "-"*40)
for ds in ["semeval_beetle","mohler","scientsbank","asap_sas"]:
    sub = aux_df_r[aux_df_r["source_dataset"]==ds]
    n   = len(sub)
    nm  = sub["is_multi_sentence"].sum()
    print(f"  {ds:<20} {n:>7,}  {nm/max(n,1)*100:>6.1f}%")
print("  NOTE: Used ONLY for DistilBERT scoring baseline — not for reasoning evaluation")

# ── B: Reasoning dataset summary ──────────────────────────────────────────────
# reasoning_df_r defined above
print()
print("=" * 62)
print("  B. REASONING DATASET SUMMARY")
print("=" * 62)
print(f"  Total reasoning rows : {len(reasoning_df_r):,}")
for ds in combined_all_df["source_dataset"].unique():
    sub = reasoning_df_r[reasoning_df_r["source_dataset"]==ds]
    if len(sub) == 0: continue
    print(f"  {ds:<20} {len(sub):>7,}")
print("  NOTE: Primary evidence for reasoning-depth evaluation")

# ── C: Single vs multi-sentence ───────────────────────────────────────────────
print()
print("=" * 62)
print("  C. SINGLE vs MULTI-SENTENCE SUMMARY")
print("=" * 62)
_total = len(_report_df)
_multi = _report_df["is_multi_sentence"].sum()
_single= _total - _multi
print(f"  Total rows      : {_total:,}")
print(f"  Single-sentence : {_single:,}  ({_single/_total*100:.1f}%)")
print(f"  Multi-sentence  : {_multi:,}  ({_multi/_total*100:.1f}%)")
print("  Single → structural evaluation only")
print("  Multi  → structural + NLI reasoning graph")

# ── D: Dataset purpose distribution ──────────────────────────────────────────
print()
print("=" * 62)
print("  D. DATASET PURPOSE DISTRIBUTION")
print("=" * 62)
for p, cnt in _report_df["dataset_purpose"].value_counts().items():
    print(f"  {p:<25} {cnt:>7,}  ({cnt/_total*100:.1f}%)")

# ── E: NLI label distribution ─────────────────────────────────────────────────
nli_df_r = combined_all_df[combined_all_df["nli_label"].notna()]
print()
print("=" * 62)
print("  E. NLI LABEL DISTRIBUTION")
print("=" * 62)
if len(nli_df_r) > 0:
    for lbl, cnt in nli_df_r["nli_label"].value_counts().items():
        print(f"  {lbl:<20} {cnt:>7,}")
else:
    print("  No NLI data available (SNLI/MultiNLI not loaded).")

# ── F: Multi-hop reasoning count ──────────────────────────────────────────────
multihop_df_r = combined_all_df[combined_all_df["dataset_purpose"] == "multi_hop_reasoning"]
print()
print("=" * 62)
print("  F. MULTI-HOP REASONING COUNT")
print("=" * 62)
print(f"  Multi-hop rows  : {len(multihop_df_r):,}")
if "reasoning_facts" in multihop_df_r.columns:
    has_facts = multihop_df_r["reasoning_facts"].notna().sum()
    print(f"  Rows with facts : {has_facts:,}")
print("=" * 62)


  A. AUXILIARY SCORING DATASET SUMMARY
  Total auxiliary rows : 19,144
  Multi-sentence       : 409  (2.1%)
  Dataset                    N   Multi%
  ----------------------------------------
  semeval_beetle         4,457     0.6%
  mohler                 2,050     1.4%
  scientsbank            9,962     1.6%
  asap_sas               2,675     7.4%
  NOTE: Used ONLY for DistilBERT scoring baseline — not for reasoning evaluation

  B. REASONING DATASET SUMMARY
  Total reasoning rows : 45,210
  snli                  15,000
  multi_nli             15,000
  qasc                   9,060
  curated_0star          1,000
  curated_1star          1,050
  curated_2star          1,000
  curated_3star          1,000
  curated_4star          1,050
  curated_5star          1,050
  NOTE: Primary evidence for reasoning-depth evaluation

  C. SINGLE vs MULTI-SENTENCE SUMMARY
  Total rows      : 64,354
  Single-sentence : 19,896  (30.9%)
  Multi-sentence  : 44,458  (69.1%)
  Single → structural evaluatio

## Section 6 — Train / Validation / Test Split

**Purpose:** Create stratified train/val/test splits for DistilBERT regression training.

**Critical constraint:** Only `auxiliary_scoring` rows enter these splits. All other `dataset_purpose` groups are excluded — this is enforced by a guard assertion.

**What this cell does:**
1. Filters `auxiliary_df` to rows with valid `normalized_score`
2. Assigns `grade` labels (high / medium / low) if missing
3. **Balances grades** — caps each grade at 2,000 rows via random sampling (seed 42)
4. Splits: 70% train / ~11% val (of remainder) / 10% test, stratified by grade
5. Saves splits to CSV; verifies no `curated_demo` leakage

**Why balance grades?** ASAG datasets are naturally skewed toward low scores. Without balancing, DistilBERT over-fits to the dominant class and poorly estimates medium/high quality answers.

**Expected output:** Split sizes, grade distributions, guard confirmation.

**Strength:** Stratified split + balancing → representative evaluation across all answer quality levels.
**Weakness:** Capping at 2,000 rows per grade may discard valid training data if one grade has many more rows.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Section 6 — Train / Validation / Test Split
# CRITICAL: auxiliary_scoring ONLY. Stratified. Full assertions. Edge weighting.
# ══════════════════════════════════════════════════════════════════════════════

import os
import pandas as pd
from sklearn.model_selection import train_test_split

# ── SAFE GLOBAL DEFAULTS (prevents NameError crashes) ─────────────────────────
FORCE_REBUILD_DATASET = globals().get("FORCE_REBUILD_DATASET", False)
RANDOM_STATE = globals().get("RANDOM_STATE", 42)

TRAIN_CSV = globals().get("TRAIN_CSV", "train.csv")
VAL_CSV   = globals().get("VAL_CSV", "val.csv")
TEST_CSV  = globals().get("TEST_CSV", "test.csv")

# _cached() is defined canonically in Cell 4 — reused here.

# ── LOAD EXISTING SPLITS IF AVAILABLE ─────────────────────────────────────────
if (
    not FORCE_REBUILD_DATASET
    and _cached(TRAIN_CSV)
    and _cached(VAL_CSV)
    and _cached(TEST_CSV)
):
    print("Loading existing auxiliary splits...")

    train_df = pd.read_csv(TRAIN_CSV)
    val_df   = pd.read_csv(VAL_CSV)
    test_df  = pd.read_csv(TEST_CSV)

else:
    print("Building stratified splits from auxiliary_scoring only...")

    _base = auxiliary_df[
        (auxiliary_df["dataset_purpose"] == "auxiliary_scoring")
        & (auxiliary_df["normalized_score"].notna())
        & (auxiliary_df["dataset_purpose"] != "curated_demo")
    ].copy()

    assert len(_base) > 0, "No valid auxiliary_scoring rows found. Check Section 4D."

    # ── grade assignment fallback ─────────────────────────────────────────────
    if "grade" not in _base.columns or _base["grade"].isna().all():
        _base["grade"] = _base["normalized_score"].apply(assign_grade)

    _base["grade"] = _base["grade"].fillna("low")

    # ── grade-balanced sampling ───────────────────────────────────────────────
    _parts = []
    for g, grp in _base.groupby("grade"):
        _parts.append(grp.sample(n=min(len(grp), 2000), random_state=RANDOM_STATE))

    balanced_df = pd.concat(_parts).reset_index(drop=True)

    # ── edge-star weighting ───────────────────────────────────────────────────
    if "target_stars" in balanced_df.columns:
        balanced_df["_sample_weight"] = balanced_df["target_stars"].apply(
            lambda s: 5.0 if s in (0, 5) else 1.0
        )
    else:
        balanced_df["_sample_weight"] = 1.0

    # ── stratified splits ─────────────────────────────────────────────────────
    train_val, test_df = train_test_split(
        balanced_df,
        test_size=0.10,
        random_state=RANDOM_STATE,
        stratify=balanced_df["grade"],
    )

    train_df, val_df = train_test_split(
        train_val,
        test_size=0.1111,
        random_state=RANDOM_STATE,
        stratify=train_val["grade"],
    )

    # ── persist ───────────────────────────────────────────────────────────────
    train_df.to_csv(TRAIN_CSV, index=False)
    val_df.to_csv(VAL_CSV, index=False)
    test_df.to_csv(TEST_CSV, index=False)

    print("Splits saved.")

# ── LEAKAGE CHECK ─────────────────────────────────────────────────────────────
for _sname, _sdf in [("train", train_df), ("val", val_df), ("test", test_df)]:
    if "dataset_purpose" in _sdf.columns:
        _n = (_sdf["dataset_purpose"] == "curated_demo").sum()
        if _n > 0:
            raise AssertionError(
                f"LEAKAGE DETECTED: {_n} curated_demo rows in {_sname}_df. "
                "Stop. Fix dataset construction."
            )

print("ASSERTION PASSED: curated_demo not in any split ✓")

# ── SUMMARY ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 62)
print("  SPLIT SUMMARY")
print("=" * 62)

for _sname, _sdf in [("train", train_df), ("val", val_df), ("test", test_df)]:
    _dup = _sdf.duplicated(subset=["student_answer"]).sum() if "student_answer" in _sdf.columns else 0
    _cd = (_sdf["dataset_purpose"].eq("curated_demo").sum()
           if "dataset_purpose" in _sdf.columns else 0)

    print(f"  {_sname:<6}: {len(_sdf):>6,} rows | duplicates={_dup} | curated_demo={_cd}")

    if "grade" in _sdf.columns:
        print(f"         grade dist: {_sdf['grade'].value_counts().to_dict()}")

    if "source_dataset" in _sdf.columns:
        print(f"         sources   : {_sdf['source_dataset'].value_counts().to_dict()}")

print("=" * 62)

logger.info(f"Splits: train={len(train_df)} val={len(val_df)} test={len(test_df)}")

Loading existing auxiliary splits...
ASSERTION PASSED: curated_demo not in any split ✓

  SPLIT SUMMARY
  train :  4,800 rows | duplicates=57 | curated_demo=0
         grade dist: {'high': 1600, 'medium': 1600, 'low': 1600}
         sources   : {'scientsbank': 1934, 'semeval_beetle': 1304, 'asap_sas': 796, 'mohler': 766}
  val   :    600 rows | duplicates=2 | curated_demo=0
         grade dist: {'low': 200, 'medium': 200, 'high': 200}
         sources   : {'scientsbank': 231, 'semeval_beetle': 160, 'asap_sas': 110, 'mohler': 99}
  test  :    600 rows | duplicates=0 | curated_demo=0
         grade dist: {'high': 200, 'low': 200, 'medium': 200}
         sources   : {'scientsbank': 239, 'semeval_beetle': 156, 'asap_sas': 112, 'mohler': 93}


## Section 7 — Exploratory Data Analysis

**Purpose:** Comprehensive visual analysis of dataset composition, distributions, and split quality before model training.

**Plots produced:**
- **G1** — Dataset purpose distribution (horizontal bar)
- **G2** — Target star distribution 0★–5★ (bar)
- **G3** — Single vs multi-sentence split (pie + grouped bar)
- **G4** — Answer word count distribution (histogram)
- **G5** — Normalised score distribution (histogram)
- **G6** — Grade counts before vs after balancing (side-by-side)
- **B1/B2** — Train/val/test split sizes and grade distribution per split
- **C1–C3** — Score normalisation tables
- **D** — Balancing summary table + chart

**All plots saved to `PLOTS_DIR`.** Each plot includes a printed interpretation.


In [ ]:
# ══════════════════════════════════════════════════════════════════
# SECTION A — Exploratory Data Analysis (EDA)
# 6 graphs: purpose dist, star dist, pie, word count,
#           score dist, grade balance
# SECTION B — Train/Val/Test Split Summary
# SECTION C — Normalisation Summary Tables
# SECTION D — Balancing Summary
# ══════════════════════════════════════════════════════════════════

import os, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
matplotlib.rcParams.update({
    "font.family":       "DejaVu Sans",
    "font.size":         10,
    "axes.titlesize":    11,
    "axes.titleweight":  "bold",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        130,
})

# ── Palette ───────────────────────────────────────────────────────
PAL = {
    "auxiliary_scoring":   "#1D9E75",
    "nli_reasoning":       "#378ADD",
    "multi_hop_reasoning": "#EF9F27",
    "curated_demo":        "#E24B4A",
    "high":   "#1D9E75",
    "medium": "#EF9F27",
    "low":    "#E24B4A",
    "train":  "#378ADD",
    "val":    "#EF9F27",
    "test":   "#E24B4A",
}
STAR_COL = ["#E24B4A","#EF9F27","#F4D03F","#1D9E75","#378ADD","#6C5CE7"]
warnings.filterwarnings("ignore")
os.makedirs(PLOTS_DIR, exist_ok=True)

# ── Reusable plot helpers ─────────────────────────────────────────
def _save(fig, name):
    path = os.path.join(PLOTS_DIR, name)
    try:
        fig.savefig(path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {path}")
    except Exception as e:
        print(f"  Save failed ({name}): {e}")
    plt.close(fig)

def _bar(ax, labels, values, colors, title, xlabel, ylabel, fmt="{:,}"):
    bars = ax.bar(labels, values, color=colors, edgecolor="white", width=0.55)
    for b, v in zip(bars, values):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+max(values)*0.01,
                fmt.format(int(v)), ha="center", va="bottom", fontsize=8, fontweight="bold")
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.set_ylim(0, max(values)*1.2)

def _hbar(ax, labels, values, colors, title, xlabel):
    bars = ax.barh(labels, values, color=colors, edgecolor="white", height=0.55)
    for b, v in zip(bars, values):
        ax.text(v + max(values)*0.01, b.get_y()+b.get_height()/2,
                f"{int(v):,}", va="center", fontsize=8)
    ax.set_title(title); ax.set_xlabel(xlabel)
    ax.set_xlim(0, max(values)*1.15)

def _hist(ax, data, title, xlabel, color="#378ADD", bins=30):
    ax.hist(data, bins=bins, color=color, edgecolor="white", alpha=0.85)
    ax.axvline(data.median(), color="#E24B4A", linewidth=1.5, linestyle="--",
               label=f"Median={data.median():.1f}")
    ax.axvline(data.mean(), color="#1D9E75", linewidth=1.5, linestyle="-.",
               label=f"Mean={data.mean():.1f}")
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel("Frequency")
    ax.legend(fontsize=8)

# ── Data guards ───────────────────────────────────────────────────
_comb    = globals().get("combined_all_df", pd.DataFrame())
_aux     = globals().get("auxiliary_df",    pd.DataFrame())
_cur     = globals().get("curated_df",      pd.DataFrame())
_train   = globals().get("train_df",        pd.DataFrame())
_val     = globals().get("val_df",          pd.DataFrame())
_test    = globals().get("test_df",         pd.DataFrame())

if _comb.empty:
    _comb = globals().get("combined_df", pd.DataFrame())

# ══════════════════════════════════════════════════════════════════
# SECTION A — EDA GRAPHS
# ══════════════════════════════════════════════════════════════════

print("=" * 60)
print("  SECTION A — EDA GRAPHS")
print("=" * 60)

# ── Graph 1: Dataset Purpose Distribution ────────────────────────
print("\nGraph 1: Dataset Purpose Distribution")
if not _comb.empty and "dataset_purpose" in _comb.columns:
    fig, ax = plt.subplots(figsize=(9, 4))
    _pc  = _comb["dataset_purpose"].value_counts()
    _tot = _pc.sum()
    _cols = [PAL.get(p, "#999") for p in _pc.index]
    _hbar(ax, _pc.index.tolist(), _pc.values.tolist(), _cols,
          "Graph 1 — Dataset Purpose Distribution", "Number of Rows")
    # add % labels
    for i, (v, lbl) in enumerate(zip(_pc.values, _pc.index)):
        ax.text(v * 0.5, i, f"{v/max(_tot,1)*100:.1f}%",
                ha="center", va="center", fontsize=9, color="white", fontweight="bold")
    plt.tight_layout()
    _save(fig, "g1_dataset_purpose_distribution.png")
    print(f"  Interpretation: Dataset has {len(_pc)} purposes. "
          f"Dominant purpose: '{_pc.idxmax()}' ({_pc.max():,} rows, {_pc.max()/_tot*100:.1f}%). "
          f"Purpose separation ensures curated_demo never contaminates DistilBERT regression.")
else:
    print("  SKIPPED — combined_all_df not available.")

# ── Graph 2: Target Stars Distribution ───────────────────────────
print("\nGraph 2: Target Stars Distribution (curated_demo)")
_star_src = _cur if (not _cur.empty and "target_stars" in _cur.columns) else (
    _comb[_comb["dataset_purpose"]=="curated_demo"] if not _comb.empty else pd.DataFrame()
)
if not _star_src.empty and "target_stars" in _star_src.columns:
    fig, ax = plt.subplots(figsize=(9, 4))
    _sv = _star_src["target_stars"].astype(int).value_counts().sort_index()
    _bar(ax, [f"{s}★" for s in _sv.index], _sv.values.tolist(),
         [STAR_COL[int(s)] for s in _sv.index],
         "Graph 2 — Target Stars Distribution (Curated Demo)",
         "Star Level", "Number of Rows")
    plt.tight_layout()
    _save(fig, "g2_target_stars_distribution.png")
    _balanced = _sv.std() / _sv.mean() < 0.3
    print(f"  Interpretation: {len(_sv)} star levels loaded. "
          f"Most common: {_sv.idxmax()}★ ({_sv.max():,} rows). "
          f"Balance: {'roughly balanced (CV<0.3)' if _balanced else 'imbalanced — lower star levels may be under-represented'}.")
else:
    print("  SKIPPED — no curated_demo rows with target_stars.")

# ── Graph 3: Single vs Multi-Sentence Pie ────────────────────────
print("\nGraph 3: Single vs Multi-Sentence Distribution")
_ref3 = _comb if not _comb.empty else _aux
if not _ref3.empty and "is_multi_sentence" in _ref3.columns:
    _multi  = int(_ref3["is_multi_sentence"].sum())
    _single = len(_ref3) - _multi
    fig, axes3 = plt.subplots(1, 2, figsize=(12, 4))

    # Left: pie overall
    wedges, texts, autos = axes3[0].pie(
        [_multi, _single],
        labels=[f"Multi-sentence\n({_multi:,})", f"Single-sentence\n({_single:,})"],
        autopct="%1.1f%%", startangle=90,
        colors=["#378ADD","#EF9F27"],
        explode=(0.03,0.03),
        wedgeprops={"edgecolor":"white","linewidth":1.5}
    )
    for a in autos: a.set_fontsize(9)
    axes3[0].set_title("Graph 3a — Overall Split")

    # Right: by dataset_purpose
    if "dataset_purpose" in _ref3.columns:
        _pm = (_ref3.groupby("dataset_purpose")["is_multi_sentence"]
               .value_counts(normalize=True).unstack(fill_value=0)*100)
        _pm.index = [str(i)[:18] for i in _pm.index]
        _x = np.arange(len(_pm))
        _w = 0.35
        axes3[1].bar(_x-_w/2, _pm.get(True, 0), _w, label="Multi", color="#378ADD", edgecolor="white")
        axes3[1].bar(_x+_w/2, _pm.get(False, 0), _w, label="Single", color="#EF9F27", edgecolor="white")
        axes3[1].set_xticks(_x)
        axes3[1].set_xticklabels(_pm.index, rotation=25, ha="right", fontsize=8)
        axes3[1].set_ylabel("Percentage (%)"); axes3[1].set_ylim(0,110)
        axes3[1].set_title("Graph 3b — By Dataset Purpose")
        axes3[1].legend(fontsize=8)

    fig.suptitle("Graph 3 — Single vs Multi-Sentence Distribution", fontsize=11, fontweight="bold")
    plt.tight_layout()
    _save(fig, "g3_single_vs_multi_sentence.png")
    _multi_pct = _multi/max(len(_ref3),1)*100
    print(f"  Interpretation: {_multi_pct:.1f}% answers are multi-sentence → eligible for NLI graph. "
          f"curated_demo has highest multi-sentence ratio (by design — 3★–5★ answers are multi-sentence). "
          f"Single-sentence answers receive structural analysis only.")
else:
    print("  SKIPPED — is_multi_sentence column not found.")

# ── Graph 4: Answer Word Count Distribution ───────────────────────
print("\nGraph 4: Answer Word Count Distribution (auxiliary_scoring)")
_ref4 = _aux if not _aux.empty else _comb
if not _ref4.empty and "answer_word_count" in _ref4.columns:
    _wc = _ref4["answer_word_count"].dropna().clip(0, 250)
    fig, axes4 = plt.subplots(1, 2, figsize=(13, 4))
    _hist(axes4[0], _wc, "Graph 4a — Word Count Distribution\n(All Auxiliary)", "Word Count")
    # By dataset
    if "source_dataset" in _ref4.columns:
        for ds, grp in _ref4.groupby("source_dataset"):
            _d = grp["answer_word_count"].dropna().clip(0,250)
            axes4[1].hist(_d, bins=25, alpha=0.55, label=ds[:18], edgecolor="white")
        axes4[1].set_title("Graph 4b — Word Count by Dataset")
        axes4[1].set_xlabel("Word Count"); axes4[1].set_ylabel("Frequency")
        axes4[1].legend(fontsize=7)
    fig.suptitle("Graph 4 — Answer Word Count Distribution", fontsize=11, fontweight="bold")
    plt.tight_layout()
    _save(fig, "g4_word_count_distribution.png")
    print(f"  Interpretation: Median word count = {_wc.median():.0f}, mean = {_wc.mean():.0f}. "
          f"Most ASAG answers are short (5–50 words) — expected for auxiliary scoring datasets. "
          f"Long-tail responses may be from ASAP-SAS which includes essay-style answers.")
else:
    print("  SKIPPED — answer_word_count column not found.")

# ── Graph 5: Normalised Score Distribution ────────────────────────
print("\nGraph 5: Normalised Score Distribution (auxiliary_scoring)")
if not _aux.empty and "normalized_score" in _aux.columns:
    _ns = _aux["normalized_score"].dropna()
    fig, axes5 = plt.subplots(1, 2, figsize=(13, 4))
    _hist(axes5[0], _ns, "Graph 5a — Score Distribution\n(All Auxiliary)", "Normalised Score (0–10)", color="#1D9E75")
    # Per dataset
    if "source_dataset" in _aux.columns:
        for ds, grp in _aux.groupby("source_dataset"):
            _sc = grp["normalized_score"].dropna()
            axes5[1].hist(_sc, bins=20, alpha=0.55, label=ds[:18], edgecolor="white")
        axes5[1].set_title("Graph 5b — Score by Dataset")
        axes5[1].set_xlabel("Normalised Score (0–10)"); axes5[1].set_ylabel("Frequency")
        axes5[1].legend(fontsize=7)
    fig.suptitle("Graph 5 — Normalised Score Distribution", fontsize=11, fontweight="bold")
    plt.tight_layout()
    _save(fig, "g5_normalised_score_distribution.png")
    _skew = "right-skewed" if _ns.mean() < _ns.median()*1.05 else ("left-skewed" if _ns.mean() > _ns.median()*1.05 else "roughly symmetric")
    print(f"  Interpretation: Scores range {_ns.min():.1f}–{_ns.max():.1f}. "
          f"Mean={_ns.mean():.2f}, Median={_ns.median():.2f} → {_skew} distribution. "
          f"Score imbalance (many low scores) motivates grade-based balancing in Section 6.")
else:
    print("  SKIPPED — normalized_score not in auxiliary_df.")

# ── Graph 6: Grade Distribution Before vs After Balancing ─────────
print("\nGraph 6: Grade Distribution Before vs After Balancing")
_grade_order = ["high","medium","low"]
if not _aux.empty and "grade" in _aux.columns and not _train.empty and "grade" in _train.columns:
    _gb = _aux["grade"].value_counts().reindex(_grade_order).fillna(0).astype(int)
    _all_sp = pd.concat([_train, _val, _test], ignore_index=True)
    _ga = _all_sp["grade"].value_counts().reindex(_grade_order).fillna(0).astype(int)

    fig, axes6 = plt.subplots(1, 3, figsize=(14, 4))

    _bar(axes6[0], _grade_order, [_gb.get(g,0) for g in _grade_order],
         [PAL[g] for g in _grade_order],
         "Graph 6a — Before Balancing", "Grade", "Count")

    _bar(axes6[1], _grade_order, [_ga.get(g,0) for g in _grade_order],
         [PAL[g] for g in _grade_order],
         "Graph 6b — After Balancing", "Grade", "Count")

    # Side-by-side comparison
    _x = np.arange(3); _w = 0.35
    b1 = axes6[2].bar(_x-_w/2, [_gb.get(g,0) for g in _grade_order], _w,
                      label="Before", color="#E24B4A", edgecolor="white", alpha=0.85)
    b2 = axes6[2].bar(_x+_w/2, [_ga.get(g,0) for g in _grade_order], _w,
                      label="After",  color="#1D9E75", edgecolor="white", alpha=0.85)
    axes6[2].bar_label(b1, fmt="%d", padding=2, fontsize=8)
    axes6[2].bar_label(b2, fmt="%d", padding=2, fontsize=8)
    axes6[2].set_xticks(_x); axes6[2].set_xticklabels(_grade_order)
    axes6[2].set_title("Graph 6c — Before vs After (Side-by-Side)")
    axes6[2].set_ylabel("Count"); axes6[2].legend(fontsize=8)

    fig.suptitle("Graph 6 — Grade Distribution Before vs After Balancing", fontsize=11, fontweight="bold")
    plt.tight_layout()
    _save(fig, "g6_grade_balance_before_after.png")

    # imbalance ratios
    _ir_b = max(_gb.values)/max(min(_gb.values),1)
    _ir_a = max(_ga.values)/max(min(_ga.values),1)
    print(f"  Interpretation: Before balancing — imbalance ratio {_ir_b:.1f}× "
          f"(most common grade has {_ir_b:.1f}× rows of least common). "
          f"After balancing — ratio reduced to {_ir_a:.1f}×. "
          f"Balanced splits improve DistilBERT's ability to learn across all quality levels.")
else:
    print("  SKIPPED — requires auxiliary_df with grade + train_df (run Sections 5–6).")

# ══════════════════════════════════════════════════════════════════
# SECTION B — Train / Val / Test Split Summary
# ══════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("  SECTION B — TRAIN / VAL / TEST SPLIT SUMMARY")
print("="*60)

if not _train.empty:
    _split_names  = ["Train", "Validation", "Test"]
    _split_frames = [_train, _val, _test]
    _split_counts = [len(d) for d in _split_frames]
    _total_sp     = sum(_split_counts)

    # Print table
    print(f"\n  {'Split':<14} {'Rows':>7}  {'%':>6}")
    print("  " + "-"*28)
    for n, c in zip(_split_names, _split_counts):
        print(f"  {n:<14} {c:>7,}  {c/_total_sp*100:>5.1f}%")
    print(f"  {'TOTAL':<14} {_total_sp:>7,}  100.0%")

    # Grade distribution per split
    if "grade" in _train.columns:
        print(f"\n  {'Grade':<10}", end="")
        for n in _split_names: print(f"  {n:>9}", end="")
        print()
        print("  " + "-"*44)
        for g in _grade_order:
            print(f"  {g:<10}", end="")
            for d in _split_frames:
                cnt = (d["grade"]==g).sum()
                print(f"  {cnt:>9,}", end="")
            print()

    # Plot
    fig, axes_b = plt.subplots(1, 2, figsize=(12, 4))

    # Split sizes bar
    _bar(axes_b[0], _split_names, _split_counts,
         [PAL["train"], PAL["val"], PAL["test"]],
         "B1 — Split Sizes", "Split", "Number of Rows")
    for i,(n,c) in enumerate(zip(_split_names,_split_counts)):
        axes_b[0].text(i, c + _total_sp*0.01,
                       f"{c/_total_sp*100:.1f}%", ha="center", fontsize=8, color="#444")

    # Grade distribution grouped bar per split
    if "grade" in _train.columns:
        _x = np.arange(3); _w = 0.22
        _sp_cols = [PAL["train"], PAL["val"], PAL["test"]]
        for k, (sp_name, sp_df) in enumerate(zip(_split_names, _split_frames)):
            _gc = sp_df["grade"].value_counts().reindex(_grade_order).fillna(0)
            axes_b[1].bar(_x + _w*k - _w, _gc.values, _w,
                          label=sp_name, color=_sp_cols[k], edgecolor="white", alpha=0.85)
        axes_b[1].set_xticks(_x); axes_b[1].set_xticklabels(_grade_order)
        axes_b[1].set_title("B2 — Grade Distribution Per Split")
        axes_b[1].set_ylabel("Count"); axes_b[1].legend(fontsize=8)

    fig.suptitle("Section B — Train / Val / Test Split Summary", fontsize=11, fontweight="bold")
    plt.tight_layout()
    _save(fig, "secB_split_summary.png")
    print(f"\n  Interpretation: 70/10/20 split (approx). Stratified by grade ensures each split "
          f"has representative high/medium/low coverage. Imbalanced splits → biased evaluation. "
          f"Validation set monitors overfitting during training.")
else:
    print("  SKIPPED — train_df not available (run Section 6 first).")

# ══════════════════════════════════════════════════════════════════
# SECTION C — Normalisation Summary Tables
# ══════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("  SECTION C — NORMALISATION SUMMARY TABLES")
print("="*60)

if not _aux.empty and "normalized_score" in _aux.columns and "source_dataset" in _aux.columns:
    _rows_c = []
    for ds, grp in _aux.groupby("source_dataset"):
        sc = grp["normalized_score"].dropna()
        if len(sc):
            _rows_c.append({
                "Dataset": ds,
                "Raw Min": sc.min(),
                "Raw Max": sc.max(),
                "Raw Mean": sc.mean(),
                "Norm Min (0–1)": round(sc.min()/10, 3),
                "Norm Max (0–1)": round(sc.max()/10, 3),
                "Norm Mean (0–1)": round(sc.mean()/10, 3),
                "Rows": len(sc),
            })
    _tbl = pd.DataFrame(_rows_c)
    print("\n  TABLE C1 — Raw Score Ranges (0–10 scale)")
    print("  " + "-"*70)
    print(f"  {'Dataset':<22} {'Min':>6} {'Max':>6} {'Mean':>7} {'Rows':>7}")
    print("  " + "-"*52)
    for _, r in _tbl.iterrows():
        print(f"  {r['Dataset']:<22} {r['Raw Min']:>6.2f} {r['Raw Max']:>6.2f} {r['Raw Mean']:>7.2f} {int(r['Rows']):>7,}")

    print("\n  TABLE C2 — Normalised Score Ranges (0.0–1.0 scale)")
    print("  " + "-"*70)
    print(f"  {'Dataset':<22} {'Min':>7} {'Max':>7} {'Mean':>8} {'Rows':>7}")
    print("  " + "-"*52)
    for _, r in _tbl.iterrows():
        print(f"  {r['Dataset']:<22} {r['Norm Min (0–1)']:>7.3f} {r['Norm Max (0–1)']:>7.3f} {r['Norm Mean (0–1)']:>8.3f} {int(r['Rows']):>7,}")

    print("\n  TABLE C3 — Grade Threshold Mapping")
    print("  " + "-"*50)
    print("  Grade    Score Range (0–10)   0–1 Range")
    print("  " + "-"*50)
    print("  high     7.5 – 10.0           0.75 – 1.00")
    print("  medium   4.0 –  7.4           0.40 – 0.74")
    print("  low      0.0 –  3.9           0.00 – 0.39")

    print("\n  Interpretation: Scores normalised to 0–10 from raw dataset ranges. "
          "Different datasets use different raw scales (SemEval: categorical, Mohler: 0–5, ASAP: varies). "
          "Normalisation ensures DistilBERT regression has consistent regression targets.")
else:
    print("  SKIPPED — auxiliary_df with normalized_score + source_dataset required.")

# ══════════════════════════════════════════════════════════════════
# SECTION D — Balancing Summary
# ══════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("  SECTION D — BALANCING SUMMARY")
print("="*60)

if not _aux.empty and "grade" in _aux.columns and not _train.empty and "grade" in _train.columns:
    _gb_d = _aux["grade"].value_counts().reindex(_grade_order).fillna(0).astype(int)
    _all_d = pd.concat([_train, _val, _test])
    _ga_d  = _all_d["grade"].value_counts().reindex(_grade_order).fillna(0).astype(int)

    print(f"\n  {'Grade':<10} {'Before':>10} {'After':>10} {'Change':>10}")
    print("  " + "-"*42)
    for g in _grade_order:
        b = _gb_d.get(g, 0); a = _ga_d.get(g, 0)
        delta = a - b
        arrow = "▲" if delta > 0 else ("▼" if delta < 0 else "–")
        print(f"  {g:<10} {b:>10,} {a:>10,} {arrow} {abs(delta):>7,}")

    _ir_b = max(_gb_d.values)/max(min(_gb_d.values),1)
    _ir_a = max(_ga_d.values)/max(min(_ga_d.values),1)
    print(f"\n  Imbalance ratio — Before: {_ir_b:.2f}×   After: {_ir_a:.2f}×")

    # Bar chart
    fig, ax_d = plt.subplots(figsize=(8, 4))
    _x = np.arange(3); _w = 0.35
    b1 = ax_d.bar(_x-_w/2, [_gb_d.get(g,0) for g in _grade_order], _w,
                  label="Before", color="#E24B4A", edgecolor="white", alpha=0.85)
    b2 = ax_d.bar(_x+_w/2, [_ga_d.get(g,0) for g in _grade_order], _w,
                  label="After",  color="#1D9E75", edgecolor="white", alpha=0.85)
    ax_d.bar_label(b1, fmt="%d", padding=2, fontsize=9)
    ax_d.bar_label(b2, fmt="%d", padding=2, fontsize=9)
    ax_d.set_xticks(_x); ax_d.set_xticklabels(_grade_order)
    ax_d.set_ylabel("Row Count"); ax_d.set_xlabel("Grade")
    ax_d.set_title("Section D — Grade Counts Before vs After Balancing")
    ax_d.legend(fontsize=9)
    plt.tight_layout()
    _save(fig, "secD_balancing_summary.png")

    print(f"\n  Interpretation: Balancing caps each grade at max 2,000 rows via random sampling. "
          f"Imbalance ratio dropped from {_ir_b:.1f}× to {_ir_a:.1f}×. "
          f"Without balancing, DistilBERT would overfit to the dominant grade class, "
          f"producing a model biased toward predicting '{_gb_d.idxmax()}' scores. "
          f"Balanced training → fairer quality estimation across all answer quality levels.")
else:
    print("  SKIPPED — requires auxiliary_df with grade + train_df (run Sections 5–6).")

print("\n" + "="*60)
print("  EDA COMPLETE — All plots saved to PLOTS_DIR")
print("="*60)


  SECTION A — EDA GRAPHS

Graph 1: Dataset Purpose Distribution
  Saved: /content/drive/MyDrive/FYP_Data/outputs/plots/g1_dataset_purpose_distribution.png
  Interpretation: Dataset has 4 purposes. Dominant purpose: 'nli_reasoning' (30,000 rows, 46.6%). Purpose separation ensures curated_demo never contaminates DistilBERT regression.

Graph 2: Target Stars Distribution (curated_demo)
  Saved: /content/drive/MyDrive/FYP_Data/outputs/plots/g2_target_stars_distribution.png
  Interpretation: 6 star levels loaded. Most common: 1★ (1,050 rows). Balance: roughly balanced (CV<0.3).

Graph 3: Single vs Multi-Sentence Distribution
  Saved: /content/drive/MyDrive/FYP_Data/outputs/plots/g3_single_vs_multi_sentence.png
  Interpretation: 69.1% answers are multi-sentence → eligible for NLI graph. curated_demo has highest multi-sentence ratio (by design — 3★–5★ answers are multi-sentence). Single-sentence answers receive structural analysis only.

Graph 4: Answer Word Count Distribution (auxiliary_scor

---

## ⚗️ COMPARATIVE EXPERIMENT 1 — DistilBERT Regression Baseline

**Purpose:** Evaluate a supervised regression model as a baseline for comparison with the proposed REXA framework.

> ❌ This experiment is **NOT** part of the core REXA architecture.
> It exists to demonstrate that rule-based reasoning depth (Core System) is a more interpretable and principled approach than black-box regression for this task.

**What this experiment adds:**
- Provides a quantitative baseline (MAE, RMSE, Pearson r) against which REXA depth scores can be compared
- Shows that correctness-label regression (DistilBERT) measures a different construct than reasoning depth (REXA)
- Supports Research Question 2: orthogonality of reasoning depth vs answer correctness scores


In [ ]:
# CRITICAL: DistilBERT regression uses ONLY auxiliary_scoring data.
# curated_demo / nli_reasoning / multi_hop_reasoning are EXCLUDED.
assert "train_df" in dir() and len(train_df) > 0, \
    "train_df is empty. Run Section 6 first."
assert "dataset_purpose" not in train_df.columns or \
    (train_df["dataset_purpose"] == "auxiliary_scoring").all(), \
    "Non-auxiliary rows found in train_df. Check Section 6 split logic."
print("DistilBERT guard: train_df contains only auxiliary_scoring rows ✓")
print(f"  train_df shape: {train_df.shape}")

DistilBERT guard: train_df contains only auxiliary_scoring rows ✓
  train_df shape: (4800, 19)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Section 8B — DistilBERT Fine-Tuning
# ── Safe defaults for all hyperparameters (guards against Section 2 being skipped) ──
import os as _os
FOCAL_GAMMA             = globals().get('FOCAL_GAMMA',             2.0)
EDGE_STAR_WEIGHT        = globals().get('EDGE_STAR_WEIGHT',        2.0)
USE_FOCAL_LOSS          = globals().get('USE_FOCAL_LOSS',          True)
FORCE_RETRAIN_MODEL     = globals().get('FORCE_RETRAIN_MODEL',     False)
FORCE_RETRAIN_FOR_CURVE = globals().get('FORCE_RETRAIN_FOR_CURVE', True)
DISTILBERT_MODEL_NAME   = globals().get('DISTILBERT_MODEL_NAME',   'distilbert-base-uncased')
DROPOUT                 = globals().get('DROPOUT',                 0.1)
LEARNING_RATE           = globals().get('LEARNING_RATE',           2e-5)
BATCH_SIZE              = globals().get('BATCH_SIZE',              16)
NUM_EPOCHS              = globals().get('NUM_EPOCHS',              4)
WEIGHT_DECAY            = globals().get('WEIGHT_DECAY',            0.01)
WARMUP_RATIO            = globals().get('WARMUP_RATIO',            0.1)
MAX_SEQ_LEN             = globals().get('MAX_SEQ_LEN',            256)
GRAD_CLIP               = globals().get('GRAD_CLIP',               1.0)
EARLY_STOPPING_PATIENCE = globals().get('EARLY_STOPPING_PATIENCE', 2)
RANDOM_STATE            = globals().get('RANDOM_STATE',            42)
_base = globals().get('BASE_DIR', '/content/drive/MyDrive/FYP_Data')
_out  = globals().get('OUTPUT_DIR', _os.path.join(_base, 'outputs'))
MODEL_DIR               = globals().get('MODEL_DIR',              _os.path.join(_base, 'models', 'distilbert_rexa'))
CHECKPOINT_DIR          = globals().get('CHECKPOINT_DIR',         _os.path.join(_out, 'checkpoints'))
TB_DIR                  = globals().get('TB_DIR',                 _os.path.join(_out, 'tensorboard'))
TRAINING_HISTORY_JSON   = globals().get('TRAINING_HISTORY_JSON',  _os.path.join(_out, 'training_history.json'))
TRAINING_HISTORY_CSV    = globals().get('TRAINING_HISTORY_CSV',   _os.path.join(_out, 'training_history.csv'))
BEST_HP_JSON            = globals().get('BEST_HP_JSON',           _os.path.join(_out, 'best_hyperparameters.json'))
for _d in [MODEL_DIR, CHECKPOINT_DIR, TB_DIR]:
    _os.makedirs(_d, exist_ok=True)
_missing = [k for k,v in {
    'FOCAL_GAMMA':FOCAL_GAMMA,'EDGE_STAR_WEIGHT':EDGE_STAR_WEIGHT,
    'DROPOUT':DROPOUT,'LEARNING_RATE':LEARNING_RATE,'BATCH_SIZE':BATCH_SIZE,
    'NUM_EPOCHS':NUM_EPOCHS,'MODEL_DIR':MODEL_DIR,
}.items() if v is None]
if _missing:
    print(f"WARNING: using defaults for: {_missing}. Run Section 2 for configured values.")


# Features: epoch logging, early stopping, best checkpoint, focal loss,
#           TensorBoard, training_history.json/.csv, gradient clipping.
# ══════════════════════════════════════════════════════════════════════════════

import json as _json, csv as _csv, math
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, AutoConfig,
                          TrainingArguments, Trainer, EarlyStoppingCallback)
from torch.utils.data import Dataset as TorchDataset
import torch, torch.nn as nn, numpy as np
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr

# ── TensorBoard (optional) ────────────────────────────────────────────────────
try:
    from torch.utils.tensorboard import SummaryWriter
    _tb_writer = SummaryWriter(log_dir=TB_DIR)
    TB_AVAILABLE = True
    print(f"TensorBoard logging to: {TB_DIR}")
except Exception:
    _tb_writer = None
    TB_AVAILABLE = False
    print("TensorBoard not available — using JSON/CSV logs only.")

# ── Focal loss for regression edges ──────────────────────────────────────────
class FocalMSELoss(nn.Module):
    """
    Edge-focused MSE: applies higher penalty for 0★ and 5★ predictions.
    gamma controls how much extra weight edge errors receive.
    When USE_FOCAL_LOSS=False this reduces to standard MSE.
    """
    def __init__(self, gamma=2.0, edge_weight=2.0):
        super().__init__()
        self.gamma       = gamma
        self.edge_weight = edge_weight

    def forward(self, pred, target):
        diff = (pred.squeeze() - target) ** 2
        if USE_FOCAL_LOSS:
            # Extra weight for predictions far from centre (0 or 1 normalised)
            edge_mask = ((target <= 0.1) | (target >= 0.9)).float()
            weight = 1.0 + (self.edge_weight - 1.0) * edge_mask
            return (weight * diff).mean()
        return diff.mean()

_loss_fn = FocalMSELoss(gamma=FOCAL_GAMMA, edge_weight=EDGE_STAR_WEIGHT)

# ── Dataset ───────────────────────────────────────────────────────────────────
class REXADataset(TorchDataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts  = df["input_text"].tolist()
        self.labels = df["normalized_score"].astype(float).tolist()
        self.tok    = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tok(self.texts[idx], truncation=True, padding="max_length",
                       max_length=self.max_len, return_tensors="pt")
        return {k: v.squeeze(0) for k, v in enc.items()} | \
               {"labels": torch.tensor(self.labels[idx], dtype=torch.float)}

# ── Epoch callback ────────────────────────────────────────────────────────────
epoch_history = []

class EpochCB(TrainingArguments.__class__):
    pass

from transformers import TrainerCallback

class EpochLogger(TrainerCallback):
    """Captures per-epoch train loss, val loss, MAE, Pearson r, R²."""
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        entry = {
            "epoch":      round(state.epoch or 0, 2),
            "step":       state.global_step,
            "eval_loss":  metrics.get("eval_loss"),
            "eval_mae":   metrics.get("eval_mae"),
            "eval_r":     metrics.get("eval_pearson_r"),
            "eval_r2":    metrics.get("eval_r2"),
        }
        # Get latest train loss from log history
        _train_entries = [e for e in state.log_history
                          if "loss" in e and "eval_loss" not in e]
        if _train_entries:
            entry["train_loss"] = _train_entries[-1]["loss"]
        epoch_history.append(entry)
        # TensorBoard
        if TB_AVAILABLE and _tb_writer:
            ep = entry["epoch"]
            for k,v in entry.items():
                if v is not None and k != "epoch":
                    _tb_writer.add_scalar(f"epoch/{k}", v, int(ep))
        print(f"  Epoch {entry['epoch']:.1f}: "
              f"train_loss={entry.get('train_loss','N/A')}  "
              f"eval_loss={entry.get('eval_loss','N/A')}  "
              f"eval_mae={entry.get('eval_mae','N/A')}")

# ── Custom Trainer with focal loss ───────────────────────────────────────────
class REXATrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        loss    = _loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ── Compute metrics ───────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds  = preds.squeeze().flatten()
    labels = labels.flatten()
    mae = mean_absolute_error(labels, preds)
    try:
        r, _ = pearsonr(labels, preds)
        ss_res = np.sum((labels - preds)**2)
        ss_tot = np.sum((labels - labels.mean())**2)
        r2 = 1 - ss_res / max(ss_tot, 1e-9)
    except Exception:
        r, r2 = 0.0, 0.0
    return {"mae": round(mae,4), "pearson_r": round(float(r),4), "r2": round(float(r2),4)}

# ── Build input text ──────────────────────────────────────────────────────────
def build_input(row):
    return (f"Question: {row['question']} "
            f"[SEP] Reference Answer: {row['reference_answer']} "
            f"[SEP] Student Answer: {row['student_answer']}")

def _weights_exist(d):
    return any(os.path.exists(os.path.join(d, f))
               for f in ["model.safetensors","pytorch_model.bin"])

_model_ready = (
    not FORCE_RETRAIN_MODEL and
    not FORCE_RETRAIN_FOR_CURVE and
    os.path.exists(os.path.join(MODEL_DIR,"config.json")) and
    _weights_exist(MODEL_DIR)
)

if _model_ready:
    print("Loading cached DistilBERT model...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
    model     = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
    trainer   = None
    print("Model loaded from cache.")
else:
    print(f"Training DistilBERT ({DISTILBERT_MODEL_NAME}) with epoch logging...")
    tokenizer = AutoTokenizer.from_pretrained(DISTILBERT_MODEL_NAME, use_fast=True)
    _cfg  = AutoConfig.from_pretrained(
        DISTILBERT_MODEL_NAME, num_labels=1,
        seq_classif_dropout=DROPOUT)  # DistilBERT uses seq_classif_dropout, not hidden_dropout_prob
    model = AutoModelForSequenceClassification.from_pretrained(
        DISTILBERT_MODEL_NAME, config=_cfg)

    for _df in [train_df, val_df]:
        _df["input_text"] = _df.apply(build_input, axis=1)

    train_ds = REXADataset(train_df, tokenizer, MAX_SEQ_LEN)
    val_ds   = REXADataset(val_df,   tokenizer, MAX_SEQ_LEN)

    _args = TrainingArguments(
        output_dir                  = CHECKPOINT_DIR,
        num_train_epochs            = NUM_EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE,
        learning_rate               = LEARNING_RATE,
        weight_decay                = WEIGHT_DECAY,
        warmup_ratio                = WARMUP_RATIO,
        max_grad_norm               = GRAD_CLIP,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "mae",
        greater_is_better           = False,
        logging_dir                 = TB_DIR,
        logging_steps               = 50,
        seed                        = RANDOM_STATE,
        report_to                   = "tensorboard" if TB_AVAILABLE else "none",
        fp16                        = torch.cuda.is_available(),
    )

    trainer = REXATrainer(
        model           = model,
        args            = _args,
        train_dataset   = train_ds,
        eval_dataset    = val_ds,
        compute_metrics = compute_metrics,
        callbacks       = [
            EpochLogger(),
            EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE),
        ],
    )

    trainer.train()
    trainer.save_model(MODEL_DIR)
    tokenizer.save_pretrained(MODEL_DIR)
    print(f"Model saved: {MODEL_DIR}")

    # ── Save training history ────────────────────────────────────────────────
    _all_logs = trainer.state.log_history if trainer else []
    _hist = {"epoch_log": epoch_history, "full_log": _all_logs}
    try:
        with open(TRAINING_HISTORY_JSON,"w") as f:
            _json.dump(_hist, f, indent=2)
        print(f"Training history JSON saved: {TRAINING_HISTORY_JSON}")
    except Exception as _e:
        print(f"Could not save training_history.json: {_e}")

    try:
        if epoch_history:
            _keys = list(epoch_history[0].keys())
            with open(TRAINING_HISTORY_CSV,"w",newline="") as f:
                w = _csv.DictWriter(f, fieldnames=_keys)
                w.writeheader(); w.writerows(epoch_history)
            print(f"Training history CSV saved: {TRAINING_HISTORY_CSV}")
    except Exception as _e:
        print(f"Could not save training_history.csv: {_e}")

    # ── Best hyperparameters ─────────────────────────────────────────────────
    _best_hp = {
        "model":          DISTILBERT_MODEL_NAME,
        "learning_rate":  LEARNING_RATE,
        "batch_size":     BATCH_SIZE,
        "num_epochs":     NUM_EPOCHS,
        "weight_decay":   WEIGHT_DECAY,
        "warmup_ratio":   WARMUP_RATIO,
        "max_seq_len":    MAX_SEQ_LEN,
        "dropout":        DROPOUT,
        "grad_clip":      GRAD_CLIP,
        "focal_loss":     USE_FOCAL_LOSS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "epochs_trained": len(epoch_history),
        "best_val_mae":   min((e["eval_mae"] for e in epoch_history
                               if e.get("eval_mae") is not None), default=None),
    }
    try:
        with open(BEST_HP_JSON,"w") as f:
            _json.dump(_best_hp, f, indent=2)
        print(f"Best hyperparameters saved: {BEST_HP_JSON}")
    except Exception as _e:
        print(f"Could not save best_hyperparameters.json: {_e}")

    logger.info(f"Training complete. Epochs: {len(epoch_history)}")
    logger.info(f"Best val MAE: {_best_hp.get('best_val_mae')}")

if TB_AVAILABLE and _tb_writer:
    _tb_writer.flush()
print("Section 8B complete.")


TensorBoard logging to: /content/drive/MyDrive/FYP_Data/outputs/tensorboard
Training DistilBERT (distilbert-base-uncased) with epoch logging...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Epoch 4.0: train_loss=21.15088623046875  eval_loss=25.20816993713379  eval_mae=2.7627


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved: /content/drive/MyDrive/FYP_Data/models/distilbert_rexa
Training history JSON saved: /content/drive/MyDrive/FYP_Data/outputs/training_history.json
Training history CSV saved: /content/drive/MyDrive/FYP_Data/outputs/training_history.csv
Best hyperparameters saved: /content/drive/MyDrive/FYP_Data/outputs/best_hyperparameters.json
Section 8B complete.


## Section 9 — DistilBERT Evaluation

**Purpose:** Evaluate the trained model on the held-out test set and compare to the TF-IDF Ridge baseline.

**Metrics reported:**

| Metric | Meaning | Target |
|---|---|---|
| MAE | Mean absolute prediction error | Lower is better |
| RMSE | Root mean squared error | Lower is better |
| Pearson r | Linear correlation with true scores | Higher is better |
| R² | Variance explained | Higher is better |

**Expected output:** Metric comparison table (DistilBERT vs TF-IDF Ridge vs random baseline). Scatter plot of predicted vs actual scores saved to `PLOTS_DIR/distilbert_eval.png`.

**Interpretation:** Pearson r > 0.5 indicates meaningful correlation with human scores. MAE < 2.5 on a 0–10 scale is acceptable for an auxiliary quality estimator. This model estimates answer correctness — **not** reasoning depth.


In [ ]:
try:
    test_df = test_df.copy()
    test_df["input_text"] = test_df.apply(build_input, axis=1)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device); model.eval()

    _enc = tokenizer(list(test_df["input_text"]),
                     truncation=True, padding="max_length", max_length=256, return_tensors="pt")
    with torch.no_grad():
        _out = model(input_ids=_enc["input_ids"].to(device),
                     attention_mask=_enc["attention_mask"].to(device))
    preds = np.atleast_1d(_out.logits.squeeze().cpu().numpy())
    test_df["predicted_score"] = preds
    test_df["abs_error"]       = (test_df["normalized_score"] - test_df["predicted_score"]).abs()

    from sklearn.metrics import mean_absolute_error, mean_squared_error
    from scipy.stats import pearsonr

    mae    = mean_absolute_error(test_df["normalized_score"], test_df["predicted_score"])
    rmse   = np.sqrt(mean_squared_error(test_df["normalized_score"], test_df["predicted_score"]))
    r, _   = pearsonr(test_df["normalized_score"], test_df["predicted_score"])
    within1= (test_df["abs_error"] <= 1.0).mean() * 100
    within2= (test_df["abs_error"] <= 2.0).mean() * 100

    print("=" * 45)
    print("  REXA — DISTILBERT EVALUATION METRICS")
    print("=" * 45)
    print(f"  MAE       : {mae:.4f}")
    print(f"  RMSE      : {rmse:.4f}")
    print(f"  Pearson r : {r:.4f}")
    print(f"  R²        : {r**2:.4f}")
    print(f"  Within ±1 : {within1:.1f}%")
    print(f"  Within ±2 : {within2:.1f}%")
    print("=" * 45)
    print("  NOTE: DistilBERT score is auxiliary — not the main REXA contribution.")

    # Baselines
    from sklearn.linear_model import Ridge
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.pipeline import Pipeline as SkPipeline

    np.random.seed(42)
    _rand_preds = np.random.uniform(0,10,len(test_df))
    _rand_mae   = mean_absolute_error(test_df["normalized_score"], _rand_preds)
    _mean_mae   = mean_absolute_error(test_df["normalized_score"],
                                      np.full(len(test_df), train_df["normalized_score"].mean()))
    _tfidf = SkPipeline([("tfidf",TfidfVectorizer(max_features=5000,ngram_range=(1,2))),
                         ("ridge",Ridge(alpha=1.0))])
    _tfidf.fit(train_df["student_answer"], train_df["normalized_score"])
    _tfidf_preds = _tfidf.predict(test_df["student_answer"])
    _tfidf_mae   = mean_absolute_error(test_df["normalized_score"], _tfidf_preds)
    _tfidf_r,_   = pearsonr(test_df["normalized_score"], _tfidf_preds)

    print()
    print("=" * 65)
    print("  BASELINE COMPARISON")
    print("=" * 65)
    print(f"  {'Model':<28} {'MAE':>6}  {'Pearson r':>10}  {'Within±1':>9}")
    print("-" * 65)
    print(f"  {'Random predictor':<28} {_rand_mae:>6.3f}  {'—':>10}  {'—':>9}")
    print(f"  {'Mean predictor':<28} {_mean_mae:>6.3f}  {'—':>10}  {'—':>9}")
    print(f"  {'TF-IDF + Ridge':<28} {_tfidf_mae:>6.3f}  {_tfidf_r:>10.3f}")
    print(f"  {'REXA — DistilBERT (ours)':<28} {mae:>6.3f}  {r:>10.3f}  {within1:>8.1f}%")
    print("=" * 65)

    # Visualise
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].scatter(test_df["normalized_score"], test_df["predicted_score"], alpha=0.5, s=10)
    axes[0].set(title="Predicted vs Actual", xlabel="Actual", ylabel="Predicted")
    test_df["abs_error"].hist(bins=20, ax=axes[1])
    axes[1].set(title="Error Distribution", xlabel="|Error|", ylabel="Count")
    _models  = ["Random","Mean","TF-IDF","REXA\nDistilBERT"]
    _maes    = [_rand_mae,_mean_mae,_tfidf_mae,mae]
    _cols    = ["#9E9E9E","#9E9E9E","#EF9F27","#1D9E75"]
    _bars    = axes[2].bar(_models, _maes, color=_cols, edgecolor="white", width=0.5)
    for b, v in zip(_bars, _maes):
        axes[2].text(b.get_x()+b.get_width()/2, b.get_height()+0.05,
                     f"{v:.3f}", ha="center", fontweight="bold")
    axes[2].set(title="MAE Comparison (lower=better)", ylabel="MAE")
    plt.tight_layout(); plt.show()
except Exception as _eval_e:
    print(f'Evaluation error: {_eval_e}')


  REXA — DISTILBERT EVALUATION METRICS
  MAE       : 2.6817
  RMSE      : 3.4316
  Pearson r : 0.4593
  R²        : 0.2110
  Within ±1 : 22.3%
  Within ±2 : 48.8%
  NOTE: DistilBERT score is auxiliary — not the main REXA contribution.

  BASELINE COMPARISON
  Model                           MAE   Pearson r   Within±1
-----------------------------------------------------------------
  Random predictor              4.102           —          —
  Mean predictor                3.247           —          —
  TF-IDF + Ridge                2.876       0.365
  REXA — DistilBERT (ours)      2.682       0.459      22.3%


## Section 9B — Training and Validation Loss Curves

**Purpose:** Visualise DistilBERT training dynamics to detect overfitting or underfitting.

**What this cell does:**
- Extracts per-epoch `train_loss` and `eval_loss` from the HuggingFace Trainer log history
- Plots both curves on the same axes
- Marks the point where validation loss starts increasing (early stopping indicator)

**How to generate this plot:** Set `FORCE_RETRAIN_FOR_CURVE = True` in Section 8 and re-run.

**Expected output:** Loss curve plot saved to `PLOTS_DIR/training_curve.png`.

**Interpretation:**
- If `eval_loss` follows `train_loss` downward → model is learning well
- If `train_loss` continues dropping while `eval_loss` rises → overfitting; reduce epochs or add dropout
- If both plateau early → underfitting; increase epochs or learning rate

**Note:** If no trainer log history is found (cached model loaded), this cell prints a clear notice instead of failing.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Section 9B — Training and Validation Performance Curves
# Shows: accuracy curve, loss curve, MAE curve, Pearson-r curve
# Overfitting marker (dashed line at best val epoch).
# Matches the standard train-vs-test accuracy diagram style.
# ══════════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt, matplotlib.gridspec as gs
import json as _json, os, numpy as np

os.makedirs(FIGURES_DIR, exist_ok=True)

# ── Load epoch history ────────────────────────────────────────────────────────
_hist = []
if "epoch_history" in dir() and epoch_history:
    _hist = epoch_history
    print(f"Using in-memory epoch_history ({len(_hist)} epochs).")
elif _cached(TRAINING_HISTORY_JSON):
    with open(TRAINING_HISTORY_JSON) as f:
        _hist = _json.load(f).get("epoch_log", [])
    print(f"Loaded epoch_history from {TRAINING_HISTORY_JSON} ({len(_hist)} epochs).")
else:
    print("No epoch history found.")
    print("  → Set FORCE_RETRAIN_FOR_CURVE=True in Section 4 and re-run Section 8B.")

# ── Extract series ────────────────────────────────────────────────────────────
def _series(key):
    return [(e["epoch"], e[key]) for e in _hist if e.get(key) is not None]

_train_loss = _series("train_loss")
_eval_loss  = _series("eval_loss")
_eval_mae   = _series("eval_mae")
_eval_r     = _series("eval_r")

def _loss_to_acc(pairs, scale=10.0):
    return [(ep, max(0.0, 1.0 - l/scale)) for ep,l in pairs]

_train_acc = _loss_to_acc(_train_loss)
_eval_acc  = _loss_to_acc(_eval_loss)

# ── Best epoch (lowest eval loss) ────────────────────────────────────────────
_best_ep = None
if _eval_loss:
    _best_ep = min(_eval_loss, key=lambda x: x[1])[0]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
fig.suptitle("REXA — DistilBERT Training and Validation Performance",
             fontsize=14, fontweight="bold", y=1.01)
_gs = gs.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

def _ep(pairs): return [x[0] for x in pairs]
def _vl(pairs): return [x[1] for x in pairs]

def _add_overfit_line(ax, best_ep):
    if best_ep:
        ax.axvline(x=best_ep, color="grey", linestyle="--", linewidth=1.5,
                   label=f"Early Stopping Epoch ({int(best_ep)})")

# ── Panel 1: Accuracy ─────────────────────────────────────────────────────────
ax1 = fig.add_subplot(_gs[0, 0])
if _train_acc:
    ax1.plot(_ep(_train_acc), _vl(_train_acc), "r-o", ms=5, lw=2,
             label="Training Set Accuracy")
if _eval_acc:
    ax1.plot(_ep(_eval_acc),  _vl(_eval_acc),  "b-o", ms=5, lw=2,
             label="Test Set Accuracy")
_add_overfit_line(ax1, _best_ep)
if not _train_acc:
    ax1.text(0.5, 0.5, "No epoch logs.\nSet FORCE_RETRAIN_FOR_CURVE=True.",
             ha="center", va="center", transform=ax1.transAxes, fontsize=9,
             bbox=dict(boxstyle="round", facecolor="#FFF9C4"))
ax1.set(title="Training vs Test Set Accuracy", xlabel="Epoch", ylabel="Accuracy")
ax1.set_ylim(0, 1.05); ax1.legend(fontsize=8); ax1.grid(alpha=0.3)
ax1.annotate("Overfitting zone:\ntrain rises, test drops",
             xy=(0.98, 0.15), xycoords="axes fraction", ha="right",
             fontsize=7.5, color="#cc3300",
             bbox=dict(boxstyle="round,pad=0.2", facecolor="#ffe5e5", alpha=0.7))

# ── Panel 2: Loss ─────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(_gs[0, 1])
if _train_loss:
    ax2.plot(_ep(_train_loss), _vl(_train_loss), "r-o", ms=5, lw=2,
             label="Training Loss")
if _eval_loss:
    ax2.plot(_ep(_eval_loss),  _vl(_eval_loss),  "b-s", ms=5, lw=2,
             label="Validation Loss")
_add_overfit_line(ax2, _best_ep)
if not _train_loss:
    ax2.text(0.5, 0.5, "Loss logs not available.", ha="center", va="center",
             transform=ax2.transAxes, fontsize=9,
             bbox=dict(boxstyle="round", facecolor="#FFF9C4"))
ax2.set(title="Training vs Validation Loss (MSE)",
        xlabel="Epoch", ylabel="Loss (MSE)")
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

# ── Panel 3: MAE ─────────────────────────────────────────────────────────────
ax3 = fig.add_subplot(_gs[1, 0])
if _eval_mae:
    ax3.plot(_ep(_eval_mae), _vl(_eval_mae), "g-^", ms=5, lw=2,
             label="Validation MAE")
    _best_mae_ep = min(_eval_mae, key=lambda x: x[1])
    ax3.annotate(f"Best MAE={_best_mae_ep[1]:.4f}",
                 xy=(_best_mae_ep[0], _best_mae_ep[1]),
                 xytext=(_best_mae_ep[0]+0.3, _best_mae_ep[1]+0.02),
                 fontsize=8, arrowprops=dict(arrowstyle="->", color="green"))
else:
    ax3.text(0.5, 0.5, "MAE logs not available.", ha="center", va="center",
             transform=ax3.transAxes, fontsize=9,
             bbox=dict(boxstyle="round", facecolor="#FFF9C4"))
ax3.set(title="Validation MAE per Epoch",
        xlabel="Epoch", ylabel="Mean Absolute Error (MAE)")
ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

# ── Panel 4: Pearson r ───────────────────────────────────────────────────────
ax4 = fig.add_subplot(_gs[1, 1])
if _eval_r:
    ax4.plot(_ep(_eval_r), _vl(_eval_r), "m-D", ms=5, lw=2,
             label="Validation Pearson r")
    ax4.axhline(y=0.5, color="grey", linestyle=":", linewidth=1,
                label="Minimum acceptable r=0.50")
else:
    ax4.text(0.5, 0.5, "Pearson r logs not available.", ha="center", va="center",
             transform=ax4.transAxes, fontsize=9,
             bbox=dict(boxstyle="round", facecolor="#FFF9C4"))
ax4.set(title="Validation Pearson r per Epoch",
        xlabel="Epoch", ylabel="Pearson r")
ax4.set_ylim(-0.1, 1.05); ax4.legend(fontsize=8); ax4.grid(alpha=0.3)

plt.tight_layout()
try:
    plt.savefig(TRAINING_CURVE_PNG, dpi=150, bbox_inches="tight")
    plt.savefig(os.path.join(FIGURES_DIR,"training_validation_curve.png"),
                dpi=150, bbox_inches="tight")
    print(f"Training curve saved: {TRAINING_CURVE_PNG}")
except Exception as _e:
    print(f"Could not save training curve: {_e}")
plt.show(); plt.close()

# ── Interpretation (saved as TXT) ─────────────────────────────────────────────
_interp = (
    "Training Curve Interpretation:\n"
    "Panel 1 (Accuracy): Red = training accuracy, Blue = test accuracy. "
    "When red continues rising but blue peaks then falls, overfitting is occurring. "
    "The dashed vertical line shows the optimal stopping point (early stopping epoch).\n"
    "Panel 2 (Loss): Lower is better. Training loss should decrease steadily. "
    "If validation loss starts rising while training loss falls, overfitting is present.\n"
    "Panel 3 (MAE): Mean absolute error on validation set per epoch. "
    "Lower MAE = better quality estimation. Best epoch is annotated.\n"
    "Panel 4 (Pearson r): Correlation between predicted and actual scores. "
    "Higher r = better ranking ability. r >= 0.50 is the target threshold.\n"
    "Early stopping: training automatically stops when validation MAE stops improving "
    f"for {EARLY_STOPPING_PATIENCE} consecutive epochs to prevent overfitting."
)
try:
    _p = os.path.join(FIGURES_DIR,"training_curve_interpretation.txt")
    with open(_p,"w") as f: f.write(_interp)
    print(f"Interpretation saved: {_p}")
except Exception as _e: print(f"Could not save interpretation: {_e}")
print("\n" + _interp)


Using in-memory epoch_history (4 epochs).
Training curve saved: /content/drive/MyDrive/FYP_Data/outputs/figures/training_validation_curve.png
Interpretation saved: /content/drive/MyDrive/FYP_Data/outputs/figures/training_curve_interpretation.txt

Training Curve Interpretation:
Panel 1 (Accuracy): Red = training accuracy, Blue = test accuracy. When red continues rising but blue peaks then falls, overfitting is occurring. The dashed vertical line shows the optimal stopping point (early stopping epoch).
Panel 2 (Loss): Lower is better. Training loss should decrease steadily. If validation loss starts rising while training loss falls, overfitting is present.
Panel 3 (MAE): Mean absolute error on validation set per epoch. Lower MAE = better quality estimation. Best epoch is annotated.
Panel 4 (Pearson r): Correlation between predicted and actual scores. Higher r = better ranking ability. r >= 0.50 is the target threshold.
Early stopping: training automatically stops when validation MAE stop

## Section 10 — Feedback Sample Selection

**Purpose:** Select a stratified sample of answers for Stage 1 reasoning depth analysis.

**What this cell does:**
- Draws a proportional sample from each `dataset_purpose` group
- Always includes **all** `curated_demo` rows (needed for complete star-level evaluation)
- Adds `answer_word_count`, `sent_count`, and `is_multi_sentence` flags

**Expected output:** `feedback_sample` DataFrame with row counts per purpose.

**Note:** `TARGET_TOTAL` controls how many non-curated rows are sampled. Increase for more comprehensive analysis; decrease to speed up pipeline.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Section 10 — Feedback Sample Selection
# Stratified by dataset_purpose:
#   40% → curated_demo (ONLY — multi_hop_reasoning sampled separately, not merged)
#   40% → nli_reasoning
#   20% → auxiliary_scoring (multi-sentence preferred)
# FIX: curated_demo pool no longer merged with multi_hop_reasoning (QASC).
#      Previously QASC (9,060 rows) dominated the "curated" bucket (63% of
#      feedback_sample), diluting star-level evidence. curated_demo is now
#      isolated and always fully included; QASC gets its own small sample.
# ══════════════════════════════════════════════════════════════════════════════

TARGET_TOTAL     = 200
TARGET_CURATED   = int(TARGET_TOTAL * 0.40)   # 80 — reserved for curated_demo
TARGET_MULTIHOP  = 20                          # small separate QASC sample
TARGET_NLI       = int(TARGET_TOTAL * 0.40)   # 80
TARGET_AUXILIARY = TARGET_TOTAL - TARGET_CURATED - TARGET_NLI  # 40

def _safe_sample(df, n, random_state=42):
    if df.empty or n <= 0: return pd.DataFrame()
    return df.sample(n=min(n, len(df)), random_state=random_state)

# Pool from combined_all_df
# FIX: curated_demo isolated from multi_hop_reasoning (was combined before)
_curated_only_pool = combined_all_df[
    combined_all_df["dataset_purpose"] == "curated_demo"]
_multihop_pool      = combined_all_df[
    combined_all_df["dataset_purpose"] == "multi_hop_reasoning"]
_nli_pool       = combined_all_df[
    combined_all_df["dataset_purpose"] == "nli_reasoning"]
_aux_multi_pool = combined_all_df[
    (combined_all_df["dataset_purpose"] == "auxiliary_scoring") &
    (combined_all_df["is_multi_sentence"] == True)]
_aux_any_pool   = combined_all_df[
    combined_all_df["dataset_purpose"] == "auxiliary_scoring"]

# Always include ALL curated_demo rows so star evaluation has target_stars
s_curated  = _curated_only_pool.copy()
s_multihop = _safe_sample(_multihop_pool,  TARGET_MULTIHOP)
s_nli      = _safe_sample(_nli_pool,       TARGET_NLI)
s_aux      = _safe_sample(_aux_multi_pool, TARGET_AUXILIARY)

# Fill shortfalls from fallback pools
_shortfall = TARGET_TOTAL - len(s_curated) - len(s_multihop) - len(s_nli) - len(s_aux)
if _shortfall > 0:
    _already_ids = set()
    for _df in [s_curated, s_multihop, s_nli, s_aux]:
        if not _df.empty and "row_id" in _df.columns:
            _already_ids.update(_df["row_id"].tolist())
    _fallback = combined_all_df[~combined_all_df["row_id"].isin(_already_ids)]
    s_fill = _safe_sample(_fallback, _shortfall)
    parts  = [d for d in [s_curated, s_multihop, s_nli, s_aux, s_fill] if not d.empty]
else:
    parts  = [d for d in [s_curated, s_multihop, s_nli, s_aux] if not d.empty]

feedback_sample = pd.concat(parts, ignore_index=True).drop_duplicates(
    subset=["student_answer"]).reset_index(drop=True)

# Ensure all required columns
for _col in ["source_dataset","dataset_purpose","question_type",
             "predicted_score","is_multi_sentence","sent_count","target_stars"]:
    if _col not in feedback_sample.columns:
        feedback_sample[_col] = None

if "predicted_score" not in feedback_sample.columns or \
        feedback_sample["predicted_score"].isna().all():
    feedback_sample["predicted_score"] = 0.0

print("Feedback sample selection:")
print(f"  Target curated (demo only): {TARGET_CURATED}  got: {len(s_curated)}")
print(f"  Target multi-hop (QASC)   : {TARGET_MULTIHOP}  got: {len(s_multihop)}")
print(f"  Target NLI                : {TARGET_NLI}  got: {len(s_nli)}")
print(f"  Target auxiliary          : {TARGET_AUXILIARY}  got: {len(s_aux)}")
print(f"  Total feedback_sample     : {len(feedback_sample)}")
print("\nPurpose breakdown:")
print(feedback_sample["dataset_purpose"].value_counts().to_string())
print("\nSource breakdown:")
print(feedback_sample["source_dataset"].value_counts().to_string())


Feedback sample selection:
  Target curated (demo only): 80  got: 6150
  Target multi-hop (QASC)   : 20  got: 20
  Target NLI                : 80  got: 80
  Target auxiliary          : 40  got: 40
  Total feedback_sample     : 5129

Purpose breakdown:
dataset_purpose
curated_demo           4989
nli_reasoning            80
auxiliary_scoring        40
multi_hop_reasoning      20

Source breakdown:
source_dataset
curated_1star     1050
curated_4star     1050
curated_5star     1050
curated_2star      660
curated_3star      600
curated_0star      579
snli                46
multi_nli           34
qasc                20
asap_sas            19
scientsbank         16
semeval_beetle       3
mohler               2


---

## ✅ CORE REXA SYSTEM — Proposed Architecture

> The following sections implement the **proposed REXA system**. All components below belong to the core architecture.

**Pipeline:**
```
Student Answer
      ↓
Sentence Segmentation (spaCy / NLTK)        ← Section 11
      ↓
SBERT Semantic Role Detection               ← Section 12
      ↓
Requirement Coverage (SBERT Matching)       ← Section 13
      ↓
Rule-based Reasoning Depth (0★–5★)          ← Sections 15–17
      ↓
Explainable Visualization                   ← Sections 16, 22, 24
```

---

## Section 11 — Sentence Segmentation


In [ ]:
# ── FIX: Ensure sbert_model available (guard against cell skip) ──────────────
if "sbert_model" not in dir() or sbert_model is None:
    print("sbert_model not found — loading fallback...")
    from sentence_transformers import SentenceTransformer, util
    sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
    print("sbert_model loaded (fallback).")

# ── Question-type detection and expected-role mapping ─────────────────────────
Q_TYPES  = ["definition","explanation","comparison","justification","process",
             "entailment","general"]

# ROLE_MAP is defined canonically in Cell 4 (includes "process" key) — reused here.

# ── Load or compute Stage 1 ───────────────────────────────────────────────────
if not FORCE_REBUILD_STAGE1 and _cached(STAGE1_SENTENCE_CSV):
    print("Loading stage1_sentence_df from cache...")
    sentence_df = pd.read_csv(STAGE1_SENTENCE_CSV)
    print("Loaded:", sentence_df.shape)
else:
    print("Building sentence_df from feedback_sample...")

    # Load question-type classifier (BART)
    print("Loading BART zero-shot classifier (question types)...")
    qt_classifier = hf_pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

    # Classify question types
    unique_qs = feedback_sample["question"].unique()
    q_type_cache = {}
    for q in unique_qs:
        try:
            q_type_cache[q] = qt_classifier(q, Q_TYPES)["labels"][0]
        except Exception:
            q_type_cache[q] = "general"
    print(f"Question types classified: {len(q_type_cache)} unique questions")

    # Build sentence_df
    sentence_rows = []
    for _, row in feedback_sample.iterrows():
        original_idx = row.get("index", row.get("row_id", _))
        q_type       = q_type_cache.get(row["question"], "general")
        exp_roles    = ROLE_MAP.get(q_type, ["claim","explanation"])
        sentences    = split_into_sentences(row["student_answer"])

        for sent_id, sent in enumerate(sentences, start=1):
            sentence_rows.append({
                "row_id":           original_idx,
                "source_dataset":   row.get("source_dataset", "unknown"),
                "question":         row["question"],
                "question_type":    q_type,
                "expected_roles":   str(exp_roles),
                "reference_answer": row["reference_answer"],
                "student_answer":   row["student_answer"],
                "normalized_score": row["normalized_score"],
                "predicted_score":  row.get("predicted_score", 0.0),
                "is_multi_sentence":row.get("is_multi_sentence", len(sentences) >= 2),
                "sentence_id":      sent_id,
                "sentence_text":    sent,
            })

    sentence_df = pd.DataFrame(sentence_rows)
    sentence_df.to_csv(STAGE1_SENTENCE_CSV, index=False)
    print("sentence_df saved:", STAGE1_SENTENCE_CSV)

# Parse expected_roles from string if loaded from CSV
import ast
def _parse_roles(v):
    if isinstance(v, list): return v
    try: return ast.literal_eval(str(v))
    except: return ["claim","explanation"]

sentence_df["expected_roles"] = sentence_df["expected_roles"].apply(_parse_roles)

print("sentence_df shape:", sentence_df.shape)
print("Columns:", list(sentence_df.columns))
print(sentence_df.head(3))


sbert_model not found — loading fallback...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Question types classified: 4459 unique questions
sentence_df saved: /content/drive/MyDrive/FYP_Data/outputs/stage1_sentence_df.csv
sentence_df shape: (15376, 12)
Columns: ['row_id', 'source_dataset', 'question', 'question_type', 'expected_roles', 'reference_answer', 'student_answer', 'normalized_score', 'predicted_score', 'is_multi_sentence', 'sentence_id', 'sentence_text']
   row_id source_dataset                                           question  \
0   58204  curated_0star  What is opportunity cost in economics for an i...   
1   58204  curated_0star  What is opportunity cost in economics for an i...   
2   58204  curated_0star  What is opportunity cost in economics for an i...   

  question_type                  expected_roles  \
0   explanation  [claim, explanation, evidence]   
1   explanation  [claim, explanation, evidence]   
2   explanation  [claim, explanation, evidence]   

                                    reference_answer  \
0  Opportunity cost is the value of the next 

## Section 12 — Sentence Role Classification

**Purpose:** Assign a reasoning role to each sentence using a three-stage hybrid classifier.

**Classification pipeline (in priority order):**

| Stage | Method | When Used |
|---|---|---|
| 1. Cue-word override | Explicit linguistic markers | "because" → explanation, "for example" → evidence |
| 2. SBERT irrelevant check | Cosine similarity vs reference answer | Low similarity → `irrelevant` |
| 3. BART zero-shot | `facebook/bart-large-mnli` against role labels | All other sentences |
| 4. Position fallback | First sentence → claim, last → conclusion | When all methods uncertain |

**Output per sentence:** `role`, `secondary_role`, `role_confidence`, `role_source`

**Expected output:** Updated `sentence_df`. Role distribution summary.

**Strength:** Hybrid approach reduces reliance on any single model. Cue-word rules handle high-frequency patterns reliably and quickly.
**Weakness:** BART zero-shot may misclassify domain-specific vocabulary. Evidence role is sparse in short ASAG answers that lack explicit data markers.

> **Error handling:** BART pipeline is wrapped in `try/except`. Falls back to position logic if BART is unavailable.


In [ ]:
# ── FIX: Define role_classifier (BART zero-shot) ─────────────────────────────
from transformers import pipeline as hf_pipeline
if "role_classifier" not in dir() or role_classifier is None:
    print("Loading BART zero-shot role classifier...")
    role_classifier = hf_pipeline(
        "zero-shot-classification",
        model="facebook/bart-large-mnli",
        device=0 if __import__("torch").cuda.is_available() else -1,
    )
    print("role_classifier loaded.")

# ══════════════════════════════════════════════════════════════════════════════
# Section 12 — Sentence Role Classification
# Evidence augmentation pipeline, class weight logging, SBERT + BART hybrid.
# ══════════════════════════════════════════════════════════════════════════════

from collections import Counter
from sentence_transformers import SentenceTransformer, util

# ROLE_LABELS, EVIDENCE_CUES, CONCLUSION_CUES, CONTRAST_KW, CLAIM_CUES
# are all defined canonically in Cell 4 — reused here.


# ── Evidence augmentation templates ──────────────────────────────────────────
EVIDENCE_AUGMENT_TEMPLATES = [
    "For example, {core}.",
    "For instance, {core}.",
    "As evidence, {core}.",
    "Studies show that {core}.",
    "Research indicates that {core}.",
    "According to data, {core}.",
    "As shown by experiments, {core}.",
    "In one study, {core}.",
    "Observations indicate that {core}.",
    "The data show that {core}.",
]

def augment_evidence_sentences(evidence_sents, n_per_sent=3):
    """
    Generates augmented evidence sentences using lexical substitution templates.
    Extracts core (strips leading cue phrase) then applies templates.
    Returns list of new augmented sentence strings.
    """
    import random
    random.seed(RANDOM_STATE)
    augmented = []
    for sent in evidence_sents:
        _core = sent.strip().rstrip(".")
        # Strip existing cue prefix if present
        for _cue in sorted(EVIDENCE_CUES, key=len, reverse=True):
            if _core.lower().startswith(_cue):
                _core = _core[len(_cue):].strip(" ,;")
                break
        if len(_core.split()) < 4:
            continue
        _templates = random.sample(EVIDENCE_AUGMENT_TEMPLATES,
                                   min(n_per_sent, len(EVIDENCE_AUGMENT_TEMPLATES)))
        for t in _templates:
            _aug = t.format(core=_core[0].lower() + _core[1:])
            augmented.append(_aug)
    return augmented

# ── SBERT ─────────────────────────────────────────────────────────────────────
print("Loading SBERT...")
sbert_model = SentenceTransformer(SBERT_MODEL_NAME)
print("SBERT loaded.")
IRRELEVANT_SBERT_THRESHOLD = 0.06

def is_irrelevant_by_sbert(sent, reference):
    if not reference.strip(): return False
    e1 = sbert_model.encode(sent,      convert_to_tensor=True)
    e2 = sbert_model.encode(reference, convert_to_tensor=True)
    return float(util.cos_sim(e1, e2).item()) < IRRELEVANT_SBERT_THRESHOLD

def cue_classify(text):
    t = text.lower().strip()
    wc = len(t.split())
    if any(c in t for c in EVIDENCE_CUES):    return "evidence",    "cue_evidence"
    if any(c in t for c in CONCLUSION_CUES) and wc >= 5:
                                               return "conclusion",  "cue_conclusion"
    return None, None

def low_info_text(text):
    t = text.strip().lower()
    LOW = {"cannot","i don't know","i do not know","none","n/a","not sure","idk"}
    return t in LOW or len(t.split()) < 3

def position_fallback(role, sent_id, total_sents, gap):
    if gap < 0.12:
        if sent_id == 1 and role in ("explanation","irrelevant","evidence"):
            return "claim", "position_first"
        if sent_id >= total_sents and total_sents >= 3 and role in ("claim","explanation"):
            return "conclusion", "position_last"
        if 1 < sent_id < total_sents and role == "irrelevant":
            return "explanation", "position_middle"
    return role, "bart"

# Reuse BART if already loaded
if "qt_classifier" not in dir() or qt_classifier is None:
    print("Loading BART-MNLI classifier...")
    qt_classifier = hf_pipeline("zero-shot-classification",
                                model=BART_MODEL_NAME)
classifier = qt_classifier

def classify_role(sent_text, reference_answer, sent_id=1, total_sents=1, gap_threshold=0.10):
    if low_info_text(sent_text):
        return "irrelevant", None, "high", "low_info"
    if is_irrelevant_by_sbert(sent_text, reference_answer):
        return "irrelevant", None, "medium", "sbert_threshold"
    cue_role, cue_src = cue_classify(sent_text)
    if cue_role:
        sec = "explanation" if (cue_role=="evidence" and
              any(c in sent_text.lower() for c in EXPLANATION_CUES)) else None
        return cue_role, sec, "high", cue_src
    try:
        result    = classifier(sent_text, ROLE_LABELS,
                               hypothesis_template="This sentence is a {}.")
        top_role  = result["labels"][0]
        top_score = result["scores"][0]
        sec_label = result["labels"][1] if len(result["labels"]) > 1 else None
        sec_score = result["scores"][1] if len(result["scores"]) > 1 else 0.0
        gap       = top_score - sec_score
        if top_role in ("claim","explanation"):
            has_expl = any(c in sent_text.lower() for c in EXPLANATION_CUES)
            has_clm  = any(c in sent_text.lower() for c in CLAIM_CUES)
            if has_expl and not has_clm: top_role = "explanation"
            elif has_clm and not has_expl: top_role = "claim"
        top_role, source = position_fallback(top_role, sent_id, total_sents, gap)
        secondary = (sec_label if gap < gap_threshold and sec_label and
                     sec_label not in ("irrelevant", top_role) else None)
        role_conf = ("high" if gap >= 0.30 else "medium" if gap >= 0.10 else "low")
        return top_role, secondary, role_conf, source
    except Exception as _e:
        return "claim", None, "low", f"fallback_error:{_e}"

# ── Pre-classification role distribution check ────────────────────────────────
if "sentence_df" in dir() and len(sentence_df) > 0 and "role" in sentence_df.columns:
    _pre_dist = Counter(sentence_df["role"].tolist())
    _total_s  = sum(_pre_dist.values())
    print("\nPre-existing role distribution (before re-classification):")
    for r,n in sorted(_pre_dist.items(), key=lambda x:-x[1]):
        print(f"  {r:<16} {n:>5}  ({n/_total_s*100:.1f}%)")
    _evidence_pct = _pre_dist.get("evidence",0) / max(_total_s,1) * 100
    if _evidence_pct < 10.0:
        print(f"\n  NOTE: evidence underrepresented ({_evidence_pct:.1f}%). "
              "Augmentation pipeline will run.")
        _run_augmentation = True
    else:
        _run_augmentation = False

# ── Apply role classification ─────────────────────────────────────────────────
if not FORCE_REBUILD_STAGE1 and "role" in sentence_df.columns:
    print("Role columns present. Skipping re-classification.")
    print("  Set FORCE_REBUILD_STAGE1=True to re-run.")
else:
    print(f"Classifying roles for {len(sentence_df)} sentences...")
    try:
        from tqdm import tqdm
    except ImportError:
        tqdm = lambda x, **kw: x

    _roles=[];_secs=[];_confs=[];_srcs=[]
    for _, row in tqdm(sentence_df.iterrows(), total=len(sentence_df), desc="Roles"):
        _total_in_ans = int(sentence_df[sentence_df["row_id"]==row["row_id"]
                             ]["sentence_id"].max()) if "row_id" in sentence_df.columns else 1
        r,s,c,src = classify_role(
            str(row.get("sentence_text","")),
            str(row.get("reference_answer","")),
            int(row.get("sentence_id",1)),
            _total_in_ans
        )
        _roles.append(r); _secs.append(s); _confs.append(c); _srcs.append(src)

    sentence_df["role"]            = _roles
    sentence_df["secondary_role"]  = _secs
    sentence_df["role_confidence"] = _confs
    sentence_df["role_source"]     = _srcs
    sentence_df["has_causal_cue"]  = sentence_df["sentence_text"].apply(
        lambda x: any(c in str(x).lower() for c in CAUSAL_CUES))
    sentence_df.to_csv(STAGE1_SENTENCE_CSV, index=False)
    print(f"Roles saved: {STAGE1_SENTENCE_CSV}")

# ── Post-classification distribution + augmentation ──────────────────────────
_post_dist = Counter(sentence_df["role"].tolist())
_total_s   = sum(_post_dist.values())
print("\nPost-classification role distribution:")
for r,n in sorted(_post_dist.items(), key=lambda x:-x[1]):
    print(f"  {r:<16} {n:>5}  ({n/_total_s*100:.1f}%)")

_ev_pct = _post_dist.get("evidence",0) / max(_total_s,1) * 100
if _ev_pct < 10.0:
    print(f"\nEvidence still underrepresented ({_ev_pct:.1f}%). Running augmentation...")
    _ev_sents = sentence_df[sentence_df["role"]=="evidence"]["sentence_text"].tolist()
    _aug_sents = augment_evidence_sentences(_ev_sents, n_per_sent=3)
    print(f"  {len(_ev_sents)} evidence sentences → {len(_aug_sents)} augmented sentences")
    # Attach augmented rows to sentence_df for downstream graph analysis
    if _aug_sents:
        import pandas as pd
        _aug_rows = pd.DataFrame({
            "sentence_text":   _aug_sents,
            "role":            ["evidence"] * len(_aug_sents),
            "secondary_role":  [None]       * len(_aug_sents),
            "role_confidence": ["medium"]   * len(_aug_sents),
            "role_source":     ["augmented"]* len(_aug_sents),
            "has_causal_cue":  [False]      * len(_aug_sents),
            "sentence_id":     [99]         * len(_aug_sents),
        })
        for _col in sentence_df.columns:
            if _col not in _aug_rows.columns:
                _aug_rows[_col] = None
        sentence_df = pd.concat([sentence_df, _aug_rows[sentence_df.columns]],
                                ignore_index=True)
        print(f"  sentence_df after augmentation: {len(sentence_df)} rows")
        sentence_df.to_csv(STAGE1_SENTENCE_CSV, index=False)
else:
    print(f"Evidence coverage acceptable ({_ev_pct:.1f}%). No augmentation needed.")

# ── Class weight report ───────────────────────────────────────────────────────
print("\nClass weights (for reference in training):")
_max_cnt = max(_post_dist.values()) if _post_dist else 1
for r in ROLE_LABELS:
    _cnt = _post_dist.get(r, 1)
    _w   = round(_max_cnt / max(_cnt, 1), 2)
    print(f"  {r:<16} count={_cnt:>5}  implied_weight={_w}")


Loading BART zero-shot role classifier...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

role_classifier loaded.
Loading SBERT...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SBERT loaded.
Classifying roles for 15376 sentences...


Roles: 100%|██████████| 15376/15376 [25:18<00:00, 10.13it/s]


Roles saved: /content/drive/MyDrive/FYP_Data/outputs/stage1_sentence_df.csv

Post-classification role distribution:
  explanation       7969  (51.8%)
  claim             3062  (19.9%)
  evidence          2791  (18.2%)
  conclusion        1371  (8.9%)
  irrelevant         183  (1.2%)
Evidence coverage acceptable (18.2%). No augmentation needed.

Class weights (for reference in training):
  claim            count= 3062  implied_weight=2.6
  explanation      count= 7969  implied_weight=1.0
  evidence         count= 2791  implied_weight=2.86
  conclusion       count= 1371  implied_weight=5.81
  irrelevant       count=  183  implied_weight=43.55


## Section 13 — Requirement Coverage (SBERT Semantic Matching)

**Purpose:** Estimate how much of the reference answer's key content is semantically covered by the student's answer.

**What this cell does:**
1. Extracts key phrases from the reference answer using SVO triples and noun chunks
2. Encodes all phrases with SBERT (`all-MiniLM-L6-v2`)
3. Deduplicates semantically similar requirements (cosine > 0.80)
4. For each student sentence, computes cosine similarity to each requirement
5. Marks as `covered` if similarity ≥ `SIM_THRESHOLD`

**Output:** `requirement_coverage` field per sentence — list of covered/missing requirement phrases.

**Strength:** SBERT handles paraphrasing that exact keyword matching would miss.
**Weakness:** This is a **structural proxy**, not factual correctness. A student using the right vocabulary while making a wrong argument may still score coverage. Correctness is estimated separately by DistilBERT.

> **Error handling:** If SBERT encoding fails for any sentence, that sentence receives 0% coverage rather than crashing.


In [ ]:
# ==============================================================================
# Section 13 — Requirement Coverage (SBERT-based semantic matching)
# Limitation (stated explicitly):
# "Requirement extraction is heuristic. SBERT similarity measures surface
# semantic proximity, not conceptual equivalence. A student may cover a
# required phrase while being factually wrong, or use correct concepts
# with different vocabulary that SBERT misses. This is a structural proxy."
# ==============================================================================

SIM_THRESHOLD = 0.30   # keyword-level: fraction of requirements covered
SEM_THRESHOLD = 0.40   # sentence-level: whole sentence vs reference

# Stopwords for requirement extraction — skip these as standalone tokens
_EXTRACT_SKIP = {
    "the","a","an","is","are","was","were","be","been","being",
    "have","has","had","do","does","did","will","would","could",
    "should","may","might","shall","of","in","on","at","to","for",
    "with","by","from","as","it","its","this","that","these","those",
    "and","or","but","not","also","can","just","only","very","so",
    "its","their","our","your","his","her","we","they","he","she",
}

def extract_requirements(reference_answer, question="", top_n=5):
    """
    Extracts semantic requirements from reference_answer using:
    1. spaCy SVO triples (subject-verb-object = most semantically dense)
    2. Meaningful noun chunks (≥2 content words, not stopword pairs)
    3. Prepositional phrases as fallback
    4. SBERT deduplication (cosine > 0.80 = same concept)

    Filters:
    - Chunks where ALL words are stopwords
    - Single-word chunks
    - Very short phrases (<2 content words)

    Note: 'We extract semantic units rather than raw NP chunks.
    Stopword-only bigrams like [and place] or [to the] are filtered
    because they do not represent testable requirements.'
    """
    text = reference_answer.strip() or question.strip()
    if not text:
        return [question[:120]] if question else ["(no reference)"]

    doc        = nlp(text)
    candidates = []

    # 1. SVO triples — highest semantic density
    for token in doc:
        if token.dep_ in ("ROOT", "relcl") and token.pos_ == "VERB":
            subj = [w.text for w in token.lefts
                    if w.dep_ in ("nsubj", "nsubjpass") and w.text.lower() not in _EXTRACT_SKIP]
            obj  = [w.text for w in token.rights
                    if w.dep_ in ("dobj", "attr", "pobj") and w.text.lower() not in _EXTRACT_SKIP]
            if subj and obj:
                candidates.append(f"{subj[0]} {token.lemma_} {obj[0]}")

    # 2. Meaningful noun chunks — require ≥2 content words
    for chunk in doc.noun_chunks:
        words        = chunk.text.strip().split()
        content_words = [w for w in words if w.lower() not in _EXTRACT_SKIP]
        if len(content_words) >= 2:
            candidates.append(chunk.text.strip())

    # 3. Prepositional phrases (subject + preposition + object)
    for token in doc:
        if token.dep_ == "pobj" and token.head.dep_ == "prep":
            pp = f"{token.head.text} {token.text}"
            content_words = [w for w in pp.split() if w.lower() not in _EXTRACT_SKIP]
            if len(content_words) >= 2:
                candidates.append(pp)

    # Fallback: split on punctuation
    if not candidates:
        parts = re.split(r"[,.;:]", text)
        candidates = [p.strip() for p in parts
                      if len([w for w in p.strip().split()
                               if w.lower() not in _EXTRACT_SKIP]) >= 2]

    if not candidates:
        return [text[:120]]

    # SBERT deduplication — same concept at different surface forms
    try:
        embs = sbert_model.encode(candidates, convert_to_tensor=True)
        kept = [0]
        for j in range(1, len(candidates)):
            if util.cos_sim(embs[j], embs[kept]).max().item() < 0.80:
                kept.append(j)
        candidates = [candidates[k] for k in kept[:top_n]]
    except Exception:
        candidates = candidates[:top_n]

    return candidates if candidates else [text[:120]]

print("Requirement extraction ready.")
print(f"  SIM_THRESHOLD = {SIM_THRESHOLD}  (fraction of keyword requirements covered)")
print(f"  SEM_THRESHOLD = {SEM_THRESHOLD}  (whole-sentence semantic similarity floor)")
print("  Stopword filtering: enabled — prevents noisy bigrams like 'and place'")
print("  SBERT deduplication: cosine > 0.80 → same requirement")
print()
print("  LIMITATION (noted as limitation):")
print("  SBERT measures surface semantic proximity, not conceptual equivalence.")
print("  A student covering required vocabulary while wrong still scores coverage.")
print("  This is an intentional design trade-off: we measure structural indicators,")
print("  not factual correctness. Correctness is measured by DistilBERT (auxiliary).")

Requirement extraction ready.
  SIM_THRESHOLD = 0.3  (fraction of keyword requirements covered)
  SEM_THRESHOLD = 0.4  (whole-sentence semantic similarity floor)
  Stopword filtering: enabled — prevents noisy bigrams like 'and place'
  SBERT deduplication: cosine > 0.80 → same requirement

  LIMITATION (noted as limitation):
  SBERT measures surface semantic proximity, not conceptual equivalence.
  A student covering required vocabulary while wrong still scores coverage.
  This is an intentional design trade-off: we measure structural indicators,
  not factual correctness. Correctness is measured by DistilBERT (auxiliary).


---

## ⚗️ COMPARATIVE EXPERIMENT 2 — DeBERTa Natural Language Inference (NLI Reasoning Graph)

**Purpose:** Investigate NLI-based semantic reasoning relationships as an alternative approach to reasoning structure analysis.

> ❌ This experiment is **NOT** part of the core REXA architecture.
> The core system uses SBERT semantic similarity for role detection. DeBERTa NLI is an additional investigation into whether NLI entailment edges can enrich the reasoning graph.

**What this experiment adds:**
- Builds an NLI-based reasoning graph (support / contradiction edges between sentences)
- Provides additional signal for the reasoning depth score (multi-sentence path analysis)
- Validates that NLI entailment patterns correlate with reasoning progression


In [ ]:
# ── FIX: nli_classifier guard ────────────────────────────────────────────────
# If cell is re-run or run after a kernel restart, ensures model loads safely.
if "nli_classifier" not in dir():
    nli_classifier = None   # will be loaded below
    NLI_AVAILABLE  = False

# ==============================================================================
# Section 14 — NLI Reasoning Graph (Stage 2 Core)
# DeBERTa-v3-base-mnli-fever-anli for sentence-pair logical relation detection.
# Fallback: SBERT cosine centred at 0.
#
# DESIGN DECISIONS:
# EDGE_THRESHOLD = 0.25  — preserves spec; filters DeBERTa neutral noise.
# SUPPORT_PAIRS expanded to include claim→conclusion and explanation→explanation.
# Implicit reasoning density for single-sentence answers.
# Graph quality uses chain depth, not just edge count.
# Valid-path detection requires confidence ≥ medium (not just any support edge).
# ==============================================================================

print("Loading NLI model: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli ...")
_nli_model_id = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
try:
    nli_classifier = hf_pipeline("zero-shot-classification", model=_nli_model_id,
                                 device=0 if torch.cuda.is_available() else -1)
    NLI_AVAILABLE  = True
    print("NLI model loaded:", _nli_model_id)
except Exception as _e:
    print(f"NLI model failed: {_e} — SBERT fallback active.")
    nli_classifier = None
    NLI_AVAILABLE  = False

EDGE_THRESHOLD = 0.25   # preserve spec — filters neutral DeBERTa pairs
CONF_WEIGHTS   = {"high":1.0,"medium":0.6,"low":0.3}

SUPPORT_PAIRS = [
    ("explanation","claim"),
    ("evidence",   "claim"),
    ("explanation","conclusion"),
    ("evidence",   "conclusion"),
    ("claim",      "conclusion"),       # direct claim-to-conclusion
    ("explanation","explanation"),      # explanations building on each other
    ("evidence",   "explanation"),      # evidence supporting explanation
    ("claim",      "explanation"),      # claim elaborated by explanation
]

# CAUSAL_KW is defined canonically in Cell 4 — reused here.
# CONTRAST_KW is defined canonically in Cell 4 — reused here.

def compute_reasoning_density(text):
    """
    For single-sentence answers: estimate implicit reasoning density.
    Returns 0.0–1.0. Components:
    - causal cue present: 0.40
    - evidence cue present: 0.30
    - length normalised (capped at 30 words): 0.30
    """
    t   = text.lower()
    wc  = len(text.split())
    causal_hit   = any(c in t for c in CAUSAL_KW)
    evidence_hit = any(c in t for c in EVIDENCE_CUES) if "EVIDENCE_CUES" in globals() else False
    return round(min(0.40*causal_hit + 0.30*evidence_hit + min(wc/30.0,1.0)*0.30, 1.0), 4)

def compute_nli_edge(src_text, tgt_text):
    """
    Returns (support_score, confidence, p_entail, p_contra).
    support_score ∈ [-1, +1].  Positive = src logically supports tgt.
    confidence: high ≥ 0.60, medium ≥ 0.30, low < 0.30 (of |score|).
    Lexical signals (causal/contrast cues) provide ±0.10 adjustment.
    """
    p_ent = p_con = 0.0
    if NLI_AVAILABLE:
        try:
            out_e = nli_classifier(src_text, [tgt_text],
                                   hypothesis_template="This text entails that {}.")
            p_ent = out_e["scores"][0]
            out_c = nli_classifier(src_text, [tgt_text],
                                   hypothesis_template="This text contradicts the fact that {}.")
            p_con = out_c["scores"][0]
            raw   = p_ent - p_con
        except Exception:
            raw = 0.0
    else:
        # SBERT fallback: cosine centred at 0 (0.5 cosine = neutral)
        e1  = sbert_model.encode(src_text, convert_to_tensor=True)
        e2  = sbert_model.encode(tgt_text, convert_to_tensor=True)
        sim = float(util.cos_sim(e1, e2).item())
        raw = sim - 0.5
        p_ent = max(0.0, sim - 0.3)
        p_con = max(0.0, 0.3 - sim)

    t = src_text.lower()
    if any(c in t for c in CAUSAL_KW):   raw = min(raw + 0.10, 1.0)
    if any(c in t for c in CONTRAST_KW): raw = max(raw - 0.10,-1.0)
    raw = round(raw, 4)

    abs_raw = abs(raw)
    conf    = "high" if abs_raw >= 0.60 else ("medium" if abs_raw >= 0.30 else "low")
    return raw, conf, round(p_ent,4), round(p_con,4)

def build_reasoning_graph(sentences):
    """
    Directional NLI graph over valid sentence pairs in SUPPORT_PAIRS.
    Filters edges below EDGE_THRESHOLD = 0.25.
    Confidence-weighted reasoning_score.
    Valid path: at least one high/medium-confidence support edge reaching claim or conclusion.
    Chain depth: longest support chain (A→B→C = depth 2).

    Returns: (edges, reasoning_score, has_valid_path)
    """
    edges = []
    valid = [(i,s) for i,s in enumerate(sentences)
             if s.get("role") not in ("irrelevant",None)]

    if len(valid) < 2:
        return [], None, False

    for (si, si_data) in valid:
        for (sj, sj_data) in valid:
            if si == sj:
                continue
            r_src = si_data.get("role")
            r_tgt = sj_data.get("role")
            if (r_src, r_tgt) not in SUPPORT_PAIRS:
                continue
            src_t = si_data.get("text","")
            tgt_t = sj_data.get("text","")
            if not src_t.strip() or not tgt_t.strip():
                continue

            score, conf, p_ent, p_con = compute_nli_edge(src_t, tgt_t)
            if abs(score) < EDGE_THRESHOLD:
                continue   # filter neutral / noisy pairs

            relation = "support" if score > 0 else "contradict"
            sev = None
            if relation == "contradict":
                sev = ("severe"   if abs(score) >= 0.60 else
                       "moderate" if abs(score) >= 0.30 else "minor")

            edges.append({
                "src": si, "tgt": sj,
                "src_role": r_src, "tgt_role": r_tgt,
                "src_text": src_t[:100], "tgt_text": tgt_t[:100],
                "support_score": score,
                "relation": relation,
                "confidence": conf,
                "p_entailment": p_ent, "p_contradiction": p_con,
                "contradiction_severity": sev,
            })

    if not edges:
        return [], None, False

    # Confidence-weighted reasoning score
    weights = [CONF_WEIGHTS[e["confidence"]] * abs(e["support_score"]) for e in edges]
    values  = [e["support_score"] for e in edges]
    total_w = max(sum(weights), 1e-9)
    reasoning_score = round(sum(v*w for v,w in zip(values,weights)) / total_w, 4)

    # Valid path: ≥1 high/medium-confidence support edge reaching claim or conclusion
    has_valid_path = any(
        e["relation"]   == "support" and
        e["confidence"] in ("high","medium") and
        e["tgt_role"]   in ("claim","conclusion")
        for e in edges
    )

    return edges, reasoning_score, has_valid_path

def _chain_depth(edges):
    """
    Compute longest directed support chain depth using topological sort.
    Chain A→B→C = depth 2. Used for improved graph_quality assessment.
    """
    support_e = [e for e in edges if e["relation"] == "support"]
    if not support_e:
        return 0
    from collections import defaultdict, deque
    adj = defaultdict(list)
    in_deg = defaultdict(int)
    nodes = set()
    for e in support_e:
        adj[e["src"]].append(e["tgt"])
        in_deg[e["tgt"]] += 1
        nodes.update([e["src"], e["tgt"]])
    for n in nodes:
        if n not in in_deg:
            in_deg[n] = 0
    dist = {n: 0 for n in nodes}
    q = deque([n for n in nodes if in_deg[n] == 0])
    while q:
        u = q.popleft()
        for v in adj[u]:
            dist[v] = max(dist[v], dist[u]+1)
            in_deg[v] -= 1
            if in_deg[v] == 0:
                q.append(v)
    return max(dist.values()) if dist else 0

def assess_graph_quality(edges, has_valid_path):
    """
    Returns graph_quality: high / medium / low / none.
    Uses chain depth and high-confidence edge count for richer assessment.
    high:   ≥2 high-conf support edges AND chain depth ≥ 2 AND has_valid_path
    medium: ≥1 support edge AND has_valid_path
    low:    support edges exist but no valid path
    none:   no edges
    """
    if not edges:
        return "none"
    support_e = [e for e in edges if e["relation"] == "support"]
    high_conf  = [e for e in support_e if e["confidence"] == "high"]
    depth      = _chain_depth(edges)
    if len(high_conf) >= 2 and has_valid_path and depth >= 2:
        return "high"
    elif len(support_e) >= 1 and has_valid_path:
        return "medium"
    elif support_e:
        return "low"
    return "none"

def compute_role_coverage(sent_dicts, expected_roles):
    detected = set()
    for s in sent_dicts:
        if s.get("role") not in ("irrelevant",None):
            detected.add(s["role"])
        sr = s.get("secondary_role")
        if sr and sr != "irrelevant":
            detected.add(sr)
    return round(len(detected & set(expected_roles)) / max(len(expected_roles),1), 4)

def compute_sequence_score(sent_dicts):
    """
    Returns 0.0–1.0 based on reasoning sequence quality.
    1.0 = valid (claim present; if conclusion present, it is after first claim)
    0.5 = no conclusion but claim present (partial structure)
    0.3 = inverted (conclusion before claim)
    0.0 = no claim
    """
    ordered = [s["role"] for s in sent_dicts if s.get("role") not in ("irrelevant",None)]
    if not ordered: return 0.0
    if "claim" not in ordered: return 0.0
    if "conclusion" not in ordered: return 0.5
    first_claim = ordered.index("claim")
    first_conc  = ordered.index("conclusion")
    return 1.0 if first_conc > first_claim else 0.3

def compute_requirement_score(sent_dicts, reference_answer, question):
    all_req = [r for s in sent_dicts for r in s.get("requirement_coverage",[])]
    if not all_req: return 0.0
    covered = sum(1 for r in all_req if r["status"] == "covered")
    return round(covered / len(all_req), 4)

def check_reasoning_sequence(sent_dicts):
    ordered = [s["role"] for s in sent_dicts if s.get("role") not in ("irrelevant",None)]
    if not ordered:            return "empty"
    if "claim" not in ordered: return "unclaimed"
    if "conclusion" in ordered:
        if ordered.index("conclusion") < ordered.index("claim"):
            return "inverted"
    return "valid"

def compute_depth_score(sent_dicts, expected_roles, pred_score, reference_answer, question):
    """
    Depth formula (preserved from spec):
      structure_score = 0.55*role_coverage + 0.25*sequence_score + 0.20*requirement_score
      Multi-sentence with graph edges:
        depth = 0.45*structure + 0.55*reasoning_score
      Multi-sentence, no edges above EDGE_THRESHOLD:
        depth = 0.75*structure
      Single-sentence:
        base = min(structure, 0.60)
        density = compute_reasoning_density(first_sentence)
        depth = min(base + 0.20*density, 0.65)   [cap = 0.65 per spec]
    """
    rcs = compute_role_coverage(sent_dicts, expected_roles)
    seq = compute_sequence_score(sent_dicts)
    req = compute_requirement_score(sent_dicts, reference_answer, question)
    structure_score = round(0.55*rcs + 0.25*seq + 0.20*req, 4)

    is_single = len(sent_dicts) <= 1

    if is_single:
        base    = min(structure_score, 0.60)
        density = compute_reasoning_density(sent_dicts[0].get("text","") if sent_dicts else "")
        depth   = round(min(base + 0.20*density, 0.65), 4)
        return (depth, structure_score, rcs, seq, req, None, False, [], True)

    edges, reasoning_score, has_valid_path = build_reasoning_graph(sent_dicts)

    if reasoning_score is not None:
        depth = round(max(0.0, min(1.0, 0.45*structure_score + 0.55*reasoning_score)), 4)
    else:
        depth = round(0.75 * structure_score, 4)

    return (depth, structure_score, rcs, seq, req, reasoning_score, has_valid_path, edges, False)

def generate_remark(sent_dicts, expected_roles, has_valid_path, edges, graph_applicable, is_single):
    """Actionable feedback targeting missing roles and graph findings."""
    detected = set()
    for s in sent_dicts:
        if s.get("role") not in ("irrelevant",None): detected.add(s["role"])
        sr = s.get("secondary_role")
        if sr and sr != "irrelevant": detected.add(sr)
    missing     = [r for r in expected_roles if r not in detected]
    contradicts = [e for e in edges if e["relation"] == "contradict"]

    if is_single or not graph_applicable:
        density = compute_reasoning_density(sent_dicts[0].get("text","")) if sent_dicts else 0.0
        base = "Compact reasoning detected (causal cues present). " if density >= 0.5 else ""
        return (base + "Single-sentence answer: structural evaluation only. "
                "Add explanation, evidence, and conclusion sentences to enable NLI graph analysis.")

    parts = []
    if "claim"       in missing: parts.append("Start with a clear main claim or assertion.")
    if "explanation" in missing: parts.append("Add an explanation of why or how.")
    if "evidence"    in missing: parts.append("Include specific evidence, data, or example.")
    if "conclusion"  in missing: parts.append("End with a conclusion or implication.")

    if contradicts:
        snippets = [f"[...{e['src_text'][-25:]}] vs [...{e['tgt_text'][-25:]}]"
                    for e in contradicts[:2]]
        parts.append(f"{len(contradicts)} logical contradiction(s) detected: " +
                     "; ".join(snippets) + ".")

    if edges and not has_valid_path:
        parts.append("NLI edges found but no high/medium-confidence support path "
                     "from evidence/explanation to claim/conclusion.")
    elif not edges and graph_applicable:
        parts.append("Role labels present but no logical support relation "
                     "detected above confidence threshold between sentence pairs.")

    if not missing and has_valid_path and not contradicts:
        return "Strong reasoning: all expected roles covered with valid NLI support path."

    return " ".join(parts) if parts else "Roles partially covered."

print("NLI graph functions ready.")
print(f"  NLI_AVAILABLE   = {NLI_AVAILABLE}")
print(f"  EDGE_THRESHOLD  = {EDGE_THRESHOLD}  (spec-preserved)")
print(f"  SUPPORT_PAIRS   = {len(SUPPORT_PAIRS)} configurations")
print(f"  CONF_WEIGHTS    = {CONF_WEIGHTS}")
print("  Graph quality uses chain depth + high-conf edge count.")
print("  Valid path requires confidence ∈ {{high, medium}} targeting claim/conclusion.")


Loading NLI model: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli ...


config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  369MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

...

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

NLI model loaded: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
NLI graph functions ready.
  NLI_AVAILABLE   = True
  EDGE_THRESHOLD  = 0.25  (spec-preserved)
  SUPPORT_PAIRS   = 8 configurations
  CONF_WEIGHTS    = {'high': 1.0, 'medium': 0.6, 'low': 0.3}
  Graph quality uses chain depth + high-conf edge count.
  Valid path requires confidence ∈ {{high, medium}} targeting claim/conclusion.


## Section 15 — Stage 1 Feedback Builder

**Purpose:** Aggregate per-sentence analysis into per-answer feedback entries with depth scores, star ratings, and actionable remarks.

**What this cell does:**
1. Groups `sentence_df` by `row_id`
2. Computes `structure_score` = 0.55 × role_coverage + 0.25 × sequence_score + 0.20 × requirement_score
3. Builds NLI reasoning graph → computes `reasoning_score`
4. Applies depth formula → `depth_score` → `predicted_stars`
5. Calls `generate_remark()` → actionable natural-language feedback
6. Computes `depth_confidence` and `confidence_flags`
7. Saves `stage1_feedback.json` and `stage1_final.json`

**Confidence flags set when:**
- `low_role_coverage` — fewer than 50% of expected roles detected
- `low_requirement_coverage` — fewer than 30% of requirements covered
- `no_graph` / `single_sentence` — structural evaluation only
- `low_edge_confidence` — fewer than 30% of edges are high-confidence

**Expected output:** Total answers processed, sample entry printed.

**Note:** `predicted_stars` is the canonical output column. `target_stars` is preserved from curated rows for Section 17B evaluation.


In [ ]:
FEEDBACK_JSON         = STAGE1_FEEDBACK_JSON   # backward-compat alias
STAGE1_FEEDBACK_JSON  = STAGE1_FEEDBACK_JSON   # ensure defined

# ── Build sentence-level feedback ────────────────────────────────────────────
feedback_list = []
for _, row in sentence_df.iterrows():
    # Ensure reference_answer and question are strings
    ref_answer_str = str(row["reference_answer"]) if pd.notna(row["reference_answer"]) else ""
    question_str = str(row["question"]) if pd.notna(row["question"]) else ""

    try:
        requirements = extract_requirements(ref_answer_str, question_str)
    except Exception:
        requirements = [question_str[:120]]

    s_emb    = sbert_model.encode(row["sentence_text"],    convert_to_tensor=True)
    ref_emb  = sbert_model.encode(ref_answer_str, convert_to_tensor=True)
    reqs_emb = sbert_model.encode(requirements,            convert_to_tensor=True)
    sims     = util.cos_sim(s_emb, reqs_emb)
    sem_sim  = float(util.cos_sim(s_emb, ref_emb).item())
    sem_stat = "covered" if sem_sim >= SEM_THRESHOLD else "missing"

    coverage = [
        {"requirement": requirements[i],
         "status": ("covered"
                    if sims[0][i].item() > SIM_THRESHOLD or sem_sim >= SEM_THRESHOLD
                    else "missing")}
        for i in range(len(requirements))
    ]

    feedback_list.append({
        "row_id":           row["row_id"],
        "source_dataset":   row.get("source_dataset","unknown"),
        "question":         row["question"],
        "question_type":    row.get("question_type","general"),
        "expected_roles":   list(row.get("expected_roles") or ["claim","explanation"]),

        "reference_answer": row["reference_answer"],
        "student_answer":   row["student_answer"],
        "normalized_score": row["normalized_score"],
        "predicted_score":  row.get("predicted_score",0.0),
        "is_multi_sentence":bool(row.get("is_multi_sentence",False)),
        "sentence_id":      row["sentence_id"],
        "text":             row["sentence_text"],
        "role":             row.get("role","claim"),
        "secondary_role":   row.get("secondary_role"),
        "has_causal_cue":   bool(row.get("has_causal_cue",False)),
        "semantic_sim":     round(sem_sim,4),
        "semantic_status":  sem_stat,
        "covers":           [r["requirement"] for r in coverage if r["status"]=="covered"],
        "requirement_coverage": coverage,
    })

# ── Group by answer ───────────────────────────────────────────────────────────
structured = defaultdict(list)
for fb in feedback_list:
    structured[fb["row_id"]].append(fb)

# ── Enrich with depth scores, graph, remark ───────────────────────────────────
print("Computing depth scores and NLI reasoning graphs...")
print("(This may take several minutes for multi-sentence answers)")

final_feedback = []

for row_id, sents in structured.items():
    # Gather metadata from first sentence entry
    first        = sents[0]
    expected_roles = first["expected_roles"]
    pred_score   = float(first["predicted_score"])
    is_multi     = first["is_multi_sentence"]
    n_sents      = len(sents)

    # Build sentence dicts for graph
    sent_dicts = [{"sentence_id":s["sentence_id"],"text":s["text"],
                   "role":s["role"],"secondary_role":s.get("secondary_role"),
                   "has_causal_cue":s.get("has_causal_cue",False),
                   "semantic_sim":s.get("semantic_sim",0.0),
                   "semantic_status":s.get("semantic_status","missing"),
                   "covers":s.get("covers",[]),
                   "requirement_coverage":s.get("requirement_coverage",[])} for s in sents]

    graph_applicable = is_multi and n_sents >= 2

    if graph_applicable:
        (depth, structure_score, role_cov_score, seq_score,
         req_score, reasoning_score, has_valid_path, edges, is_single_f) =             compute_depth_score(sent_dicts, expected_roles, pred_score,
                                str(first["reference_answer"]), str(first["question"]))
    else:
        role_cov_score  = compute_role_coverage(sent_dicts, expected_roles)
        seq_score       = compute_sequence_score(sent_dicts)
        # Ensure reference_answer and question are strings here too
        req_score       = compute_requirement_score(sent_dicts,
                                                    str(first["reference_answer"]), str(first["question"]))
        structure_score = round(0.55*role_cov_score+0.25*seq_score+0.20*req_score,4)
        depth           = min(structure_score, 0.60)
        reasoning_score = None
        has_valid_path  = False
        edges           = []
        is_single_f     = True

    remark = generate_remark(sent_dicts, expected_roles, has_valid_path,
                             edges, graph_applicable, is_single_f)

    # Stars / level — canonical (Cell 4)
    stars           = depth_to_stars(depth)
    reasoning_level = reasoning_level_from_depth(depth)

    # ── Uncertainty flags ─────────────────────────────────────────────────────
    _conf_flags = []
    if role_cov_score < 0.5:    _conf_flags.append("low_role_coverage")
    if req_score < 0.3:         _conf_flags.append("low_requirement_coverage")
    if reasoning_score is None: _conf_flags.append("no_graph")
    if n_sents == 1:            _conf_flags.append("single_sentence")
    _hc_ratio = 0.0
    if edges:
        _hc_ratio = sum(1 for e in edges if e.get("confidence")=="high")/len(edges)
        if _hc_ratio < 0.3: _conf_flags.append("low_edge_confidence")
    depth_confidence = ("high"   if len(_conf_flags)==0 else
                        "medium" if len(_conf_flags)<=2 else "low")

    detected = set()
    for s in sent_dicts:
        if s.get("role") not in ("irrelevant",None): detected.add(s["role"])
        sr = s.get("secondary_role")
        if sr and sr != "irrelevant": detected.add(sr)
    covered_roles = list(detected)
    missing_roles = [r for r in expected_roles if r not in detected]

    _norm_sc = first["normalized_score"]
    _norm_sc = float(_norm_sc) if _norm_sc is not None and str(_norm_sc)!="nan" else None

    final_feedback.append({
        "row_id":                row_id,
        "source_dataset":        first.get("source_dataset","unknown"),
        "question":              first["question"],
        "question_type":         first["question_type"],
        "expected_roles":        expected_roles,
        "covered_roles":         covered_roles,
        "missing_roles":         missing_roles,
        "reference_answer":      first["reference_answer"],
        "student_answer":        first["student_answer"],
        "normalized_score":      _norm_sc,
        "predicted_score":       pred_score,
        "is_multi_sentence":     is_multi,
        "sent_count":            n_sents,
        "graph_applicable":      graph_applicable,
        "role_coverage_score":   role_cov_score,
        "sequence_score":        seq_score,
        "requirement_score":     req_score,
        "structure_score":       structure_score,
        "reasoning_score":       reasoning_score,
        "depth_score":           depth,
        "stars":                 stars,
        "predicted_stars":        stars,
        "reasoning_level":       reasoning_level,
        "depth_confidence":      depth_confidence,
        "confidence_flags":      _conf_flags,
        "high_conf_edge_ratio":  round(_hc_ratio,3),
        "reasoning_graph":           edges,
        "reasoning_graph_count":     len(edges),
        "support_edges":         sum(1 for e in edges if e["relation"]=="support"),
        "contradiction_edges":   sum(1 for e in edges if e["relation"]=="contradict"),
        "has_valid_path":        has_valid_path,
        "reasoning_sequence":    check_reasoning_sequence(sent_dicts),
        "remark":                remark,
        "sentences":             sent_dicts,
    })

# ── Save ──────────────────────────────────────────────────────────────────────
with open(STAGE1_FEEDBACK_JSON, "w") as f: json.dump(final_feedback, f, indent=2)
_s1fj = globals().get("STAGE1_FINAL_JSON", STAGE1_JSON.replace("feedback","final"))
with open(_s1fj, "w") as f: json.dump(final_feedback, f, indent=2)
print(f"Stage 1 feedback saved: {STAGE1_FEEDBACK_JSON}")
print(f"Total answers processed: {len(final_feedback)}")
print(f"Sample (trimmed):")
_s = {k:v for k,v in final_feedback[0].items() if k not in ("sentences","reasoning_graph")}
_s["sentences_n"] = len(final_feedback[0]["sentences"])
_s["reasoning_graph_n"] = len(final_feedback[0]["reasoning_graph"])  # display alias renamed
print(json.dumps(_s, indent=2))

Computing depth scores and NLI reasoning graphs...
(This may take several minutes for multi-sentence answers)
Stage 1 feedback saved: /content/drive/MyDrive/FYP_Data/outputs/stage1_feedback.json
Total answers processed: 5129
Sample (trimmed):
{
  "row_id": 58204,
  "source_dataset": "curated_0star",
  "question": "What is opportunity cost in economics for an introductory class?",
  "question_type": "explanation",
  "expected_roles": [
    "claim",
    "explanation",
    "evidence"
  ],
  "covered_roles": [
    "evidence",
    "explanation"
  ],
  "missing_roles": [
    "claim"
  ],
  "reference_answer": "Opportunity cost is the value of the next best alternative given up when a choice is made. In economics, it is used to understand trade-offs in everyday decisions. A complete answer would also distinguish it from explicit monetary cost.",
  "student_answer": "opportunity cost is basically a fixed government rule that sets the same price for everyone. It matters because it supposedly re

In [ ]:
# ── Post-processing: reasoning edge case export ───────────────────────────────
# Exports answers with: no valid path, low depth, missing evidence/claim,
# or detected contradiction to error_analysis/reasoning_edge_cases.csv

import pandas as pd, os

_edge_cases = []

for ans in (final_feedback if "final_feedback" in dir() else []):
    _sents = ans.get("sentences",[])
    _roles = {s.get("role") for s in _sents}
    _edges = ans.get("reasoning_graph",[])
    _depth = ans.get("depth_score",0) or 0
    _flags = []

    if not ans.get("has_valid_path",False):  _flags.append("no_valid_path")
    if _depth < 0.3:                          _flags.append("low_depth")
    if "evidence" not in _roles:              _flags.append("missing_evidence")
    if "claim"    not in _roles:              _flags.append("missing_claim")
    if any(e.get("relation")=="contradict" for e in _edges):
        _flags.append("contradiction_detected")

    if _flags:
        _edge_cases.append({
            "row_id":           ans.get("row_id",""),
            "question_type":    ans.get("question_type",""),
            "student_answer":   str(ans.get("student_answer",""))[:200],
            "depth_score":      _depth,
            "reasoning_level":  ans.get("reasoning_level","Weak"),
            "flags":            "; ".join(_flags),
            "remark":           ans.get("remark",""),
        })

if _edge_cases:
    try:
        os.makedirs(ERROR_DIR, exist_ok=True)
        pd.DataFrame(_edge_cases).to_csv(REASONING_EDGE_CSV, index=False)
        print(f"Reasoning edge cases saved: {REASONING_EDGE_CSV} ({len(_edge_cases)} cases)")
    except Exception as _e:
        print(f"Could not save reasoning_edge_cases.csv: {_e}")
else:
    print("No reasoning edge cases found.")


Reasoning edge cases saved: /content/drive/MyDrive/FYP_Data/outputs/error_analysis/reasoning_edge_cases.csv (4338 cases)


In [ ]:
# ── Section: Training accuracy tracking ──────────────────────────────────────
# Training accuracy history is captured by EpochCB in Section 8 (DistilBERT).
# Plots are generated in Section 9B.
# This cell is a placeholder — no action required here.
# Set FORCE_RETRAIN_FOR_CURVE = True in Section 8 to capture epoch-level logs.
print("Training diagnostics: see Section 9B output.")
print("To capture epoch-level accuracy curves: set FORCE_RETRAIN_FOR_CURVE=True in Section 8.")


Training diagnostics: see Section 9B output.
To capture epoch-level accuracy curves: set FORCE_RETRAIN_FOR_CURVE=True in Section 8.


## Section 16 — Pipeline Analysis Visualisations

**Purpose:** Visualise the reasoning pipeline outputs to understand system behaviour across the full answer set.

**Plots produced (6 panels):**

| Panel | What It Shows | Interpretation |
|---|---|---|
| A. Sentence Role Distribution | Count of each role type | Expected: `claim` dominant; `evidence` sparse in short answers |
| B. Reasoning Level | Weak / Moderate / Strong counts | Shows how many answers have meaningful reasoning structures |
| C. Predicted Stars | 0–5★ distribution | Confirms star range coverage; uniform distribution is ideal for curated rows |
| D. Graph Applicability | Multi-sentence % with valid NLI graphs | High % = richer reasoning analysis available |
| E. Edge Types | Support vs contradiction counts | High support:contradiction ratio = coherent answer set |
| F. Orthogonality Scatter | depth_score vs normalized_score | **Low Pearson r confirms depth ≠ correctness proxy** (supports RQ2) |

**Additional reports:** Failure taxonomy (8 types), weight ablation (5 configurations).

**All plots saved to `PLOTS_DIR`.**


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Section 16 — Pipeline Analysis Visualisations
# Every graph: title, axes, legend, interpretation text, PNG + TXT saved.
# ══════════════════════════════════════════════════════════════════════════════

from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt, matplotlib as mpl, os, json as _json
import numpy as np
from collections import Counter
mpl.rcParams.update({"font.size":11,"axes.titlesize":12,"figure.dpi":120})

os.makedirs(FIGURES_DIR, exist_ok=True)

def _safe_save(fig, name, interp_text=None):
    for _dir in [PLOTS_DIR, FIGURES_DIR]:
        try:
            fig.savefig(os.path.join(_dir, name), dpi=150, bbox_inches="tight")
        except Exception as _e:
            print(f"  Save failed ({_dir}/{name}): {_e}")
    if interp_text:
        try:
            _tp = os.path.join(FIGURES_DIR, name.replace(".png",".txt"))
            with open(_tp,"w") as f: f.write(interp_text)
        except Exception: pass
    plt.close(fig)
    print(f"  Saved: {name}")

if not final_feedback:
    print("final_feedback empty — run Section 15 first.")
else:
    all_roles   = Counter(s["role"] for ans in final_feedback for s in ans["sentences"])
    total_sents = max(sum(all_roles.values()), 1)
    level_dist  = Counter(a["reasoning_level"] for a in final_feedback)
    star_dist   = Counter(a.get("predicted_stars", a.get("stars", 0)) for a in final_feedback)

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("REXA — Pipeline Analysis Overview", fontsize=14, fontweight="bold")

    # 1. Role distribution
    ax = axes[0][0]
    _labels = list(all_roles.keys()); _vals = list(all_roles.values())
    _bars = ax.bar(_labels, _vals,
                   color=["#1D9E75","#378ADD","#EF9F27","#9B59B6","#E24B4A"])
    ax.set(title="Sentence Role Distribution",
           xlabel="Assigned Role", ylabel="Count")
    for b in _bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+1,
                f"{b.get_height()}", ha="center", va="bottom", fontsize=8)
    # Percentage labels
    for i,(l,v) in enumerate(zip(_labels,_vals)):
        ax.text(i, v/2, f"{v/total_sents*100:.1f}%",
                ha="center", va="center", fontsize=7.5, color="white", fontweight="bold")

    # 2. Reasoning level distribution
    ax = axes[0][1]
    _lv = ["Strong","Moderate","Weak","Zero"]
    _lc = [level_dist.get(l,0) for l in _lv]
    _bc = ax.bar(_lv, _lc, color=["#1D9E75","#EF9F27","#E24B4A","#888888"])
    ax.set(title="Reasoning Level Distribution",
           xlabel="Reasoning Level", ylabel="Count")
    for b in _bc:
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.3,
                str(int(b.get_height())), ha="center", fontsize=9)

    # 3. Predicted star distribution
    ax = axes[0][2]
    _sv = sorted(star_dist.keys()); _sc = [star_dist[s] for s in _sv]
    ax.bar([f"{s}★" for s in _sv], _sc, color="#378ADD", edgecolor="white")
    ax.set(title="Predicted Stars Distribution",
           xlabel="Predicted Stars (0–5)", ylabel="Count")

    # 4. Depth score distribution
    ax = axes[1][0]
    _depths = [a.get("depth_score",0) or 0 for a in final_feedback]
    ax.hist(_depths, bins=20, color="#9B59B6", edgecolor="white", alpha=0.8)
    ax.axvline(np.mean(_depths), color="red", linestyle="--",
               label=f"Mean={np.mean(_depths):.3f}")
    ax.set(title="Depth Score Distribution",
           xlabel="Depth Score (0.0–1.0)", ylabel="Frequency")
    ax.legend(fontsize=9)

    # 5. Graph quality distribution
    ax = axes[1][1]
    _gq = Counter(a.get("graph_quality","none") or "none" for a in final_feedback)
    _gq_labels = ["high","medium","low","none","not_applicable"]
    _gq_vals   = [_gq.get(l,0) for l in _gq_labels]
    ax.bar(_gq_labels, _gq_vals,
           color=["#1D9E75","#EF9F27","#E24B4A","#888888","#CCCCCC"])
    ax.set(title="NLI Graph Quality Distribution",
           xlabel="Graph Quality", ylabel="Count")

    # 6. Single vs multi sentence
    ax = axes[1][2]
    _ms = sum(1 for a in final_feedback if a.get("is_multi_sentence",False))
    _ss = len(final_feedback) - _ms
    ax.pie([_ms,_ss], labels=[f"Multi-sentence\n({_ms})",f"Single-sentence\n({_ss})"],
           autopct="%1.1f%%", colors=["#378ADD","#EF9F27"],
           startangle=140, explode=(0.03,0.03))
    ax.set_title("Answer Type Distribution")

    plt.tight_layout()
    _safe_save(fig, "pipeline_overview.png",
               "Role Distribution: shows how sentences are classified across the 5 roles. "
               "Evidence underrepresentation (below 10%) triggers augmentation. "
               "Reasoning Level: Strong=valid full path, Moderate=partial, Weak=no graph or single-sentence. "
               "Depth Scores: distribution centred near 0.5 is expected for mixed-quality answers.")

    # ── Depth vs Correctness orthogonality ───────────────────────────────────
    import pandas as pd
    try:
        _fs = pd.read_csv(FINAL_SUMMARY_CSV) if os.path.exists(FINAL_SUMMARY_CSV) else pd.DataFrame()
        if _fs.empty:
            print(f"WARNING: {(FINAL_SUMMARY_CSV)} not found or empty — skipping this section.") if _cached(FINAL_SUMMARY_CSV) else None
    except Exception: _fs = None

    if _fs is not None and "depth_score" in _fs.columns and "normalized_score" in _fs.columns:
        _dp = _fs[["depth_score","normalized_score"]].dropna()
        if len(_dp) > 5:
            _r,_p     = pearsonr(_dp["depth_score"],_dp["normalized_score"])
            _sr,_sp   = spearmanr(_dp["depth_score"],_dp["normalized_score"])
            fig2, ax2 = plt.subplots(figsize=(7,5))
            ax2.scatter(_dp["normalized_score"],_dp["depth_score"],
                        alpha=0.4, s=18, color="#378ADD")
            ax2.set(title=f"Depth Score vs Correctness Score\n"
                          f"Pearson r={_r:.3f}  Spearman ρ={_sr:.3f}  "
                          f"(low r supports RQ2 construct validity)",
                    xlabel="Normalized Correctness Score (DistilBERT)",
                    ylabel="Depth Score (Reasoning Structure)")
            ax2.grid(alpha=0.3)
            plt.tight_layout()
            _safe_save(fig2, "depth_vs_correctness_scatter.png",
                       f"Depth score vs correctness score scatter plot. "
                       f"Pearson r={_r:.3f} (low = depth is independent of correctness). "
                       f"This supports RQ2: reasoning depth is a structurally distinct "
                       f"measurement from answer correctness.")

    # ── Failure taxonomy ───────────────────────────────────────────────────────
    FAILURE_TAXONOMY = {
        "single_sentence_only":     "Answer has only one sentence — graph not applicable.",
        "no_claim":                 "No claim role detected.",
        "no_evidence":              "No evidence role detected.",
        "no_explanation":           "No explanation role detected.",
        "no_conclusion":            "No conclusion role detected.",
        "no_graph_edges":           "Multi-sentence answer but no NLI edges above threshold.",  # taxonomy identifier — not the data key ("reasoning_graph"); both intentionally co-exist
        "inverted_sequence":        "Conclusion appears before claim.",
        "contradictions_dominate":  "More contradiction edges than support edges.",
        "low_confidence_pred":      "Role confidence all low — prediction unreliable.",
        "nli_contradiction":        "At least one severe NLI contradiction detected.",
        "edge_star_failure":        "Prediction failed on 0★ or 5★ answer.",
        "claim_missing_in_multi":   "Multi-sentence answer lacks any claim sentence.",
    }

    _taxonomy_counts = {k:0 for k in FAILURE_TAXONOMY}
    _taxonomy_examples = {k:[] for k in FAILURE_TAXONOMY}

    for a in final_feedback:
        _sents = a.get("sentences",[])
        _roles_in = {s.get("role") for s in _sents}
        _edges = a.get("reasoning_graph",[])

        if not a.get("is_multi_sentence",False):
            _taxonomy_counts["single_sentence_only"] += 1
            _taxonomy_examples["single_sentence_only"].append(
                {"answer": str(a.get("student_answer",""))[:80], "depth": a.get("depth_score")})

        if "claim" not in _roles_in:
            _taxonomy_counts["no_claim"] += 1
            _taxonomy_examples["no_claim"].append(
                {"answer": str(a.get("student_answer",""))[:80]})

        if "evidence" not in _roles_in:
            _taxonomy_counts["no_evidence"] += 1

        if "explanation" not in _roles_in:
            _taxonomy_counts["no_explanation"] += 1

        if "conclusion" not in _roles_in:
            _taxonomy_counts["no_conclusion"] += 1

        if a.get("is_multi_sentence") and not _edges:
            _taxonomy_counts["no_graph_edges"] += 1
            _taxonomy_examples["no_graph_edges"].append(
                {"answer": str(a.get("student_answer",""))[:80],
                 "sent_count": a.get("sent_count",0)})

        if a.get("reasoning_sequence") == "inverted":
            _taxonomy_counts["inverted_sequence"] += 1

        _sup = sum(1 for e in _edges if e.get("relation")=="support")
        _con = sum(1 for e in _edges if e.get("relation")=="contradict")
        if _con > _sup:
            _taxonomy_counts["contradictions_dominate"] += 1

        if all(s.get("role_confidence")=="low" for s in _sents if _sents):
            _taxonomy_counts["low_confidence_pred"] += 1

        if any(e.get("contradiction_severity")=="severe" for e in _edges):
            _taxonomy_counts["nli_contradiction"] += 1

        if a.get("predicted_stars",3) in (0,5) and a.get("target_stars") is not None:
            if abs(a["predicted_stars"] - a["target_stars"]) >= 3:
                _taxonomy_counts["edge_star_failure"] += 1

        if a.get("is_multi_sentence") and "claim" not in _roles_in:
            _taxonomy_counts["claim_missing_in_multi"] += 1

    _total_ans = max(len(final_feedback), 1)
    _taxonomy_report = {}
    for k, cnt in _taxonomy_counts.items():
        pct = round(cnt/_total_ans*100, 2)
        _taxonomy_report[k] = {
            "description": FAILURE_TAXONOMY[k],
            "count": cnt,
            "percentage": pct,
            "examples": _taxonomy_examples.get(k,[])[:3],
            "recommended_fix": (
                "Add sentence length requirement" if k=="single_sentence_only" else
                "Check CLAIM_CUES expansion or role fallback logic" if k=="no_claim" else
                "Run evidence augmentation" if k=="no_evidence" else
                "Expand EXPLANATION_CUES or lower SBERT threshold" if k=="no_explanation" else
                "Add conclusion cue or position fallback at last sentence" if k=="no_conclusion" else
                f"Lower EDGE_THRESHOLD below {EDGE_THRESHOLD}" if k=="no_graph_edges" else
                "Improve sequence_score penalty in depth formula" if k=="inverted_sequence" else
                "Review answer quality — contradictions may be genuine" if k=="contradictions_dominate" else
                "Use BART confidence threshold; re-check cue rules" if k=="low_confidence_pred" else
                "Flag for human review" if k=="nli_contradiction" else
                "Apply edge-star oversampling (EDGE_STAR_WEIGHT)" if k=="edge_star_failure" else
                "Ensure first sentence classified as claim" if k=="claim_missing_in_multi" else
                "Investigate manually"
            ),
        }

    # Save taxonomy
    try:
        with open(FAILURE_TAXONOMY_JSON,"w") as f:
            _json.dump(_taxonomy_report, f, indent=2)
        print(f"Failure taxonomy saved: {FAILURE_TAXONOMY_JSON}")
    except Exception as _e: print(f"Taxonomy JSON error: {_e}")

    try:
        import pandas as pd
        _rows = []
        for k,v in _taxonomy_report.items():
            for ex in v["examples"]:
                _rows.append({"failure_type":k, "count":v["count"],
                              "percentage":v["percentage"], **ex})
        if _rows:
            pd.DataFrame(_rows).to_csv(FAILURE_TAXONOMY_CSV, index=False)
            print(f"Taxonomy examples saved: {FAILURE_TAXONOMY_CSV}")
    except Exception as _e: print(f"Taxonomy CSV error: {_e}")

    # Taxonomy bar chart
    _tk = [k for k,v in _taxonomy_counts.items() if v > 0]
    _tv = [_taxonomy_counts[k] for k in _tk]
    if _tk:
        fig3, ax3 = plt.subplots(figsize=(12,5))
        _bc = ax3.barh([k.replace("_"," ") for k in _tk], _tv,
                       color="#E24B4A", edgecolor="white")
        ax3.set(title="Failure Taxonomy Distribution\n(Count of answers per failure type)",
                xlabel="Count", ylabel="Failure Type")
        for b in _bc:
            ax3.text(b.get_width()+0.1, b.get_y()+b.get_height()/2,
                     str(int(b.get_width())), va="center", fontsize=8)
        plt.tight_layout()
        _safe_save(fig3, "failure_taxonomy.png",
                   "Failure Taxonomy: counts how many answers fall into each identified "
                   "failure mode. 'no_evidence' and 'single_sentence_only' are expected "
                   "to be highest for short-answer datasets. These are system self-diagnosis "
                   "indicators, not project failures.")

    print("\nSection 16 complete.")


  Saved: pipeline_overview.png
Failure taxonomy saved: /content/drive/MyDrive/FYP_Data/outputs/error_analysis/failure_taxonomy_report.json
Taxonomy examples saved: /content/drive/MyDrive/FYP_Data/outputs/error_analysis/failure_taxonomy_examples.csv
  Saved: failure_taxonomy.png

Section 16 complete.


## Section 17 — Final Summary CSV

**Purpose:** Save the complete per-answer analysis results to a single structured CSV for downstream use and evaluation.

**Output file:** `csv/final_feedback_summary.csv`

**Key columns:**

| Column | Meaning |
|---|---|
| `target_stars` | Ground-truth reasoning depth (curated rows only; NaN for others) |
| `predicted_stars` | REXA system prediction from `depth_score` |
| `depth_score` | Continuous 0–1 reasoning depth score |
| `structure_score` | Role + sequence + requirement contribution |
| `reasoning_score` | NLI graph contribution |
| `reasoning_level` | Weak / Moderate / Strong |
| `depth_confidence` | System confidence in the depth score |
| `confidence_flags` | Flags for low coverage, no graph, etc. |
| `remark` | Actionable feedback text |

**Expected output:** CSV saved; column list and row count printed.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Section 17 — Final Summary CSV
# FIX: all required columns included; safe saves with try/except.
# ══════════════════════════════════════════════════════════════════════════════

def _compute_confidence_flags(ans):
    flags = []
    if ans.get("role_coverage_score",0) < 0.40:     flags.append("low_role_coverage")
    if ans.get("sequence_score",1) < 0.5:            flags.append("poor_sequence")
    if ans.get("reasoning_graph_count",0)==0 and ans.get("is_multi_sentence",False):
        flags.append("no_graph_edges")  # failure-taxonomy flag name — not a data dict key
    if ans.get("contradiction_edges",0) > ans.get("support_edges",0):
        flags.append("contradictions_dominate")
    if len(str(ans.get("student_answer","")).split()) < 8:
        flags.append("very_short_answer")
    return flags

def _high_conf_edge_ratio(ans):
    edges = ans.get("reasoning_graph",[])
    if not edges: return None
    high = sum(1 for e in edges if abs(e.get("support_score",0)) >= 0.50)
    return round(high/len(edges),4)

def _graph_quality_from_ans(ans):
    if not ans.get("graph_applicable",False): return "not_applicable"
    n_e = ans.get("reasoning_graph_count",0)
    sup = ans.get("support_edges",0)
    con = ans.get("contradiction_edges",0)
    if n_e==0:    return "no_edges"
    if con > sup: return "contradictory"
    if sup >= 3:  return "strong"
    if sup >= 1:  return "partial"
    return "weak"

def _depth_confidence(flags, depth):
    if not flags: return "high" if depth >= 0.6 else "medium"
    if len(flags) >= 3: return "low"
    return "medium"

# _stars_from_depth removed — canonical depth_to_stars() in Cell 4 is identical

summary_rows = []
for ans in final_feedback:
    flags      = _compute_confidence_flags(ans)
    hcer       = _high_conf_edge_ratio(ans)
    gq         = _graph_quality_from_ans(ans)
    depth      = float(ans.get("depth_score",0.0) or 0.0)
    dc         = _depth_confidence(flags, depth)
    pred_stars = depth_to_stars(depth)

    _rid = ans.get("row_id")

    # target_stars from feedback_sample (curated_demo only)
    target_stars = None
    try:
        _ts = feedback_sample[feedback_sample["row_id"]==_rid]["target_stars"]
        if len(_ts) and pd.notna(_ts.iloc[0]):
            target_stars = int(_ts.iloc[0])
    except Exception:
        pass

    # dataset_purpose
    dp = ans.get("dataset_purpose","unknown")
    try:
        _dp = feedback_sample[feedback_sample["row_id"]==_rid]["dataset_purpose"]
        if len(_dp): dp = str(_dp.iloc[0])
    except Exception:
        pass

    summary_rows.append({
        # Identifiers
        "row_id":               _rid,
        "source_dataset":       ans.get("source_dataset","unknown"),
        "dataset_purpose":      dp,
        # Question
        "question":             ans.get("question",""),
        "question_type":        ans.get("question_type","general"),
        "student_answer":       ans.get("student_answer",""),
        # Stars
        "target_stars":         target_stars,
        "predicted_stars":      pred_stars,
        "stars":                ans.get("stars", pred_stars),
        # Sentence structure
        "sent_count":           ans.get("sent_count",1),
        "is_multi_sentence":    ans.get("is_multi_sentence",False),
        # Role information
        "expected_roles":       str(ans.get("expected_roles",[])),
        "covered_roles":        str(ans.get("covered_roles",[])),
        "missing_roles":        str(ans.get("missing_roles",[])),
        # Scores
        "role_coverage_score":  ans.get("role_coverage_score",0.0),
        "sequence_score":       ans.get("sequence_score",0.0),
        "requirement_score":    ans.get("requirement_score",0.0),
        "structure_score":      ans.get("structure_score",0.0),
        "reasoning_score":      ans.get("reasoning_score"),
        "depth_score":          depth,
        # Output labels
        "reasoning_level":      ans.get("reasoning_level","Weak"),
        "depth_confidence":     dc,
        "confidence_flags":     str(flags),
        # Graph
        "high_conf_edge_ratio": hcer,
        "graph_applicable":     ans.get("graph_applicable",False),
        "reasoning_graph_count":    ans.get("reasoning_graph_count",0),
        "support_edges":        ans.get("support_edges",0),
        "contradiction_edges":  ans.get("contradiction_edges",0),
        "has_valid_path":       ans.get("has_valid_path",False),
        "graph_quality":        gq,
        # Feedback
        "remark":               ans.get("remark",""),
    })

summary_df = pd.DataFrame(summary_rows)

# Safe save
try:
    summary_df.to_csv(FINAL_SUMMARY_CSV, index=False)
    print(f"Final summary CSV saved: {FINAL_SUMMARY_CSV}")
    print(f"Rows: {len(summary_df)}  Columns: {len(summary_df.columns)}")
except Exception as _e:
    print(f"Could not save final_feedback_summary.csv: {_e}")

print(f"\nColumns ({len(summary_df.columns)}):")
print(list(summary_df.columns))
print("\nSample (key columns):")
_preview = ["row_id","dataset_purpose","question_type","target_stars",
            "predicted_stars","depth_score","reasoning_level","graph_quality","remark"]
print(summary_df[[c for c in _preview if c in summary_df.columns]].head(6).to_string(index=False))


Final summary CSV saved: /content/drive/MyDrive/FYP_Data/outputs/final_feedback_summary.csv
Rows: 5129  Columns: 31

Columns (31):
['row_id', 'source_dataset', 'dataset_purpose', 'question', 'question_type', 'student_answer', 'target_stars', 'predicted_stars', 'stars', 'sent_count', 'is_multi_sentence', 'expected_roles', 'covered_roles', 'missing_roles', 'role_coverage_score', 'sequence_score', 'requirement_score', 'structure_score', 'reasoning_score', 'depth_score', 'reasoning_level', 'depth_confidence', 'confidence_flags', 'high_conf_edge_ratio', 'graph_applicable', 'reasoning_graph_count', 'support_edges', 'contradiction_edges', 'has_valid_path', 'graph_quality', 'remark']

Sample (key columns):
 row_id dataset_purpose question_type  target_stars  predicted_stars  depth_score reasoning_level graph_quality                                                                                                                                                                                     

## Section 17B — Star-Level Evaluation

**Purpose:** Evaluate how well `predicted_stars` aligns with `target_stars` on curated_demo rows.

**What this cell does:**
- Filters to rows with both `target_stars` and `predicted_stars`
- Uses percentile-anchored thresholds when depth_score distribution is compressed
- Computes: Accuracy, MAE, Macro F1, Weighted F1, Cohen's κ

**Plots produced:**
1. **Confusion matrix** — rows = target stars, columns = predicted stars
2. **Scatter plot** — predicted vs target with jitter
3. **Distribution comparison** — target vs predicted counts per star level

**Interpreting results:**

| Metric | Acceptable Range | Meaning |
|---|---|---|
| MAE | < 1.0 | Predictions within one star of ground truth on average |
| Cohen's κ | ≥ 0.40 | Fair-to-moderate agreement with curated labels |
| Accuracy | > 0.50 | Better than random (6-class: random = 0.17) |

**All plots saved to `PLOTS_DIR`.**

**Note:** This evaluation is only possible for `curated_demo` rows. ASAG rows have no `target_stars` and are excluded.


In [ ]:
# ── Section 17B — Star-Level Evaluation ─────────────────────────────────────
# FIX: improved predicted_stars calibration on curated_demo rows.
# Uses percentile-anchored thresholds instead of fixed absolute thresholds
# when depth_score distribution is compressed or skewed.
# Absolute thresholds are preserved if distribution is well-spread.
# ══════════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import (accuracy_score, mean_absolute_error,
                              f1_score, cohen_kappa_score, confusion_matrix)

if "predicted_stars" not in summary_df.columns:
    summary_df["predicted_stars"] = summary_df.get("stars", 0)
    print("NOTE: predicted_stars derived from stars column.")

# ── Filter to curated_demo rows with known target_stars ──────────────────────
if "dataset_purpose" in summary_df.columns:
    eval_df = summary_df[
        summary_df["target_stars"].notna() &
        (summary_df["dataset_purpose"] == "curated_demo")
    ].copy()
else:
    eval_df = summary_df[summary_df["target_stars"].notna()].copy()

if eval_df.empty:
    # Fallback: any row with target_stars
    eval_df = summary_df[summary_df["target_stars"].notna()].copy()

print(f"Evaluation rows: {len(eval_df)}")

if len(eval_df) < 2:
    print(f"Insufficient rows for star evaluation ({len(eval_df)}).")
    print("Load star-rated CSVs and set FORCE_REBUILD_CURATED=True.")
else:
    eval_df["target_stars"]    = eval_df["target_stars"].astype(int).clip(0,5)
    eval_df["predicted_stars"] = eval_df["predicted_stars"].astype(int).clip(0,5)

    # ── Calibration check ─────────────────────────────────────────────────────
    # If predicted_stars distribution has ≤2 unique values or is heavily
    # concentrated at one value, apply percentile-based recalibration.
    _pred_unique = eval_df["predicted_stars"].nunique()
    _depths = eval_df["depth_score"].dropna().values

    def _calibrate_stars(depth_series):
        """
        Recalibrate predicted_stars using percentile anchors on depth_score.
        Percentile boundaries: 0–16%=0★, 17–33%=1★, 34–50%=2★, 51–66%=3★, 67–83%=4★, 84–100%=5★.
        Applied ONLY when absolute-threshold prediction collapses to ≤2 distinct values.
        """
        if len(depth_series) < 6:
            return None
        p = np.percentile(depth_series, [16.7, 33.3, 50.0, 66.7, 83.3])
        def _map(d):
            if   d >= p[4]: return 5
            elif d >= p[3]: return 4
            elif d >= p[2]: return 3
            elif d >= p[1]: return 2
            elif d >= p[0]: return 1
            else:           return 0
        return depth_series.apply(_map)

    _calibrated = False
    if _pred_unique <= 2 and len(_depths) >= 6:
        print(f"NOTE: predicted_stars has only {_pred_unique} unique value(s). "
              "Applying percentile-based recalibration for evaluation.")
        _recal = _calibrate_stars(eval_df["depth_score"].fillna(0))
        if _recal is not None:
            eval_df["predicted_stars_calibrated"] = _recal.clip(0,5).astype(int)
            _calibrated = True
            print("Calibrated predicted_stars distribution:")
            print(eval_df["predicted_stars_calibrated"].value_counts().sort_index().to_string())
    else:
        eval_df["predicted_stars_calibrated"] = eval_df["predicted_stars"]

    y_true = eval_df["target_stars"].values
    y_pred = eval_df["predicted_stars_calibrated"].values
    labels = list(range(6))

    acc  = accuracy_score(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    f1_m = f1_score(y_true, y_pred, average="macro",   labels=labels, zero_division=0)
    f1_w = f1_score(y_true, y_pred, average="weighted",labels=labels, zero_division=0)
    kap  = cohen_kappa_score(y_true, y_pred)

    print("\n" + "="*62)
    print("  STAR-LEVEL EVALUATION" + (" (calibrated)" if _calibrated else ""))
    print("="*62)
    print(f"  N evaluated    : {len(eval_df)}")
    print(f"  Accuracy       : {acc:.4f}  ({acc*100:.1f}%)")
    print(f"  MAE            : {mae:.4f}")
    print(f"  Macro F1       : {f1_m:.4f}")
    print(f"  Weighted F1    : {f1_w:.4f}")
    print(f"  Cohen's κ      : {kap:.4f}")
    if   kap >= 0.80: print("  κ: Substantial to near-perfect")
    elif kap >= 0.60: print("  κ: Moderate to substantial ✓")
    elif kap >= 0.40: print("  κ: Fair agreement")
    else:             print("  κ: Weak — reported as limitation")

    print("\n  Per-star accuracy:")
    print(f"  {'Star':<8} {'N':>5}  {'Correct':>8}  {'Acc':>7}")
    for st in range(6):
        _sub = eval_df[eval_df["target_stars"]==st]
        if len(_sub)==0: continue
        _cor = (_sub["target_stars"]==_sub["predicted_stars_calibrated"]).sum()
        print(f"  {st}★       {len(_sub):>5,}  {_cor:>8,}  {_cor/len(_sub):>6.1%}")

    # Safe save
    try:
        eval_df[["row_id","source_dataset","dataset_purpose",
                 "target_stars","predicted_stars","predicted_stars_calibrated",
                 "depth_score","reasoning_level","remark"]].to_csv(STAR_EVAL_CSV, index=False)
        print(f"\n  Saved: {STAR_EVAL_CSV}")
    except Exception as _e:
        print(f"  Could not save star_level_evaluation.csv: {_e}")

    # ── Plots ─────────────────────────────────────────────────────────────────
    os.makedirs(PLOTS_DIR, exist_ok=True)

    try:
        cm = confusion_matrix(y_true, y_pred, labels=labels)
        fig, ax = plt.subplots(figsize=(8,6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=[f"{s}★" for s in labels],
                    yticklabels=[f"{s}★" for s in labels], ax=ax)
        ax.set(title="Star-Level Confusion Matrix\n(Rows=Target, Cols=Predicted)",
               xlabel="Predicted Stars", ylabel="Target Stars")
        plt.tight_layout()
        _p = os.path.join(PLOTS_DIR,"star_confusion_matrix.png")
        plt.savefig(_p, dpi=120, bbox_inches="tight")
        plt.show(); plt.close()
        print(f"  Saved: {_p}")
    except Exception as _e: print(f"  Confusion matrix plot error: {_e}")

    try:
        _x = np.arange(6); _w = 0.35
        _tc = [sum(y_true==s) for s in range(6)]
        _pc = [sum(y_pred==s) for s in range(6)]
        fig, ax = plt.subplots(figsize=(9,4))
        ax.bar(_x-_w/2, _tc, _w, label="Target Stars",    color="#1D9E75", edgecolor="white")
        ax.bar(_x+_w/2, _pc, _w, label="Predicted Stars", color="#378ADD", edgecolor="white")
        ax.set(title="Target vs Predicted Star Distribution",
               xlabel="Stars", ylabel="Count", xticks=_x,
               xticklabels=[f"{s}★" for s in range(6)])
        ax.legend(); plt.tight_layout()
        _p = os.path.join(PLOTS_DIR,"predicted_vs_target_stars.png")
        plt.savefig(_p, dpi=120, bbox_inches="tight")
        plt.show(); plt.close()
        print(f"  Saved: {_p}")
    except Exception as _e: print(f"  Distribution plot error: {_e}")


Evaluation rows: 4989

  STAR-LEVEL EVALUATION
  N evaluated    : 4989
  Accuracy       : 0.1864  (18.6%)
  MAE            : 1.3610
  Macro F1       : 0.1580
  Weighted F1    : 0.1531
  Cohen's κ      : 0.0426
  κ: Weak — reported as limitation

  Per-star accuracy:
  Star         N   Correct      Acc
  0★         579        27    4.7%
  1★       1,050        85    8.1%
  2★         660       233   35.3%
  3★         600       287   47.8%
  4★       1,050       293   27.9%
  5★       1,050         5    0.5%

  Saved: /content/drive/MyDrive/FYP_Data/outputs/star_level_evaluation.csv
  Saved: /content/drive/MyDrive/FYP_Data/outputs/plots/star_confusion_matrix.png
  Saved: /content/drive/MyDrive/FYP_Data/outputs/plots/predicted_vs_target_stars.png


---

## ⚗️ COMPARATIVE EXPERIMENT 3 — BART Zero-shot Classification (Optional)

**Purpose:** Evaluate zero-shot role classification as an alternative semantic approach to SBERT-based role detection.

> ❌ This experiment is **OPTIONAL** and **NOT** part of the core REXA architecture.
> The core system uses SBERT semantic similarity. BART zero-shot is a comparative experiment only.

**What this experiment adds:**
- Provides an alternative role label source (BART zero-shot vs SBERT similarity)
- Enables comparison of zero-shot LLM classification against rule-based semantic matching
- Optional LLM remark refinement via OpenRouter API (can be skipped entirely)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Section 18 — Stage 2 API Configuration
# Provider: OpenRouter ONLY.
# Key loaded from Colab Secrets or environment variable.
# NEVER hardcode the key here. NEVER print it.
# ══════════════════════════════════════════════════════════════════════════════

import os
stage2_data        = []   # always initialise
OPENROUTER_API_KEY = None
STAGE2_PROVIDER    = None

def _load_secret(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v and len(str(v).strip()) > 10:
            return str(v).strip()
    except Exception:
        pass
    return None

def _load_env(name):
    v = os.environ.get(name,"")
    return v.strip() if len(v.strip()) > 10 else None

# Load key from Colab Secrets (preferred) or environment variable
OPENROUTER_API_KEY = _load_secret("Rexa") or _load_env("OPENROUTER_API_KEY")

# ── Manual slot: paste key via Colab Secrets sidebar (key icon), not here ─────
# If you must override: uncomment next line and use Colab Secrets
# OPENROUTER_API_KEY = _load_secret("OPENROUTER_API_KEY")  # already done above

def _valid(k):
    return k is not None and isinstance(k,str) and len(k.strip()) > 10

if _valid(OPENROUTER_API_KEY):
    STAGE2_PROVIDER = "openrouter"
    os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
    _masked = OPENROUTER_API_KEY[:8] + "..." + OPENROUTER_API_KEY[-4:]
    print(f"Stage 2 provider : OpenRouter")
    print(f"Key loaded       : {_masked}  (masked for security)")
else:
    print("No valid OPENROUTER_API_KEY found.")
    print("Stage 2 will be skipped. Stage 1 results are complete and independent.")
    print("To enable Stage 2:")
    print("  1. Click the key icon in the Colab sidebar")
    print("  2. Add secret name: OPENROUTER_API_KEY")
    print("  3. Paste your key as the value")
    print("  4. Re-run this cell")


Stage 2 provider : OpenRouter
Key loaded       : sk-or-v1...98e6  (masked for security)


## Section 19 — Stage 2: Optional LLM Remark Refinement

**Purpose:** Use an LLM (via OpenRouter) to produce more natural-sounding feedback remarks.

**What this cell does:**
- Sends each answer's role coverage, depth score, star rating, and Stage 1 remark to the LLM
- Receives a refined 1–2 sentence remark (JSON format)
- Stores refined remarks as `remark_stage2`
- Saves `json/stage2_nli_reasoning_feedback.json`

**Model used:** `openai/gpt-oss-120b:free` (free-tier via OpenRouter — no credits required)

**Constraint:** LLM only rewrites remark text. All numeric scores are unchanged.

**Expected output:** Progress log with per-answer confirmation. Skipped gracefully if no API key.

> **Error handling:** Each LLM call is wrapped in `try/except`. Failed calls retain the Stage 1 remark.


In [ ]:
from google.colab import userdata
from openai import OpenAI
import time

ZAI_KEY = userdata.get("ZAI_API_KEY")

client = OpenAI(
    api_key=ZAI_KEY,
    base_url="https://api.z.ai/api/coding/paas/v4"
)

FREE_MODEL_NAME = "glm-4.7-flash"

# ══════════════════════════════════════════════════════════════════════════════
# Section 19 — Stage 2: GLM-4.7-Flash Remark Refinement (Optional)
# Provider: Z.ai Coding Plan
# FIX: added retry-with-backoff for 429/1305 rate-limit errors, and a fixed
#      delay between calls so 5,129 sequential requests don't overwhelm the
#      free-tier rate limit. Without this, nearly every call fails.
# ══════════════════════════════════════════════════════════════════════════════

import json as _json

stage2_data = []

S2_SYSTEM = """You are an educational feedback assistant for REXA.
You receive a student answer analysis (roles, depth, graph info).

Return ONLY a JSON object:

{"remark":"your improved remark here"}

Rules:
- 1-2 sentences maximum.
- Mention missing reasoning roles if any.
- Mention whether the reasoning chain is complete or broken.
- Never modify any score.
- Improve only the feedback.
- Return JSON only.
""".strip()


def _safe_json(text):
    try:
        return _json.loads(text)
    except Exception:
        try:
            import re
            m = re.search(r"\{.*?\}", text, re.DOTALL)
            if m:
                return _json.loads(m.group())
        except Exception:
            pass
    return None


def _build_prompt(ans):
    return (
        f"Question type   : {ans.get('question_type','general')}\n"
        f"Expected roles  : {ans.get('expected_roles',[])}\n"
        f"Covered roles   : {ans.get('covered_roles',[])}\n"
        f"Missing roles   : {ans.get('missing_roles',[])}\n"
        f"Depth score     : {ans.get('depth_score',0)}\n"
        f"Reasoning level : {ans.get('reasoning_level','Weak')}\n"
        f"Valid path      : {ans.get('has_valid_path',False)}\n"
        f"Support edges   : {ans.get('support_edges',0)}\n"
        f"Contradiction   : {ans.get('contradiction_edges',0)}\n"
        f"Current remark  : {ans.get('remark','')}\n"
        f"Answer snippet  : {str(ans.get('student_answer',''))[:200]}"
    )


# FIX: retry wrapper with exponential backoff for 429 / code 1305 errors
def _call_glm(prompt, max_retries=5, base_delay=2.0):
    """Call GLM-4.7-Flash via Z.ai Coding Plan endpoint, with retry on rate limits."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=FREE_MODEL_NAME,
                messages=[
                    {"role": "system", "content": S2_SYSTEM},
                    {"role": "user",   "content": prompt}
                ],
                temperature=0,
                max_tokens=200
            )
            return response.choices[0].message.content.strip()

        except Exception as e:
            err_str = str(e)
            is_rate_limit = ("429" in err_str) or ("1305" in err_str) or ("overloaded" in err_str.lower())
            if is_rate_limit and attempt < max_retries - 1:
                wait = base_delay * (2 ** attempt)   # 2s, 4s, 8s, 16s, 32s
                print(f"  Rate limited (attempt {attempt+1}/{max_retries}), waiting {wait:.0f}s...")
                time.sleep(wait)
                continue
            print(f"GLM Error: {e}")
            return None
    return None


if not ZAI_KEY:

    print("Stage 2 SKIPPED — no ZAI_API_KEY configured.")
    print("stage2_data = []")

    stage2_data = []

else:

    print(f"Running Stage 2 (GLM-4.7-Flash) on {len(final_feedback)} answers...")
    print("NOTE: throttled to avoid rate limits — this will take a while for 5,000+ answers.")

    _ok = 0

    for _i, ans in enumerate(final_feedback):

        raw = _call_glm(_build_prompt(ans))

        out = _safe_json(raw) if raw else None

        refined = (
            out.get("remark", ans.get("remark", ""))
            if out else
            ans.get("remark", "")
        )

        item = dict(ans)
        item["remark_stage2"] = refined

        if refined != ans.get("remark", ""):
            _ok += 1

        stage2_data.append(item)

        # FIX: small fixed delay between successful calls to stay under rate limit
        time.sleep(0.5)

        # Progress checkpoint every 200 answers, with periodic save
        if (_i + 1) % 200 == 0:
            print(f"  Progress: {_i+1}/{len(final_feedback)}  ({_ok} refined so far)")
            try:
                with open(STAGE2_JSON, "w") as f:
                    _json.dump(stage2_data, f, indent=2)
            except Exception:
                pass

    try:

        with open(STAGE2_JSON, "w") as f:
            _json.dump(stage2_data, f, indent=2)

        print(f"Stage 2 saved: {STAGE2_JSON}")

    except Exception as e:

        print(f"Could not save Stage 2 JSON: {e}")

    print(f"Stage 2: {_ok}/{len(stage2_data)} remarks refined.")

print(f"stage2_data ready: {len(stage2_data)} entries")

logger.info(
    f"Stage 2 complete. Provider=GLM-4.7-Flash entries={len(stage2_data)}"
)

Running Stage 2 (GLM-4.7-Flash) on 5129 answers...
NOTE: throttled to avoid rate limits — this will take a while for 5,000+ answers.
  Rate limited (attempt 1/5), waiting 2s...
  Rate limited (attempt 1/5), waiting 2s...
  Rate limited (attempt 1/5), waiting 2s...
  Rate limited (attempt 1/5), waiting 2s...
  Rate limited (attempt 1/5), waiting 2s...
  Rate limited (attempt 2/5), waiting 4s...
  Progress: 200/5129  (0 refined so far)


---

## 📊 FINAL COMPARISON — Core System vs Comparative Experiments

> This section compares REXA (proposed system) against the three comparative experiments.

Metrics reported: MAE, RMSE, Pearson r, role accuracy (Cohen's κ), star accuracy, reasoning depth coverage.

---

## Section 20 — Stage 1 vs Stage 2 Role Distribution


In [ ]:
s1_roles = Counter(s["role"] for ans in final_feedback for s in ans["sentences"])
s2_roles = Counter(s.get("role","irrelevant") for ans in stage2_data
                   for s in ans.get("sentences",[]))

if s2_roles:
    all_keys = sorted(set(list(s1_roles.keys())+list(s2_roles.keys())))
    s1_v = [s1_roles.get(r,0) for r in all_keys]
    s2_v = [s2_roles.get(r,0) for r in all_keys]
    x = range(len(all_keys)); w = 0.35
    fig, ax = plt.subplots(figsize=(10,5))
    ax.bar([i-w/2 for i in x], s1_v, w, label="Stage 1 (BART+SBERT)", color="steelblue")
    ax.bar([i+w/2 for i in x], s2_v, w, label="Stage 2 (LLM refined)", color="coral")
    ax.set_xticks(list(x)); ax.set_xticklabels(all_keys, rotation=30)
    ax.set(title="Role Distribution: Stage 1 vs Stage 2", xlabel="Role", ylabel="Count")
    ax.legend(); plt.tight_layout(); plt.show()
else:
    print("Stage 2 not available — comparison chart skipped.")


## Section 21 — Human Validation Template

> **Human annotation has not yet been conducted. The CSV template below is provided for future annotator-based evaluation. No labels have been fabricated or pre-filled.**

**Purpose:** Create a blank CSV template for independent human annotation.

**Template columns for annotators:**
- `human_rater_1_stars` — integer 0–5 (reasoning depth)
- `human_rater_2_stars` — integer 0–5
- `human_reasoning_level_1` — Weak / Moderate / Strong / Zero
- `human_reasoning_level_2` — Weak / Moderate / Strong / Zero
- `human_comment` — free text notes
- `human_agrees_system` — Yes / No / Partial

**Sampling:** Up to 10 rows per star level from curated_demo, stratified by `predicted_stars`.

**Output:** `csv/human_validation_template.csv`

**Next steps:**
1. Share the CSV with two independent annotators
2. Each fills only their own columns
3. Run Section 21B after both columns are complete to compute inter-rater reliability (IRR)


In [ ]:
# ── Section 21: Human Validation Template ────────────────────────────────────
# Human annotation has not yet been conducted.
# This cell creates a blank template for future annotators.
# Human rater columns are LEFT BLANK intentionally.
# DO NOT fill them programmatically — real human annotation required.
# FIX: uses HUMAN_VALIDATION_CSV consistently (not HUMAN_EVAL_CSV).

import os, pandas as pd, numpy as np

os.makedirs(OUTPUT_DIR, exist_ok=True)

# HUMAN_VALIDATION_CSV is defined in Section 2
_hv_path = globals().get("HUMAN_VALIDATION_CSV",
                          os.path.join(OUTPUT_DIR,"human_validation_template.csv"))

# Build candidate pool
if "summary_df" not in dir() or summary_df is None or summary_df.empty:
    try:
        summary_df = pd.read_csv(FINAL_SUMMARY_CSV) if os.path.exists(FINAL_SUMMARY_CSV) else pd.DataFrame()
        if summary_df.empty:
            print(f"WARNING: {(FINAL_SUMMARY_CSV)} not found or empty — skipping this section.")
        else:
            print(f"Loaded summary_df from {FINAL_SUMMARY_CSV}")
    except Exception as _e:
        print(f"Could not load summary_df: {_e}")
        summary_df = pd.DataFrame()

if summary_df.empty:
    print("summary_df is empty — run Section 17 first.")
else:
    _pool = pd.DataFrame()
    if "dataset_purpose" in summary_df.columns:
        _pool = summary_df[summary_df["dataset_purpose"] == "curated_demo"].copy()
    if _pool.empty:
        print("  NOTE: No curated_demo rows — using is_multi_sentence rows as fallback.")
        _pool = (summary_df[summary_df.get("is_multi_sentence",pd.Series([False]*len(summary_df)))==True].copy()
                 if "is_multi_sentence" in summary_df.columns else summary_df.copy())

    if _pool.empty:
        print("  No rows available for human validation template.")
    else:
        for _col in ["predicted_stars","stars"]:
            if _col in _pool.columns:
                _pool["_hv_star"] = pd.to_numeric(_pool[_col], errors="coerce")
                break
        if "_hv_star" not in _pool.columns:
            _pool["_hv_star"] = 0

        _pool = _pool[_pool["_hv_star"].notna()].copy()
        _pool["_hv_star"] = _pool["_hv_star"].astype(int).clip(0,5)

        hv_rows = []
        for star in range(6):
            _sp = _pool[_pool["_hv_star"]==star]
            if _sp.empty:
                print(f"  NOTE: No rows for {star}★ — skipping")
                continue
            _samp = _sp.sample(n=min(10,len(_sp)), random_state=42)
            for _, r in _samp.iterrows():
                hv_rows.append({
                    "row_id":              r.get("row_id",""),
                    "source_dataset":      r.get("source_dataset",""),
                    "dataset_purpose":     r.get("dataset_purpose",""),
                    "question":            r.get("question",""),
                    "student_answer":      r.get("student_answer",""),
                    "target_stars":        r.get("target_stars",""),
                    "predicted_stars":     r.get("predicted_stars", r.get("stars","")),
                    "depth_score":         r.get("depth_score",""),
                    "reasoning_level":     r.get("reasoning_level",""),
                    "remark":              r.get("remark",""),
                    # ── Blank annotation columns (fill manually) ────────────────
                    "human_rater_1_stars":      "",
                    "human_rater_2_stars":      "",
                    "human_reasoning_level_1":  "",
                    "human_reasoning_level_2":  "",
                    "human_comment":            "",
                    "human_agrees_system":      "",
                })

        hv_df = pd.DataFrame(hv_rows)
        if not hv_df.empty:
            try:
                hv_df.to_csv(_hv_path, index=False)
                # Keep HUMAN_VALIDATION_CSV in sync
                HUMAN_VALIDATION_CSV = _hv_path
                print("="*62)
                print("  HUMAN VALIDATION TEMPLATE")
                print("="*62)
                print(f"  Saved : {_hv_path}")
                print(f"  Rows  : {len(hv_df)}")
                print(f"  Stars covered: {sorted(hv_df['predicted_stars'].dropna().unique().tolist())}")
                print("\n  Annotator columns (fill manually):")
                print("    human_rater_1_stars       — integer 0-5")
                print("    human_rater_2_stars       — integer 0-5")
                print("    human_reasoning_level_1   — Weak / Moderate / Strong / Zero")
                print("    human_reasoning_level_2   — Weak / Moderate / Strong / Zero")
                print("    human_comment             — free text")
                print("    human_agrees_system       — Yes / No / Partial")
                print("\n  Human columns are intentionally BLANK.")
                print("  Fill manually later. DO NOT generate fake labels.")
                print("="*62)
            except Exception as _e:
                print(f"Could not save human_validation_template.csv: {_e}")
        else:
            print("  hv_df is empty — no rows sampled.")


In [ ]:
# ── Optional Human Validation Results ────────────────────────────────────────
# Run this cell AFTER filling human_validation_template.csv with real labels.
# If columns are blank, exits gracefully with a notice.
# FIX: uses HUMAN_VALIDATION_CSV (not HUMAN_EVAL_CSV).

from sklearn.metrics import cohen_kappa_score, mean_absolute_error
import numpy as np, pandas as pd

_hv_path = globals().get("HUMAN_VALIDATION_CSV",
                          os.path.join(OUTPUT_DIR,"human_validation_template.csv"))

try:
    hv_filled = pd.read_csv(_hv_path) if os.path.exists(_hv_path) else pd.DataFrame()
    if hv_filled.empty:
        print(f"WARNING: {(_hv_path)} not found or empty — skipping this section.")
except Exception as _e:
    hv_filled = pd.DataFrame()
    print(f"Could not load human validation file: {_e}")

if hv_filled.empty:
    print("Human validation data not available yet.")
    print(f"Expected file: {_hv_path}")
else:
    _r1 = hv_filled.get("human_rater_1_stars", pd.Series(dtype=str)).replace("", np.nan).dropna()
    _r2 = hv_filled.get("human_rater_2_stars", pd.Series(dtype=str)).replace("", np.nan).dropna()

    if len(_r1)==0 or len(_r2)==0:
        print("Human validation data not available yet.")
        print(f"File found at: {_hv_path}")
        print("Fill human_rater_1_stars and human_rater_2_stars columns and re-run.")
    else:
        _common = hv_filled.dropna(subset=["human_rater_1_stars","human_rater_2_stars"])
        _common = _common[_common["human_rater_1_stars"].astype(str) != ""]
        _common = _common[_common["human_rater_2_stars"].astype(str) != ""]

        if len(_common) < 2:
            print("Human validation data not available yet (insufficient filled rows).")
        else:
            r1c  = _common["human_rater_1_stars"].astype(int).values
            r2c  = _common["human_rater_2_stars"].astype(int).values
            sysc = _common.get("predicted_stars", pd.Series([0]*len(_common))).astype(int).values

            hh_kap = cohen_kappa_score(r1c, r2c)
            sh_kap = cohen_kappa_score(r1c, sysc)
            sh_mae = mean_absolute_error(r1c, sysc)

            print("="*62)
            print("  HUMAN VALIDATION RESULTS")
            print("="*62)
            print(f"  Annotated rows    : {len(_common)}")
            print(f"  Human-Human κ     : {hh_kap:.4f}")
            print(f"  System-Human κ    : {sh_kap:.4f}")
            print(f"  System-Human MAE  : {sh_mae:.4f}")
            if hh_kap > 0 and sh_kap/hh_kap >= 0.80:
                print("  System achieves ≥ 80% of human-human agreement ✓")
            else:
                print("  System below 80% of human-human agreement — noted as limitation.")


## Section 22 — Reasoning Graph Demo Examples

**Purpose:** Visualise representative NLI reasoning graphs for six answer quality levels, making the pipeline behaviour transparent and interpretable.

**Demo types:**

| Demo | Description | Graph behaviour |
|---|---|---|
| 0★ | Incorrect / empty reasoning | No edges; isolated nodes |
| 1★ | Weak, minimal reasoning | Single low-confidence support edge |
| 3★ | Moderate reasoning | Partial role coverage; 2–3 edges |
| 5★ | Strong reasoning | Valid claim → explanation → conclusion path |
| Contradiction | Logically inconsistent answer | Red contradiction edges dominant |
| Single-sentence | One-sentence answer | Graph not applicable; structural only |

**Graph conventions:**
- Node colour = sentence role (see legend)
- Green edges = logical support (entailment)
- Red edges = contradiction
- Edge thickness ∝ confidence level
- Layout fixed with `seed=42` for reproducibility

**All plots saved to `PLOTS_DIR`. `graph_demo_examples.json` saved to `JSON_DIR`.**

**Strength:** Concrete examples make the pipeline transparent.
**Weakness:** Demo answers are hand-crafted to illustrate each case. Real answer distributions may look different.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Section — Graph Demo Examples
# Shows reasoning graphs for representative answer types:
#   0★ (wrong), 1★, 3★, 5★, contradiction-heavy, single-sentence
# Visualises with networkx. Saves graph_demo_examples.json + PNGs.
# ══════════════════════════════════════════════════════════════════════════════
import subprocess as _sp
_sp.run(["pip","install","-q","networkx"], check=True)
import networkx as nx

ROLE_COLORS = {
    "claim":       "#1D9E75",
    "explanation": "#378ADD",
    "evidence":    "#EF9F27",
    "conclusion":  "#9B59B6",
    "irrelevant":  "#9E9E9E",
}
EDGE_COLORS = {"support": "#1D9E75", "contradict": "#E24B4A"}

def _pick_example(target_star=None, prefer_contradiction=False, prefer_single=False):
    """
    For star-specific examples search combined_all_df directly.
    For contradiction/single queries search final_feedback first.
    """
    if target_star is None:  # only search final_feedback for non-star queries
     for ans in final_feedback:
        _rid = ans.get("row_id")
        ts_col = combined_all_df[combined_all_df["row_id"] == _rid]["target_stars"] \
                 if "row_id" in combined_all_df.columns else pd.Series(dtype=float)
        ts = float(ts_col.iloc[0]) if len(ts_col) and pd.notna(ts_col.iloc[0]) else None
        if prefer_single and not ans.get("is_multi_sentence", True):
            return ans, ts
        if prefer_contradiction and ans.get("contradiction_edges", 0) > 0:
            return ans, ts
        if target_star is not None and ts is not None and int(ts) == target_star:
            return ans, ts

    if target_star is not None and "combined_all_df" in dir():
        pool = combined_all_df[
            (combined_all_df["dataset_purpose"] == "curated_demo") &
            (combined_all_df["target_stars"] == target_star)
        ]
        if not pool.empty:
            row = pool.sample(1, random_state=42).iloc[0]
            sents = split_into_sentences(str(row["student_answer"]))
            sent_dicts = []
            for i, s_txt in enumerate(sents, 1):
                try:
                    role, sec, _, _ = classify_role(
                        s_txt, str(row.get("reference_answer", "")), i, len(sents))
                except Exception:
                    role, sec = "claim", None
                sent_dicts.append({
                    "sentence_id": i, "text": s_txt, "role": role,
                    "secondary_role": sec, "has_causal_cue": False,
                    "semantic_sim": 0.0, "semantic_status": "missing",
                    "covers": [], "requirement_coverage": []
                })
            expected_roles = ROLE_MAP.get("explanation", ["claim", "explanation"])
            try:
                res = compute_depth_score(
                    sent_dicts, expected_roles, 0.0,
                    str(row.get("reference_answer", "")), str(row.get("question", "")))
                depth, structure_score, rcs, seq, req, reasoning_score, has_valid_path, edges, _ = res
            except Exception:
                depth, structure_score, rcs, seq, req = 0.0, 0.0, 0.0, 0.0, 0.0
                reasoning_score, has_valid_path, edges = None, False, []
            stars = depth_to_stars(depth)  # canonical (Cell 4)
            return {
                "row_id": row.get("row_id", 0),
                "source_dataset": row.get("source_dataset", "curated"),
                "question": str(row.get("question", "")),
                "question_type": "explanation",
                "expected_roles": expected_roles,
                "covered_roles": list({s2["role"] for s2 in sent_dicts
                                       if s2["role"] not in ("irrelevant", None)}),
                "missing_roles": [],
                "reference_answer": str(row.get("reference_answer", "")),
                "student_answer": str(row.get("student_answer", "")),
                "normalized_score": None, "predicted_score": 0.0,
                "is_multi_sentence": len(sents) >= 2,
                "sent_count": len(sents), "graph_applicable": len(sents) >= 2,
                "role_coverage_score": rcs, "sequence_score": seq,
                "requirement_score": req, "structure_score": structure_score,
                "reasoning_score": reasoning_score, "depth_score": depth,
                "stars": stars,
                "reasoning_level": reasoning_level_from_depth(depth),  # canonical (Cell 4)
                "depth_confidence": "medium", "confidence_flags": [],
                "high_conf_edge_ratio": 0.0, "reasoning_graph": edges,
                "reasoning_graph_count": len(edges),
                "support_edges": sum(1 for e in edges if e["relation"] == "support"),
                "contradiction_edges": sum(1 for e in edges if e["relation"] == "contradict"),
                "has_valid_path": has_valid_path,
                "reasoning_sequence": check_reasoning_sequence(sent_dicts),
                "remark": generate_remark(sent_dicts, expected_roles, has_valid_path,
                                          edges, len(sents) >= 2, len(sents) <= 1),
                "sentences": sent_dicts,
            }, float(target_star)
    return None, None

def _draw_graph(ans, ts, title, save_path=None):
    """Draw NLI reasoning graph for one answer."""
    sents = ans.get("sentences", [])
    edges = ans.get("reasoning_graph", [])
    is_multi = ans.get("is_multi_sentence", False)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    # Left: sentence roles
    ax = axes[0]
    ax.axis("off")
    _lines = [
        f"Q-Type  : {ans.get('question_type','?')}",
        f"Target★ : {ts if ts is not None else 'N/A'}",
        f"Depth   : {ans.get('depth_score',0):.3f}",
        f"Stars   : {ans.get('stars',0)}★",
        f"Level   : {ans.get('reasoning_level','?')}",
        f"Graph?  : {'Yes' if ans.get('graph_applicable') else 'No (single-sent)'}",
        "",
        f"Q: {str(ans.get('question',''))[:70]}",
        "",
    ]
    for s in sents:
        _role = s.get("role","?")
        _sr   = f"+{s['secondary_role']}" if s.get("secondary_role") else ""
        _line = f"[S{s.get('sentence_id','')}]({_role}{_sr}) {str(s.get('text',''))[:60]}"
        _lines.append(_line)
    _lines.append("")
    _lines.append(f"Remark: {str(ans.get('remark',''))[:80]}")
    ax.text(0.02, 0.98, "\n".join(_lines), transform=ax.transAxes,
            va="top", ha="left", fontsize=8.5, family="monospace",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="#F9F9F9", alpha=0.9))
    ax.set_title("Answer Details", fontsize=10)

    # Right: graph
    ax2 = axes[1]
    if not is_multi or not edges:
        ax2.axis("off")
        msg = ("Single-sentence answer:\nreasoning graph not applicable."
               if not is_multi else "No graph edges detected.")
        ax2.text(0.5, 0.5, msg, ha="center", va="center",
                 transform=ax2.transAxes, fontsize=12, color="#666",
                 bbox=dict(boxstyle="round", facecolor="#EEE"))
        ax2.set_title("Reasoning Graph", fontsize=10)
    else:
        G = nx.DiGraph()
        for s in sents:
            sid = s.get("sentence_id")
            role = s.get("role","?")
            G.add_node(sid, role=role,
                       label=f"S{sid}\n({role})\n{str(s.get('text',''))[:25]}...")
        for e in edges:
            G.add_edge(e["src"], e["tgt"],
                       relation=e["relation"],
                       weight=abs(e.get("support_score",0)))
        pos = nx.spring_layout(G, seed=42, k=2.5)
        node_colors = [ROLE_COLORS.get(G.nodes[n].get("role","irrelevant"),"#9E9E9E")
                       for n in G.nodes()]
        edge_colors = [EDGE_COLORS.get(G.edges[e].get("relation","support"),"#888")
                       for e in G.edges()]
        labels_dict = {n: G.nodes[n].get("label","") for n in G.nodes()}
        nx.draw_networkx(G, pos, ax=ax2, labels=labels_dict,
                         node_color=node_colors, edge_color=edge_colors,
                         node_size=1800, font_size=7, arrows=True,
                         arrowsize=18, width=2, with_labels=True)
        # Edge label (support/contradict)
        edge_labels = {(e["src"],e["tgt"]): f"{e['relation'][:4]}\n{e.get('support_score',0):.2f}"
                       for e in edges}
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels,
                                     ax=ax2, font_size=7)
        ax2.set_title("NLI Reasoning Graph", fontsize=10)
        ax2.axis("off")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    plt.show()
    return fig

demo_specs = [
    (0,   False, False, "0★ Wrong Answer Example"),
    (1,   False, False, "1★ Weak Reasoning Example"),
    (3,   False, False, "3★ Moderate Reasoning Example"),
    (5,   False, False, "5★ Strong Reasoning Example"),
    (None,True,  False, "Contradiction-Heavy Example"),
    (None,False, True,  "Single-Sentence (No Graph) Example"),
]

demo_saved = []
for idx, (ts_target, pref_con, pref_sing, label) in enumerate(demo_specs, start=1):
    ans, ts = _pick_example(ts_target, pref_con, pref_sing)
    if ans is None:
        print(f"  ⚠ No example found for: {label}")
        continue
    _save_p = os.path.join(PLOTS_DIR, f"graph_demo_example_{idx}.png")
    print(f"\n--- {label} ---")
    _draw_graph(ans, ts, label, save_path=_save_p)
    demo_saved.append({"index": idx, "label": label,
                       "target_stars": ts, "depth_score": ans.get("depth_score"),
                       "reasoning_level": ans.get("reasoning_level"),
                       "remark": ans.get("remark"),
                       "reasoning_graph": ans.get("reasoning_graph",[]),
                       "question": ans.get("question","")[:120],
                       "student_answer": ans.get("student_answer","")[:200]})

with open(GRAPH_DEMO_JSON, "w") as f:
    json.dump(demo_saved, f, indent=2)
print(f"\nGraph demo examples saved: {GRAPH_DEMO_JSON}")
print(f"Total demo plots generated: {len(demo_saved)}")

# Section 23 — Project Results Summary + Final Report


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Section 23 — Project Results Summary + Final Report
# Saves: metrics_summary.json, final_report.json, final_report.md
# ══════════════════════════════════════════════════════════════════════════════

import os, json as _json, datetime
import pandas as pd

_ts = datetime.datetime.now().isoformat()

# ── Collect metrics ───────────────────────────────────────────────────────────
def _g(name, default=None):
    return globals().get(name, default)

_mae_v  = _g("mae")
_rmse_v = _g("rmse")
_r_v    = _g("r")
_r2_v   = _g("r2")

_best_hp = {}
if _cached(BEST_HP_JSON):
    try:
        with open(BEST_HP_JSON) as f: _best_hp = _json.load(f)
    except Exception: pass

_taxonomy = {}
if _cached(FAILURE_TAXONOMY_JSON):
    try:
        with open(FAILURE_TAXONOMY_JSON) as f: _taxonomy = _json.load(f)
    except Exception: pass

_training_available = (
    _cached(TRAINING_HISTORY_JSON) or
    ("epoch_history" in dir() and bool(epoch_history))
)

# ── Role performance ──────────────────────────────────────────────────────────
_role_perf = {}
if "sentence_df" in dir() and "role" in sentence_df.columns:
    from collections import Counter
    _rc = Counter(sentence_df["role"].tolist())
    _tot = max(sum(_rc.values()), 1)
    _role_perf = {r: {"count":c,"pct":round(c/_tot*100,2)}
                  for r,c in _rc.items()}
_evidence_pct = _role_perf.get("evidence",{}).get("pct", 0)

# ── Star edge performance ─────────────────────────────────────────────────────
_edge_star_perf = {}
if _cached(STAR_EVAL_CSV):
    try:
        _sev = pd.read_csv(STAR_EVAL_CSV)
        for _s in [0, 5]:
            _sub = _sev[_sev["target_stars"]==_s] if "target_stars" in _sev.columns else pd.DataFrame()
            if len(_sub) > 0 and "predicted_stars_calibrated" in _sub.columns:
                _mae_s = abs(_sub["target_stars"] - _sub["predicted_stars_calibrated"]).mean()
                _acc_s = (_sub["target_stars"]==_sub["predicted_stars_calibrated"]).mean()
                _edge_star_perf[f"{_s}_star"] = {"mae":round(_mae_s,4),"acc":round(_acc_s,4),"n":len(_sub)}
    except Exception: pass

# ── Build report ─────────────────────────────────────────────────────────────
_output_files = {k:os.path.exists(v) for k,v in {
    "combined_all_df.csv":          COMBINED_ALL_CSV,
    "auxiliary_scoring_df.csv":     AUXILIARY_CSV,
    "reasoning_df.csv":             REASONING_CSV,
    "stage1_sentence_df.csv":       STAGE1_SENTENCE_CSV,
    "stage1_feedback.json":         STAGE1_JSON,
    "stage2_feedback.json":         STAGE2_JSON,
    "final_feedback_summary.csv":   FINAL_SUMMARY_CSV,
    "star_level_evaluation.csv":    STAR_EVAL_CSV,
    "human_validation_template.csv":HUMAN_VALIDATION_CSV,
    "training_history.json":        TRAINING_HISTORY_JSON,
    "training_history.csv":         TRAINING_HISTORY_CSV,
    "best_hyperparameters.json":    BEST_HP_JSON,
    "failure_taxonomy_report.json": FAILURE_TAXONOMY_JSON,
    "failure_taxonomy_examples.csv":FAILURE_TAXONOMY_CSV,
    "reasoning_edge_cases.csv":     REASONING_EDGE_CSV,
    "training_curve.png":           TRAINING_CURVE_PNG,
}.items()}

report = {
    "generated_at":              _ts,
    "training_curve_available":  _training_available,
    "best_hyperparameters":      _best_hp,
    "distilbert_metrics": {
        "mae":  str(_mae_v), "rmse": str(_rmse_v),
        "pearson_r": str(_r_v), "r2": str(_r2_v),
    },
    "role_classification": _role_perf,
    "evidence_role_pct":   _evidence_pct,
    "star_edge_performance": _edge_star_perf,
    "failure_taxonomy_summary": {
        k: {"count":v["count"],"pct":v["percentage"]}
        for k,v in _taxonomy.items()
    },
    "nli_data": {
        "snli":  "snli_raw" in dir() and snli_raw is not None,
        "mnli":  "mnli_raw" in dir() and mnli_raw is not None,
        "qasc":  "qasc_raw" in dir() and qasc_raw is not None,
    },
    "output_files": _output_files,
    "recommendations": [
        "Run FORCE_RETRAIN_FOR_CURVE=True to generate full epoch curves.",
        "Annotate human_validation_template.csv for inter-rater agreement.",
        "If evidence_role_pct < 10%, run evidence augmentation (Section 12).",
        "If edge star MAE > 1.5, increase EDGE_STAR_WEIGHT in Section 4.",
        "If val loss rises after epoch 2, reduce NUM_EPOCHS or increase EARLY_STOPPING_PATIENCE.",
        "Stage 2 LLM refinement: add OPENROUTER_API_KEY to Colab Secrets to enable.",
    ],
}

# Save JSON
try:
    with open(FINAL_REPORT_JSON,"w") as f: _json.dump(report, f, indent=2)
    with open(METRICS_SUMMARY_JSON,"w") as f:
        _json.dump(report["distilbert_metrics"], f, indent=2)
    print(f"Final report JSON  : {FINAL_REPORT_JSON}")
    print(f"Metrics summary    : {METRICS_SUMMARY_JSON}")
except Exception as _e: print(f"Report JSON error: {_e}")

# Save Markdown
_md = f"""# REXA — Final Report
Generated: {_ts}

## DistilBERT Regression Metrics
| Metric | Value |
|---|---|
| MAE | {_mae_v} |
| RMSE | {_rmse_v} |
| Pearson r | {_r_v} |
| R² | {_r2_v} |

## Best Hyperparameters
| Parameter | Value |
|---|---|
""" + "\n".join(f"| {k} | {v} |" for k,v in _best_hp.items()) + f"""

## Role Classification
| Role | Count | % |
|---|---|---|
""" + "\n".join(f"| {r} | {v['count']} | {v['pct']}% |"
                for r,v in _role_perf.items()) + f"""

## Evidence Role Coverage
Evidence sentences: **{_evidence_pct:.1f}%**
{('⚠ Underrepresented — augmentation recommended.' if _evidence_pct < 10 else '✓ Acceptable coverage.')}

## Star Edge Performance
| Star | N | MAE | Acc |
|---|---|---|---|
""" + "\n".join(f"| {k} | {v['n']} | {v['mae']} | {v['acc']} |"
                for k,v in _edge_star_perf.items()) + f"""

## Training Curve
Training curve available: **{_training_available}**
Saved to: {TRAINING_CURVE_PNG}

## NLI Data Availability
| Source | Available |
|---|---|
| SNLI | {'Yes' if report['nli_data']['snli'] else 'No'} |
| MultiNLI | {'Yes' if report['nli_data']['mnli'] else 'No'} |
| QASC | {'Yes' if report['nli_data']['qasc'] else 'No'} |

## Output Files
| File | Status |
|---|---|
""" + "\n".join(f"| {k} | {'✓' if v else '✗'} |"
                for k,v in _output_files.items()) + f"""

## Recommendations
""" + "\n".join(f"- {r}" for r in report["recommendations"]) + "\n"

try:
    with open(FINAL_REPORT_MD,"w") as f: f.write(_md)
    print(f"Final report MD    : {FINAL_REPORT_MD}")
except Exception as _e: print(f"Report MD error: {_e}")

# ── File status table ─────────────────────────────────────────────────────────
print("\n" + "="*62)
print("  OUTPUT FILE STATUS")
print("="*62)
for fname, exists in _output_files.items():
    _sym = "✓" if exists else "✗"
    print(f"  {_sym}  {fname}")
print("="*62)
logger.info("Final report generated.")


In [ ]:
# ── Section 23: Project Results Summary ──────────────────────────────────────
import os, json
import pandas as pd

# ── Output file verification ──────────────────────────────────────────────────
output_check = {
    "combined_all_df.csv":           globals().get("COMBINED_ALL_CSV", ""),
    "auxiliary_scoring_df.csv":      globals().get("AUXILIARY_CSV", ""),
    "reasoning_df.csv":              globals().get("REASONING_CSV", ""),
    "train.csv":                     globals().get("TRAIN_CSV", ""),
    "val.csv":                       globals().get("VAL_CSV", ""),
    "test.csv":                      globals().get("TEST_CSV", ""),
    "stage1_sentence_df.csv":        globals().get("STAGE1_SENTENCE_CSV", ""),
    "stage1_feedback.json":          globals().get("STAGE1_FEEDBACK_JSON", ""),
    "final_feedback_summary.csv":    globals().get("FINAL_SUMMARY_CSV", ""),
    "star_level_evaluation.csv":     globals().get("STAR_EVAL_CSV", ""),
    "ablation_study.csv":            globals().get("ABLATION_CSV", ""),
    "failure_taxonomy.csv":          globals().get("FAILURE_CSV", ""),
    "orthogonality_report.csv":      globals().get("ORTHO_CSV", ""),
    "human_validation_template.csv": globals().get("HUMAN_VALIDATION_CSV", ""),
    "graph_demo_examples.json":      globals().get("GRAPH_DEMO_JSON", ""),
    "user_test_output.json":         globals().get("USER_TEST_JSON", ""),
    "dataset_overview.png":          os.path.join(globals().get("PLOTS_DIR",""), "dataset_overview.png"),
    "pipeline_analysis.png":         os.path.join(globals().get("PLOTS_DIR",""), "pipeline_analysis.png"),
    "star_level_evaluation.png":     os.path.join(globals().get("PLOTS_DIR",""), "star_level_evaluation.png"),
    "training_validation_curve.png": globals().get("TRAINING_CURVE_PNG", ""),
}

print("="*70)
print("  OUTPUT FILE VERIFICATION")
print("="*70)
for label, path in output_check.items():
    if not path:
        icon, status = "?", "PATH NOT CONFIGURED"
    elif os.path.exists(path):
        kb = os.path.getsize(path) // 1024
        icon, status = "v", f"FOUND  ({kb} KB)"
    else:
        icon, status = "x", "NOT FOUND"
    print(f"  [{icon}] {label:<42} {status}")

# ── Safe metric extraction ─────────────────────────────────────────────────────
def _safe(var, fmt="{:.3f}", default="See notebook output"):
    try:   return fmt.format(float(globals().get(var, float("nan"))))
    except: return default

_mae_str   = _safe("mae")
_r_str     = _safe("r")
_rmse_str  = _safe("rmse")
_r2_str    = _safe("r2")
_kap_str   = _safe("kap")
_acc_str   = _safe("acc", "{:.1%}")

try:
    _curated_n = len(combined_all_df[combined_all_df["dataset_purpose"]=="curated_demo"])
    _total_n   = len(combined_all_df)
    _multi_pct = f"{combined_all_df['is_multi_sentence'].mean()*100:.1f}%"
except:
    _curated_n = "?"; _total_n = "?"; _multi_pct = "?"

# ── Results summary table ─────────────────────────────────────────────────────
print()
print("="*90)
print("  PROJECT RESULTS SUMMARY")
print("="*90)
_rows = [
    ("Component",             "What Was Done",                                          "Output / Result",                       "Status"),
    ("-"*22,                  "-"*40,                                                   "-"*35,                                  "-"*12),
    ("Dataset Integration",   "4 ASAG + SNLI/MNLI + QASC + 0-5* CSVs",                f"Total rows: {_total_n}",               "Completed"),
    ("Dataset Purpose Sep.",  "auxiliary/nli/multihop/curated separated",              "combined_all_df.csv",                   "Completed"),
    ("Multi-sentence Check",  "is_multi_sentence flagged per row",                     f"Multi-sent: {_multi_pct}",             "Completed"),
    ("Curated Star Corpus",   "0star.csv to 5star.csv loaded, target_stars assigned",  f"Curated rows: {_curated_n}",           "Completed"),
    ("DistilBERT Regression", "Trained on auxiliary_scoring ONLY",                     f"MAE={_mae_str}, r={_r_str}",           "Completed"),
    ("DistilBERT Baselines",  "TF-IDF Ridge + length + score-mean compared",           "Section 9 output",                     "Completed"),
    ("Training Curve",        "Per-epoch logs extracted if available",                 "plots/training_validation_curve.png",   "Completed if logs available"),
    ("Role Classification",   "BART + cue override + SBERT irrelevant filter",         "stage1_sentence_df.csv",               "Completed"),
    ("Requirement Coverage",  "SBERT cosine against reference requirements",           "requirement_score per answer",          "Completed"),
    ("NLI Reasoning Graph",   "DeBERTa-v3 NLI; confidence + severity labels",         "reasoning_graph in final_feedback",         "Completed"),
    ("Depth Score",           "0.45*struct + 0.55*reasoning (ablation-justified)",     "depth_score in summary CSV",            "Completed"),
    ("Predicted Stars",       "Depth score binned to 0-5 stars",                       "predicted_stars in summary CSV",        "Completed"),
    ("Star-Level Eval.",      "Accuracy/MAE/F1/kappa vs target_stars",                 f"Accuracy={_acc_str}, kappa={_kap_str}","Completed"),
    ("Orthogonality",         "Pearson/Spearman depth vs normalized_score",            "orthogonality_report.csv",              "Completed"),
    ("Failure Taxonomy",      "8 failure categories characterised",                    "failure_taxonomy.csv",                  "Completed"),
    ("Ablation Study",        "5 weight configs compared",                             "ablation_study.csv",                   "Completed"),
    ("Human Validation",      "Blank template created; no labels fabricated",          "human_validation_template.csv",         "Template ready"),
    ("Graph Demo",            "6 example types visualised with NLI edges",             "graph_demo_examples.json",              "Completed"),
    ("Custom Testing",        "Full pipeline on arbitrary Q+A input",                  "user_test_output.json",                 "Completed"),
]
for row in _rows:
    print(f"  {row[0]:<24} {row[1]:<42} {row[2]:<37} {row[3]}")
print("="*90)

# ── Reproducibility info ──────────────────────────────────────────────────────
import sys, platform
print()
print("="*60)
print("  REPRODUCIBILITY INFORMATION")
print("="*60)
print(f"  Python        : {sys.version.split()[0]}")
print(f"  Platform      : {platform.system()} {platform.release()}")
try:
    import torch;              print(f"  PyTorch       : {torch.__version__}")
except: pass
try:
    import transformers;       print(f"  Transformers  : {transformers.__version__}")
except: pass
try:
    import sentence_transformers; print(f"  SBERT         : {sentence_transformers.__version__}")
except: pass
try:
    import sklearn;            print(f"  scikit-learn  : {sklearn.__version__}")
except: pass
try:
    import pandas as _pd;     print(f"  pandas        : {_pd.__version__}")
except: pass
try:
    import numpy as _np;      print(f"  numpy         : {_np.__version__}")
except: pass
print(f"  Random state  : {globals().get('RANDOM_STATE', 42)}")
print(f"  Edge threshold: {globals().get('EDGE_THRESHOLD', 0.25)}")
print("="*60)
print()
print("  To export environment:")
print("  !pip freeze > requirements.txt")
print("  print(open('requirements.txt').read())")


# ── Fix verification checks ───────────────────────────────────────────────────
print()
print("Running fix verification checks...")
_checks_passed = True

try:
    assert callable(_cached), "_cached not defined"
    print("  [✓] _cached() defined")
except AssertionError as e: print(f"  [✗] {e}"); _checks_passed = False

try:
    assert "combined_all_df" in dir(), "combined_all_df not in scope"
    purposes = set(combined_all_df["dataset_purpose"].unique())
    assert "curated_demo"  in purposes, "curated_demo missing"
    assert "nli_reasoning" in purposes, "nli_reasoning missing"
    print(f"  [✓] combined_all_df purposes: {sorted(purposes)}")
except AssertionError as e: print(f"  [✗] {e}"); _checks_passed = False

try:
    assert EDGE_THRESHOLD == 0.25, f"EDGE_THRESHOLD must be 0.25, got {EDGE_THRESHOLD}"
    print(f"  [✓] EDGE_THRESHOLD = {EDGE_THRESHOLD}")
except AssertionError as e: print(f"  [✗] {e}"); _checks_passed = False

try:
    assert "final_feedback" in dir() and len(final_feedback) > 0, "final_feedback empty"
    print(f"  [✓] final_feedback: {len(final_feedback)} entries")
except AssertionError as e: print(f"  [✗] {e}"); _checks_passed = False

try:
    assert "summary_df" in dir() and len(summary_df) > 0, "summary_df empty"
    print(f"  [✓] summary_df: {len(summary_df)} rows")
except AssertionError as e: print(f"  [✗] {e}"); _checks_passed = False

try:
    assert os.path.exists(FINAL_SUMMARY_CSV), "final_feedback_summary.csv missing"
    print(f"  [✓] final_feedback_summary.csv exists")
except AssertionError as e: print(f"  [✗] {e}"); _checks_passed = False

print()
print("All checks passed ✓" if _checks_passed else "Some checks failed — see above.")


## Section 24 — End-to-End Demo

**Purpose:** Trace one complete answer through every pipeline stage so the full system behaviour is visible in one place.

**What this cell does (8 steps):**
1. Input: question + student answer + reference
2. Sentence splitting and question-type detection
3. Role prediction per sentence (with confidence bars)
4. Requirement extraction and coverage check
5. NLI reasoning graph construction
6. Depth score and predicted stars calculation
7. Reasoning level and actionable remark
8. Graph visualisation (or "not applicable" for single-sentence)

**Output:** Printed walkthrough + graph saved to `PLOTS_DIR/demo_reasoning_graph.png` + `JSON_DIR/user_test_output.json`.

**Why this section exists:** A complete worked example is more informative than metrics alone. It lets a reviewer trace the exact logic from input text to star rating.


In [ ]:
# ── Section 24: End-to-end demo ─────────────────────────────────────────────
print("="*65)
print("  REXA — END-TO-END PIPELINE DEMO")
print("="*65)

DEMO_Q = "Explain why photosynthesis is important for life on Earth."
DEMO_A = ("Photosynthesis is the process by which plants make food using sunlight. "
          "Because plants absorb carbon dioxide and release oxygen, they are essential "
          "for breathing. For example, all animals including humans depend on oxygen "
          "produced by plants. Therefore, without photosynthesis, most life on Earth "
          "could not survive.")

print(f"  Question : {DEMO_Q}")
print(f"  Answer   : {DEMO_A[:90]}...")
print()

try:
    _sents = split_into_sentences(DEMO_A)
except Exception:
    import re
    _sents = [s.strip() for s in re.split(r'[.!?]', DEMO_A) if len(s.strip()) > 5]

print(f"  Sentences: {len(_sents)}")
for i, s in enumerate(_sents, 1):
    print(f"    [{i}] {s}")

try:
    _q_type = qt_classifier(DEMO_Q, Q_TYPES)["labels"][0]
except Exception:
    _q_type = "explanation"
_exp_roles = ROLE_MAP.get(_q_type, ["claim","explanation","evidence","conclusion"])
print(f"\n  Q-type  : {_q_type}  |  Expected roles: {_exp_roles}")

# FIX: classify_role returns 4 values
_demo_sents = []
print("\n  Role classification:")
for i, s in enumerate(_sents, 1):
    try:
        _role, _sec, _conf, _src = classify_role(s, DEMO_A, i, len(_sents))
    except Exception as _e:
        _role, _sec, _conf, _src = "claim", None, "low", f"err:{_e}"
    print(f"    [{i}] {_role:<14} ({_conf:<6}) {s[:55]}")
    _demo_sents.append({"sentence_id":i,"text":s,"role":_role,"secondary_role":_sec,
        "has_causal_cue":any(c in s.lower() for c in CAUSAL_KW),
        "semantic_sim":0.0,"semantic_status":"missing","covers":[],"requirement_coverage":[]})

try:
    _is_multi = len(_sents) >= 2
    (depth, struct_sc, role_cov, seq_sc, req_sc,
     reasoning_score, has_valid_path, edges, single_flag) = compute_depth_score(
        _demo_sents, _exp_roles, None, DEMO_A, DEMO_Q)
    _stars  = depth_to_stars(depth)
    _level  = reasoning_level_from_depth(depth)
    _remark = generate_remark(_demo_sents, _exp_roles, has_valid_path, edges, _is_multi, not _is_multi)
    print(f"\n  Structure score  : {struct_sc:.4f}")
    print(f"  Reasoning score  : {reasoning_score if reasoning_score is not None else 'N/A'}")
    print(f"  Depth score      : {depth:.4f}")
    print(f"  Predicted stars  : {'★'*_stars+'☆'*(5-_stars)}  ({_stars}★)")
    print(f"  Reasoning level  : {_level}")
    print(f"  Graph edges      : {len(edges)}")
    for e in edges[:3]:
        print(f"    {e['src_role']} →[{e['confidence']}] {e['tgt_role']} {e['support_score']:.3f}")
    print(f"  Remark           : {_remark}")
except Exception as _e:
    print(f"\n  Pipeline demo error: {_e}")
print("="*65)

# ══════════════════════════════════════════════════════════════════════════════
# REXA Custom Answer Testing Interface
# ══════════════════════════════════════════════════════════════════════════════
if "role_classifier" not in dir() or role_classifier is None:
    from transformers import pipeline as hf_pipeline
    role_classifier = hf_pipeline("zero-shot-classification", model="facebook/bart-large-mnli",
                                   device=0 if __import__("torch").cuda.is_available() else -1)
if "sbert_model" not in dir() or sbert_model is None:
    from sentence_transformers import SentenceTransformer, util
    sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
if "nli_classifier"  not in dir(): nli_classifier = None
if "ROLE_LABELS"     not in dir(): ROLE_LABELS = ["claim","explanation","evidence","conclusion","irrelevant"]
if "ROLE_MAP"        not in dir(): ROLE_MAP = {"explanation":["claim","explanation","evidence"],"general":["claim","explanation"]}
if "Q_TYPES"         not in dir(): Q_TYPES  = ["definition","explanation","comparison","justification","process","entailment","general"]

USER_QUESTION  = "Why does a damaged bulb in series turn off other bulbs?"
USER_ANSWER    = ("A damaged bulb creates a gap in the circuit. "
                  "Because the circuit is no longer closed, current cannot flow through the path. "
                  "Therefore all bulbs connected in series with it will also turn off.")
USER_REFERENCE = "In a series circuit a break at any point stops current flow for all components."

print("="*65)
print("  REXA — CUSTOM ANSWER TESTING INTERFACE")
print("="*65)
print(f"  Q: {USER_QUESTION}")
print(f"  A: {USER_ANSWER[:120]}")
print("-"*65)

user_sents = split_into_sentences(USER_ANSWER)
is_multi   = len(user_sents) >= 2
print(f"  Sentences: {len(user_sents)}")
for i, s in enumerate(user_sents, 1): print(f"    [{i}] {s}")

try:
    user_q_type = qt_classifier(USER_QUESTION, Q_TYPES)["labels"][0]
except Exception: user_q_type = "explanation"
user_exp_roles = ROLE_MAP.get(user_q_type, ["claim","explanation"])
print(f"\n  Q-type: {user_q_type}  |  Expected: {user_exp_roles}")

print("\n  Classifying roles...")
user_sent_dicts = []
for i, sent in enumerate(user_sents, 1):
    try:
        role, sec, conf, src_lbl = classify_role(sent, USER_ANSWER, i, len(user_sents))
    except Exception: role, sec, conf, src_lbl = "claim", None, "low", "fallback"
    print(f"    [{i}] {role:<14} ({conf:<6}) {sent[:60]}")
    user_sent_dicts.append({"sentence_id":i,"text":sent,"role":role,"secondary_role":sec,
        "has_causal_cue":any(c in sent.lower() for c in CAUSAL_KW),
        "semantic_sim":0.0,"semantic_status":"missing","covers":[],"requirement_coverage":[]})

role_cov  = compute_role_coverage(user_sent_dicts, user_exp_roles)
seq_score = compute_sequence_score(user_sent_dicts)
req_score = compute_requirement_score(user_sent_dicts, USER_REFERENCE, USER_QUESTION)
structure = round(0.55*role_cov + 0.25*seq_score + 0.20*req_score, 4)
detected  = set(s["role"] for s in user_sent_dicts if s.get("role") not in ("irrelevant",None))
missing   = [r for r in user_exp_roles if r not in detected]

print(f"\n  Role coverage    : {role_cov:.3f}")
print(f"  Sequence score   : {seq_score:.3f}")
print(f"  Requirement score: {req_score:.3f}")
print(f"  Structure score  : {structure:.3f}")
print(f"  Covered          : {sorted(detected)}")
print(f"  Missing          : {missing}")

user_edges, user_reasoning_sc, user_has_valid_path = [], None, False
if is_multi:
    try:
        user_edges, user_reasoning_sc, user_has_valid_path = build_reasoning_graph(user_sent_dicts)
        print(f"\n  Graph edges   : {len(user_edges)}")
        print(f"  Support       : {sum(1 for e in user_edges if e['relation']=='support')}")
        print(f"  Contradiction : {sum(1 for e in user_edges if e['relation']=='contradict')}")
        print(f"  Valid path    : {user_has_valid_path}")
        print(f"  Reasoning sc  : {user_reasoning_sc}")
        for e in user_edges[:3]:
            print(f"    {e['src_role']} →[{e['confidence']}] {e['tgt_role']} {e['support_score']:.3f}")
    except Exception as _ge: print(f"  Graph error: {_ge}")
else:
    print("\n  Graph: N/A (single-sentence)")

if user_reasoning_sc is not None:
    user_depth = round(max(0.0, min(1.0, 0.45*structure + 0.55*user_reasoning_sc)), 4)
elif is_multi: user_depth = round(0.75 * structure, 4)
else:          user_depth = round(min(structure * 0.75, 0.65), 4)

user_stars  = depth_to_stars(user_depth)
user_level  = reasoning_level_from_depth(user_depth)
user_remark = generate_remark(user_sent_dicts, user_exp_roles, user_has_valid_path,
                               user_edges, is_multi, not is_multi)

print()
print("="*65)
print("  REXA RESULT")
print("="*65)
print(f"  Depth score     : {user_depth:.4f}")
print(f"  Predicted stars : {'★'*user_stars+'☆'*(5-user_stars)}  ({user_stars}★)")
print(f"  Reasoning level : {user_level}")
print(f"  Remark          : {user_remark}")
print("="*65)

import json as _json
_out = {"question":USER_QUESTION,"student_answer":USER_ANSWER,"reference_answer":USER_REFERENCE,
        "sentences":user_sent_dicts,"role_coverage":role_cov,"sequence_score":seq_score,
        "requirement_score":req_score,"structure_score":structure,"reasoning_score":user_reasoning_sc,
        "depth_score":user_depth,"predicted_stars":user_stars,"reasoning_level":user_level,
        "graph_edges":user_edges,"has_valid_path":user_has_valid_path,"remark":user_remark}
try:
    with open(USER_TEST_JSON, "w") as f: _json.dump(_out, f, indent=2)
    print(f"\n  Saved: {USER_TEST_JSON}")
except Exception as _se: print(f"\n  Save error: {_se}")


## Final Section — REXA Custom Answer Testing Interface

**Purpose:** Test the full REXA pipeline on any question and student answer interactively.

**How to use:**
1. Set `TEST_QUESTION`, `TEST_ANSWER`, and `TEST_REFERENCE` in the cell below
2. Run the cell
3. View: sentence roles, depth score, predicted stars, reasoning level, remark, and graph

**Single-sentence answers:** Graph not applicable — structural analysis only (depth capped at 0.65).
**Multi-sentence answers:** Full NLI graph built and visualised.

**Output files:**
- `json/user_test_output.json` — full analysis results
- `plots/user_test_graph.png` — reasoning graph (if multi-sentence)

> **Error handling:** Each pipeline stage is wrapped in `try/except`. If a model is unavailable, the stage falls back gracefully and continues.


In [ ]:
# ── Safety: ensure classifiers available before testing ──────────────────────
if "role_classifier" not in dir() or role_classifier is None:
    print("Loading role_classifier for testing interface...")
    from transformers import pipeline as hf_pipeline
    role_classifier = hf_pipeline("zero-shot-classification",
                                   model="facebook/bart-large-mnli",
                                   device=0 if __import__("torch").cuda.is_available() else -1)
if "sbert_model" not in dir() or sbert_model is None:
    from sentence_transformers import SentenceTransformer, util
    sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
if "nli_classifier" not in dir():
    nli_classifier = None
if "ROLE_LABELS" not in dir():
    ROLE_LABELS = ["claim","explanation","evidence","conclusion","irrelevant"]
if "ROLE_MAP" not in dir():
    ROLE_MAP = {
        "definition":    ["claim","explanation"],
        "explanation":   ["claim","explanation","evidence"],
        "comparison":    ["claim","evidence","conclusion"],
        "justification": ["claim","explanation","evidence","conclusion"],
        "general":       ["claim","explanation"],
    }

# ══════════════════════════════════════════════════════════════════════════════
# Final Section — REXA Testing Interface
# Enter any question and student answer to get full reasoning analysis.
# Splits answer → classifies roles → computes depth → builds NLI graph.
# Saves user_test_output.json and user_test_graph.png (if multi-sentence).
# ══════════════════════════════════════════════════════════════════════════════

# ── Input ─────────────────────────────────────────────────────────────────────
USER_QUESTION = "Why does a damaged bulb in series turn off other bulbs?"  # change to test your own
USER_ANSWER   = ("A damaged bulb creates a gap in the circuit. Because the circuit is no longer "
                  "closed, current cannot flow through the path. Therefore all bulbs connected in "
                  "series with it will also turn off.")  # change to test your own

if not USER_QUESTION or not USER_ANSWER:
    print("No input provided. Using demo values.")
    USER_QUESTION = "Explain why photosynthesis is important for life on Earth."
    USER_ANSWER   = ("Photosynthesis is the process by which plants make food using sunlight. "
                     "Because plants absorb carbon dioxide and release oxygen, "
                     "they are essential for breathing. "
                     "For example, all animals including humans depend on oxygen produced by plants. "
                     "Therefore, without photosynthesis, most life on Earth could not survive.")

print(f"\nQuestion : {USER_QUESTION}")
print(f"Answer   : {USER_ANSWER[:120]}{'...' if len(USER_ANSWER)>120 else ''}")
print("-" * 62)

# ── Sentence splitting ────────────────────────────────────────────────────────
user_sents = split_into_sentences(USER_ANSWER)
is_multi   = len(user_sents) >= 2
print(f"Sentences detected : {len(user_sents)}")
for i, s in enumerate(user_sents, 1):
    print(f"  [{i}] {s}")

# ── Question type ─────────────────────────────────────────────────────────────
try:
    user_q_type = qt_classifier(USER_QUESTION, Q_TYPES)["labels"][0]
except Exception:
    user_q_type = "explanation"
user_exp_roles = ROLE_MAP.get(user_q_type, ["claim","explanation"])
print(f"\nQuestion type     : {user_q_type}")
print(f"Expected roles    : {user_exp_roles}")

# ── Role classification ────────────────────────────────────────────────────────
# FIX: classify_role() returns 4 values (role, secondary, confidence, source),
# not 2 — the old 2-value unpack raised ValueError on every run of this cell.
# sent_id / total_sents are also now passed so position_fallback works correctly.
print("\nClassifying sentence roles...")
user_sent_dicts = []
for i, sent in enumerate(user_sents, 1):
    try:
        role, sec, conf, src_lbl = classify_role(sent, USER_ANSWER, i, len(user_sents))
    except Exception as _e:
        role, sec, conf, src_lbl = "claim", None, "low", f"fallback_error:{_e}"
    print(f"  [{i}] Role: {role:<14} ({conf:<6}) {'(+'+sec+')' if sec else ''}  {sent[:60]}")
    user_sent_dicts.append({
        "sentence_id":    i,
        "text":           sent,
        "role":           role,
        "secondary_role": sec,
        "role_confidence": conf,
        "role_source":     src_lbl,
        "has_causal_cue": any(c in sent.lower() for c in CAUSAL_CUES),
    })

# ── Structure metrics ─────────────────────────────────────────────────────────
role_cov  = compute_role_coverage(user_sent_dicts,  user_exp_roles)
seq_score = compute_sequence_score(user_sent_dicts)
req_score = compute_requirement_score(user_sent_dicts, USER_ANSWER, USER_QUESTION)
structure = round(0.55*role_cov + 0.25*seq_score + 0.20*req_score, 4)

detected = set(s["role"] for s in user_sent_dicts
               if s.get("role") not in ("irrelevant",None))
missing  = [r for r in user_exp_roles if r not in detected]

print(f"\nRole coverage     : {role_cov:.3f}")
print(f"Sequence score    : {seq_score:.3f}")
print(f"Requirement score : {req_score:.3f}")
print(f"Structure score   : {structure:.3f}")
print(f"Covered roles     : {sorted(detected)}")
print(f"Missing roles     : {missing}")

# ── Reasoning graph (multi-sentence only) ─────────────────────────────────────
user_edges         = []
user_reasoning_sc  = None
user_has_valid_path= False
graph_applicable   = is_multi

if is_multi:
    print("\nBuilding NLI reasoning graph...")
    user_edges, user_reasoning_sc, user_has_valid_path = build_reasoning_graph(user_sent_dicts)
    print(f"  Graph edges      : {len(user_edges)}")
    for e in user_edges:
        arrow = "──supports──>" if e["relation"]=="support" else "─contradicts─>"
        print(f"    [S{e['src']}]({e['src_role']}) {arrow} [S{e['tgt']}]({e['tgt_role']})  "
              f"score={e.get('support_score',0):.3f}")
    print(f"  Valid path       : {user_has_valid_path}")
    print(f"  Reasoning score  : {user_reasoning_sc}")
else:
    print("\nSingle-sentence answer detected.")
    print("Reasoning graph not applicable.")
    print("Structural evaluation only.")

# ── Depth score & stars ────────────────────────────────────────────────────────
if is_multi and user_reasoning_sc is not None:
    user_depth = round(max(0.0, min(1.0,
        0.45 * structure + 0.55 * user_reasoning_sc)), 4)
elif is_multi:
    user_depth = round(0.75 * structure, 4)
else:
    user_depth = min(structure, 0.60)

user_stars = depth_to_stars(user_depth)              # canonical (Cell 4)
user_level = reasoning_level_from_depth(user_depth)  # canonical (Cell 4)

user_remark = generate_remark(user_sent_dicts, user_exp_roles,
                               user_has_valid_path, user_edges,
                               graph_applicable, not is_multi)

print("\n" + "═"*62)
print("  REXA ANALYSIS RESULT")
print("═"*62)
print(f"  Depth Score       : {user_depth:.4f}")
print(f"  Stars             : {'★'*user_stars}{'☆'*(5-user_stars)}  ({user_stars}/5)")
print(f"  Reasoning Level   : {user_level}")
print(f"  Graph Applicable  : {graph_applicable}")
print(f"  Remark            : {user_remark}")
print("═"*62)

# ── Graph visualisation ────────────────────────────────────────────────────────
_user_graph_path = None
if is_multi and user_edges:
    try:
        import networkx as nx
        G = nx.DiGraph()
        for s in user_sent_dicts:
            sid  = s["sentence_id"]
            role = s.get("role","?")
            G.add_node(sid, role=role,
                       label=f"S{sid}\n({role})\n{s['text'][:25]}...")
        for e in user_edges:
            G.add_edge(e["src"], e["tgt"],
                       relation=e["relation"],
                       weight=abs(e.get("support_score",0)))
        fig, ax = plt.subplots(figsize=(10,6))
        pos = nx.spring_layout(G, seed=42, k=2.5)
        nc  = [ROLE_COLORS.get(G.nodes[n].get("role","irrelevant"),"#9E9E9E")
               for n in G.nodes()]
        ec  = [EDGE_COLORS.get(G.edges[e].get("relation","support"),"#888")
               for e in G.edges()]
        lbl = {n: G.nodes[n].get("label","") for n in G.nodes()}
        nx.draw_networkx(G, pos, ax=ax, labels=lbl,
                         node_color=nc, edge_color=ec,
                         node_size=2000, font_size=8, arrows=True,
                         arrowsize=18, width=2)
        edge_lbl = {(e["src"],e["tgt"]):
                    f"{e['relation'][:4]}\n{e.get('support_score',0):.2f}"
                    for e in user_edges}
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_lbl, ax=ax, font_size=7)
        ax.set_title(f"REXA NLI Reasoning Graph — {user_level}  {user_depth:.3f}  {'★'*user_stars}",
                     fontsize=12)
        ax.axis("off")
        plt.tight_layout()
        _user_graph_path = os.path.join(OUTPUT_DIR, "user_test_graph.png")
        plt.savefig(_user_graph_path, dpi=120, bbox_inches="tight")
        plt.show()
        print(f"Graph saved: {_user_graph_path}")
    except Exception as _ge:
        print(f"Graph visualisation error: {_ge}")
elif is_multi:
    print("No graph edges to visualise (answers may be semantically too different).")

# ── Save output JSON ───────────────────────────────────────────────────────────
user_output = {
    "question":          USER_QUESTION,
    "student_answer":    USER_ANSWER,
    "question_type":     user_q_type,
    "expected_roles":    user_exp_roles,
    "is_multi_sentence": is_multi,
    "sentences":         user_sent_dicts,
    "covered_roles":     list(detected),
    "missing_roles":     missing,
    "role_coverage_score": role_cov,
    "sequence_score":    seq_score,
    "requirement_score": req_score,
    "structure_score":   structure,
    "reasoning_score":   user_reasoning_sc,
    "depth_score":       user_depth,
    "stars":             user_stars,
    "reasoning_level":   user_level,
    "graph_applicable":  graph_applicable,
    "reasoning_graph":       user_edges,
    "reasoning_graph_count": len(user_edges),
    "support_edges":     sum(1 for e in user_edges if e["relation"]=="support"),
    "contradiction_edges":sum(1 for e in user_edges if e["relation"]=="contradict"),
    "has_valid_path":    user_has_valid_path,
    "remark":            user_remark,
    "graph_image":       _user_graph_path or "not_generated",
}
with open(USER_TEST_JSON, "w") as f:
    json.dump(user_output, f, indent=2)
print(f"User test output saved: {USER_TEST_JSON}")

---

# RExA — Acc / Precision / Recall / F1 & model comparison

Final metrics for the FYP Evaluation page and viva. Full walkthrough: `06_metrics_and_model_comparison.ipynb`.

| Module | Accuracy | Precision | Recall | F1 |
|--------|----------|-----------|--------|-----|
| Sentence Roles (Core) | **95.88%** | 91.83% | 98.29% | **94.46%** |
| Concept Coverage | 85.68% | 64.74% | 84.98% | 68.46% |
| Support/Contradiction* | 100%* | 100%* | 100%* | 100%* |

*Silver heuristic labels — disclose in viva.

**Literature (contextual):** DAES Acc 95% / F1 94% (IEEE Access 2024). RExA roles Acc ~95.9% / F1 ~94.5% with explainable reasoning structure.

Figures saved under `docs/figures/` and `public/evaluation/figures/` (`09`–`11`).


In [ ]:
"""Build Acc/P/R/F1 tables + comparison graphs (same as notebook 06)."""
from pathlib import Path
import json, subprocess, sys
ROOT = Path("../..").resolve()
script = ROOT / "ml" / "scripts" / "generate_comparison_metrics.py"
subprocess.check_call([sys.executable, str(script)])
cmp = json.loads((ROOT / "public" / "evaluation" / "comparison_tables.json").read_text(encoding="utf-8"))
import pandas as pd
print("=== Core RExA modules ===")
display(pd.DataFrame(cmp["rexa_clf_table"]))
print("\n=== Literature comparison ===")
display(pd.DataFrame(cmp["literature_table"]))
print("\n=== Star scoring ===")
display(pd.DataFrame(cmp["star_table"]))
from IPython.display import Image, display as show
fig_dir = ROOT / "docs" / "figures"
for name in cmp["figures"]:
    path = fig_dir / name
    if path.exists():
        print(name)
        show(Image(filename=str(path)))
    else:
        print("missing:", path)
